<a href="https://colab.research.google.com/github/Platinum04/EgoSpatial-Dataset/blob/main/EgoSpatial_Gemma_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Sep  9 16:41:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.23" "peft" "accelerate" "bitsandbytes"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.9 MB/s et

In [ ]:
# Check the versions currently installed
import sys
import fsspec
import gcsfs

print("Python:", sys.version)
print("fsspec:", fsspec.__version__)
print("gcsfs:", gcsfs.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
fsspec: 2025.9.0
gcsfs: 2025.12.0


In [ ]:
import os
import requests

DATA_DIR = "/content/egospatial"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/Platinum04/EgoSpatial-Dataset/main"

files = [
    "v1_balanced_questions_train_scannetv2.json",
    "v1_balanced_questions_val_scannetv2.json",
    "v1_balanced_questions_test_scannetv2.json",
    "v1_balanced_sqa_annotations_train_scannetv2.json",
    "v1_balanced_sqa_annotations_val_scannetv2.json",
    "v1_balanced_sqa_annotations_test_scannetv2.json",
]

for filename in files:
    url = f"{BASE_URL}/{filename}"
    output_path = os.path.join(DATA_DIR, filename)

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    print(f"✓ {filename} — {len(response.content):,} bytes")

print("\nAll 6 files downloaded successfully.")

✓ v1_balanced_questions_train_scannetv2.json — 11,140,002 bytes
✓ v1_balanced_questions_val_scannetv2.json — 1,385,508 bytes
✓ v1_balanced_questions_test_scannetv2.json — 1,460,185 bytes
✓ v1_balanced_sqa_annotations_train_scannetv2.json — 9,093,541 bytes
✓ v1_balanced_sqa_annotations_val_scannetv2.json — 1,114,948 bytes
✓ v1_balanced_sqa_annotations_test_scannetv2.json — 1,200,381 bytes

All 6 files downloaded successfully.


In [ ]:
import json
import os

expected = {
    "v1_balanced_questions_train_scannetv2.json": ("questions", 26623),
    "v1_balanced_questions_val_scannetv2.json": ("questions", 3261),
    "v1_balanced_questions_test_scannetv2.json": ("questions", 3519),
    "v1_balanced_sqa_annotations_train_scannetv2.json": ("annotations", 26623),
    "v1_balanced_sqa_annotations_val_scannetv2.json": ("annotations", 3261),
    "v1_balanced_sqa_annotations_test_scannetv2.json": ("annotations", 3519),
}

for filename, (key, expected_count) in expected.items():
    path = os.path.join(DATA_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    actual_count = len(data[key])

    print(f"{filename}")
    print(f"  Key: {key}")
    print(f"  Records: {actual_count:,}")

    assert actual_count == expected_count, (
        f"COUNT ERROR: expected {expected_count:,}, got {actual_count:,}"
    )

print("\n✅ STEP 4 PASSED — ALL 6 JSON FILES VERIFIED.")

v1_balanced_questions_train_scannetv2.json
  Key: questions
  Records: 26,623
v1_balanced_questions_val_scannetv2.json
  Key: questions
  Records: 3,261
v1_balanced_questions_test_scannetv2.json
  Key: questions
  Records: 3,519
v1_balanced_sqa_annotations_train_scannetv2.json
  Key: annotations
  Records: 26,623
v1_balanced_sqa_annotations_val_scannetv2.json
  Key: annotations
  Records: 3,261
v1_balanced_sqa_annotations_test_scannetv2.json
  Key: annotations
  Records: 3,519

✅ STEP 4 PASSED — ALL 6 JSON FILES VERIFIED.


In [ ]:
import json
import os

def load_json(filename):
    path = os.path.join(DATA_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def verify_pairing(split):
    questions_file = f"v1_balanced_questions_{split}_scannetv2.json"
    annotations_file = f"v1_balanced_sqa_annotations_{split}_scannetv2.json"

    questions = load_json(questions_file)["questions"]
    annotations = load_json(annotations_file)["annotations"]

    errors = []

    for i, (q, a) in enumerate(zip(questions, annotations)):

        if q["question_id"] != a["question_id"]:
            errors.append(
                f"Index {i}: question_id mismatch "
                f"{q['question_id']} != {a['question_id']}"
            )

        if q["scene_id"] != a["scene_id"]:
            errors.append(
                f"Index {i}: scene_id mismatch "
                f"{q['scene_id']} != {a['scene_id']}"
            )

    print(f"{split.upper()}:")
    print(f"  Questions checked: {len(questions):,}")
    print(f"  Annotations checked: {len(annotations):,}")
    print(f"  Pairing errors: {len(errors)}")

    if errors:
        print("\nFirst errors:")
        for error in errors[:10]:
            print(" ", error)

    return len(errors) == 0


train_ok = verify_pairing("train")
val_ok = verify_pairing("val")
test_ok = verify_pairing("test")

assert train_ok and val_ok and test_ok

print("\n✅ STEP 5 PASSED — ALL QUESTION/ANNOTATION PAIRS MATCH.")

TRAIN:
  Questions checked: 26,623
  Annotations checked: 26,623
  Pairing errors: 0
VAL:
  Questions checked: 3,261
  Annotations checked: 3,261
  Pairing errors: 0
TEST:
  Questions checked: 3,519
  Annotations checked: 3,519
  Pairing errors: 0

✅ STEP 5 PASSED — ALL QUESTION/ANNOTATION PAIRS MATCH.


In [ ]:
from collections import Counter

def inspect_split(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    answer_types = Counter()
    question_types = Counter()
    answers = Counter()
    scenes = Counter()

    for q, a in zip(questions, annotations):
        question_types[a["question_type"]] += 1
        answer_types[a["answer_type"]] += 1
        answers[a["answers"][0]["answer"]] += 1
        scenes[q["scene_id"]] += 1

    print(f"\n{'=' * 50}")
    print(f"{split.upper()} DATASET")
    print(f"{'=' * 50}")

    print(f"Examples: {len(questions):,}")
    print(f"Unique scenes: {len(scenes):,}")
    print(f"Unique answers: {len(answers):,}")

    print("\nQuestion types:")
    for k, v in question_types.most_common():
        print(f"  {k}: {v:,}")

    print("\nAnswer types:")
    for k, v in answer_types.most_common():
        print(f"  {k}: {v:,}")

    print("\nTop 20 answers:")
    for answer, count in answers.most_common(20):
        print(f"  {answer}: {count:,}")


inspect_split("train")
inspect_split("val")
inspect_split("test")


TRAIN DATASET
Examples: 26,623
Unique scenes: 518
Unique answers: 1,277

Question types:
  N/A: 26,623

Answer types:
  other: 26,623

Top 20 answers:
  yes: 2,818
  no: 2,576
  right: 1,485
  left: 1,414
  one: 1,256
  two: 1,242
  even: 718
  odd: 664
  brown: 478
  backward: 428
  white: 411
  three: 410
  four: 400
  rectangular: 381
  closed: 355
  table: 351
  window: 298
  black: 289
  forward: 281
  chair: 241

VAL DATASET
Examples: 3,261
Unique scenes: 65
Unique answers: 396

Question types:
  N/A: 3,261

Answer types:
  other: 3,261

Top 20 answers:
  no: 322
  yes: 302
  two: 184
  right: 175
  one: 152
  left: 145
  odd: 102
  even: 83
  brown: 77
  table: 63
  rectangular: 58
  black: 55
  white: 46
  three: 43
  four: 37
  closed: 36
  chair: 36
  backward: 35
  window: 33
  forward: 29

TEST DATASET
Examples: 3,519
Unique scenes: 67
Unique answers: 403

Question types:
  N/A: 3,519

Answer types:
  other: 3,519

Top 20 answers:
  yes: 385
  no: 283
  left: 218
  right: 

In [ ]:
import statistics

train_questions = load_json(
    "v1_balanced_questions_train_scannetv2.json"
)["questions"]

train_annotations = load_json(
    "v1_balanced_sqa_annotations_train_scannetv2.json"
)["annotations"]


# ---------------------------------------------------------
# 1. Inspect representative examples
# ---------------------------------------------------------

print("=" * 70)
print("REPRESENTATIVE TRAINING EXAMPLES")
print("=" * 70)

for i in [0, 100, 1000, 5000, 10000, 20000]:
    q = train_questions[i]
    a = train_annotations[i]

    print(f"\n--- Example {i} ---")
    print("Scene:", q["scene_id"])
    print("Question ID:", q["question_id"])
    print("Situation:", q["situation"])
    print("Question:", q["question"])
    print("Answer:", a["answers"][0]["answer"])

    print("Position:", a["position"])
    print("Rotation:", a["rotation"])


# ---------------------------------------------------------
# 2. Calculate character lengths
# ---------------------------------------------------------

situation_lengths = [
    len(q["situation"])
    for q in train_questions
]

question_lengths = [
    len(q["question"])
    for q in train_questions
]

combined_lengths = [
    len(q["situation"]) + len(q["question"])
    for q in train_questions
]


def show_stats(name, values):
    print(f"\n{name}")
    print(f"  Minimum: {min(values)}")
    print(f"  Maximum: {max(values)}")
    print(f"  Mean:    {statistics.mean(values):.1f}")
    print(f"  Median:  {statistics.median(values):.1f}")
    print(f"  95th %:  {statistics.quantiles(values, n=20)[18]:.1f}")


print("\n" + "=" * 70)
print("TEXT LENGTH ANALYSIS")
print("=" * 70)

show_stats("Situation length", situation_lengths)
show_stats("Question length", question_lengths)
show_stats("Situation + question", combined_lengths)


# ---------------------------------------------------------
# 3. Alternative situations
# ---------------------------------------------------------

alternative_counts = [
    len(q.get("alternative_situation", []))
    for q in train_questions
]

print("\n" + "=" * 70)
print("ALTERNATIVE SITUATIONS")
print("=" * 70)

print("Examples with alternatives:",
      sum(x > 0 for x in alternative_counts),
      "/",
      len(train_questions))

print("Maximum alternatives:",
      max(alternative_counts))

print("Average alternatives:",
      statistics.mean(alternative_counts))

REPRESENTATIVE TRAINING EXAMPLES

--- Example 0 ---
Scene: scene0380_00
Question ID: 220602000000
Situation: I am facing a window and there is a desk on my right and a chair behind me.
Question: What color is the desk to my right?
Answer: brown
Position: {'x': -0.9651003385573296, 'y': -1.2417634435553606, 'z': 0}
Rotation: {'_x': 0, '_y': 0, '_z': 0.09983341664682724, '_w': 0.9950041652780182}

--- Example 100 ---
Scene: scene0058_00
Question ID: 220602000124
Situation: I am sitting on a chair and there is one chair on my left and another chair on my right.
Question: Is the amount of chair I am facing odd or even?
Answer: odd
Position: {'x': -0.3170599249628107, 'y': 1.3800116841063363, 'z': 0}
Rotation: {'_x': 0, '_y': 0, '_z': -0.7568024953079273, '_w': -0.6536436208636103}

--- Example 1000 ---
Scene: scene0506_00
Question ID: 220602001264
Situation: I am putting the mirror on the wall.
Question: Which way would I turn to get to the door to exit?
Answer: backward
Position: {'x': 0.

In [ ]:
import re
from collections import Counter

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def answer_in_context(record):
    context = normalize_text(
        record["situation"] + " " + record["question"]
    )

    answer = normalize_text(record["answer"])

    return answer in context


# Build temporary records from the training split
diagnostic_records = []

for q, a in zip(train_questions, train_annotations):
    diagnostic_records.append({
        "scene_id": q["scene_id"],
        "question": q["question"],
        "situation": q["situation"],
        "answer": a["answers"][0]["answer"],
    })


# ---------------------------------------------------------
# Check whether the answer literally appears in the
# situation + question.
# ---------------------------------------------------------

contained = [
    r for r in diagnostic_records
    if answer_in_context(r)
]

not_contained = [
    r for r in diagnostic_records
    if not answer_in_context(r)
]

print("=" * 70)
print("ANSWER-IN-CONTEXT DIAGNOSTIC")
print("=" * 70)

print(f"Total training examples: {len(diagnostic_records):,}")
print(f"Answer appears in context: {len(contained):,}")
print(f"Answer NOT in context: {len(not_contained):,}")

print(
    f"\nAnswer explicitly present: "
    f"{len(contained) / len(diagnostic_records) * 100:.2f}%"
)

print(
    f"Answer requires information not literally present: "
    f"{len(not_contained) / len(diagnostic_records) * 100:.2f}%"
)


# ---------------------------------------------------------
# Show examples where answer is NOT explicitly present
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXAMPLES REQUIRING INFERENCE / EXTERNAL SCENE INFORMATION")
print("=" * 70)

for i, r in enumerate(not_contained[:10], 1):

    print(f"\n--- Example {i} ---")
    print("Scene:", r["scene_id"])
    print("Situation:", r["situation"])
    print("Question:", r["question"])
    print("Answer:", r["answer"])


# ---------------------------------------------------------
# Most common answers among those NOT in context
# ---------------------------------------------------------

missing_answer_counts = Counter(
    normalize_text(r["answer"])
    for r in not_contained
)

print("\n" + "=" * 70)
print("TOP ANSWERS NOT EXPLICITLY PRESENT IN CONTEXT")
print("=" * 70)

for answer, count in missing_answer_counts.most_common(20):
    print(f"  {answer}: {count:,}")

ANSWER-IN-CONTEXT DIAGNOSTIC
Total training examples: 26,623
Answer appears in context: 7,370
Answer NOT in context: 19,253

Answer explicitly present: 27.68%
Answer requires information not literally present: 72.32%

EXAMPLES REQUIRING INFERENCE / EXTERNAL SCENE INFORMATION

--- Example 1 ---
Scene: scene0380_00
Situation: I am facing a window and there is a desk on my right and a chair behind me.
Question: What color is the desk to my right?
Answer: brown

--- Example 2 ---
Scene: scene0480_00
Situation: I am sitting on the edge of the couch with a curtain right next to me on the left.
Question: What is on the 12 o'clock of the coffee table that is on my 1 o'clock?
Answer: TV stand

--- Example 3 ---
Scene: scene0642_00
Situation: I am drying my hand with the towel.
Question: What color is the wall of the room I am in?
Answer: yellow

--- Example 4 ---
Scene: scene0489_00
Situation: I am throwing something out and I am between two office chairs.
Question: What color is the desk to my

In [ ]:
import re
from collections import Counter

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def answer_in_all_context(record):
    context_parts = [
        record["situation"],
        record["question"]
    ]

    context_parts.extend(
        record.get("alternative_situation", [])
    )

    context = normalize_text(" ".join(context_parts))
    answer = normalize_text(record["answer"])

    return answer in context


# Build records
records = []

for q, a in zip(train_questions, train_annotations):

    records.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "alternative_situation": q.get("alternative_situation", []),
        "question": q["question"],
        "answer": a["answers"][0]["answer"]
    })


contained = [
    r for r in records
    if answer_in_all_context(r)
]

not_contained = [
    r for r in records
    if not answer_in_all_context(r)
]


print("=" * 70)
print("ALTERNATIVE-SITUATION DIAGNOSTIC")
print("=" * 70)

print(f"Total examples: {len(records):,}")
print(f"Answer found somewhere in ALL text: {len(contained):,}")
print(f"Answer still missing: {len(not_contained):,}")

print(
    f"\nAnswer recoverable from text: "
    f"{len(contained) / len(records) * 100:.2f}%"
)

print(
    f"Answer still requires external scene information: "
    f"{len(not_contained) / len(records) * 100:.2f}%"
)


print("\n" + "=" * 70)
print("EXAMPLES WHERE ALTERNATIVES RECOVER THE ANSWER")
print("=" * 70)

shown = 0

for r in records:
    original_context = normalize_text(
        r["situation"] + " " + r["question"]
    )

    answer = normalize_text(r["answer"])

    # Answer wasn't in original context
    # but appears in one of the alternatives
    if answer not in original_context:

        alternatives = r.get("alternative_situation", [])

        matched = [
            alt for alt in alternatives
            if answer in normalize_text(alt)
        ]

        if matched:
            print(f"\nScene: {r['scene_id']}")
            print("Situation:", r["situation"])
            print("Question:", r["question"])
            print("Answer:", r["answer"])
            print("Alternative containing answer:", matched[0])

            shown += 1

            if shown >= 5:
                break

ALTERNATIVE-SITUATION DIAGNOSTIC
Total examples: 26,623
Answer found somewhere in ALL text: 9,729
Answer still missing: 16,894

Answer recoverable from text: 36.54%
Answer still requires external scene information: 63.46%

EXAMPLES WHERE ALTERNATIVES RECOVER THE ANSWER

Scene: scene0003_00
Situation: I am throwing trash with the microwave on my left.
Question: Where are the kitchen cabinets?
Answer: right
Alternative containing answer: I am throwing trash away right of the cabinets right of the water cooler I am using to fill my cup.

Scene: scene0515_00
Situation: I am standing in front of the microwave waiting for the popcorn to be ready.
Question: Which object behind me may I use to boil a kettle?
Answer: stove
Alternative containing answer: I am taking the sauce out of the microwave while the noodles boil on the stove behind me.

Scene: scene0410_00
Situation: I am facing the bathtub and there is a door on my right.
Question: What is below the bar that is in front of me?
Answer: so

In [ ]:
def get_scene_ids(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    return set(q["scene_id"] for q in questions)


train_scenes = get_scene_ids("train")
val_scenes = get_scene_ids("val")
test_scenes = get_scene_ids("test")


train_val = train_scenes & val_scenes
train_test = train_scenes & test_scenes
val_test = val_scenes & test_scenes


print("=" * 70)
print("SCENE SPLIT INTEGRITY CHECK")
print("=" * 70)

print(f"Train scenes:      {len(train_scenes):,}")
print(f"Validation scenes: {len(val_scenes):,}")
print(f"Test scenes:       {len(test_scenes):,}")

print("\nScene overlap:")
print(f"Train ↔ Validation: {len(train_val)}")
print(f"Train ↔ Test:       {len(train_test)}")
print(f"Validation ↔ Test:  {len(val_test)}")


assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0

print("\n✅ STEP 8 PASSED — NO SCENE LEAKAGE BETWEEN SPLITS.")

SCENE SPLIT INTEGRITY CHECK
Train scenes:      518
Validation scenes: 65
Test scenes:       67

Scene overlap:
Train ↔ Validation: 0
Train ↔ Test:       0
Validation ↔ Test:  0

✅ STEP 8 PASSED — NO SCENE LEAKAGE BETWEEN SPLITS.


In [ ]:
def build_example(split, index):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    q = questions[index]
    a = annotations[index]

    # Convert quaternion to a compact readable representation
    rotation = a["rotation"]
    position = a["position"]

    example = {
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ],
        "question": q["question"],
        "answer": a["answers"][0]["answer"]
    }

    return example


# Inspect three training examples
for i in range(3):
    example = build_example("train", i)

    print("=" * 70)
    print(f"EXAMPLE {i + 1}")
    print("=" * 70)

    for key, value in example.items():
        print(f"{key}: {value}")


EXAMPLE 1
scene_id: scene0380_00
situation: I am facing a window and there is a desk on my right and a chair behind me.
position: [-0.9651003385573296, -1.2417634435553606, 0]
rotation: [0, 0, 0.09983341664682724, 0.9950041652780182]
question: What color is the desk to my right?
answer: brown
EXAMPLE 2
scene_id: scene0480_00
situation: I am sitting on the edge of the couch with a curtain right next to me on the left.
position: [-1.571828073249186, 0.08811655769195924, 0]
rotation: [0, 0, 0.8632093666488773, -0.5048461045998595]
question: What is on the 12 o'clock of the coffee table that is on my 1 o'clock?
answer: TV stand
EXAMPLE 3
scene_id: scene0642_00
situation: I am drying my hand with the towel.
position: [-2.3076532505017173, -0.9447763452734602, 0]
rotation: [0, 0, -0.6754631805511526, -0.7373937155412478]
question: What color is the wall of the room I am in?
answer: yellow


In [ ]:
def format_for_gemma(example):
    return (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}\n\n"
        "Answer:"
    )


# Build one formatted example
example = build_example("train", 0)
formatted = format_for_gemma(example)

print("=" * 70)
print("GEMMA TRAINING FORMAT")
print("=" * 70)
print(formatted)
print(example["answer"])

GEMMA TRAINING FORMAT
You are a spatial reasoning assistant.

Situation:
I am facing a window and there is a desk on my right and a chair behind me.

Agent position:
[-0.9651003385573296, -1.2417634435553606, 0]

Agent rotation:
[0, 0, 0.09983341664682724, 0.9950041652780182]

Question:
What color is the desk to my right?

Answer:
brown


In [ ]:
# STEP 11 — Prepare evaluation examples

def get_split_examples(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    examples = []

    for q, a in zip(questions, annotations):
        position = a["position"]
        rotation = a["rotation"]

        examples.append({
            "scene_id": q["scene_id"],
            "situation": q["situation"],
            "position": [
                position["x"],
                position["y"],
                position["z"]
            ],
            "rotation": [
                rotation["_x"],
                rotation["_y"],
                rotation["_z"],
                rotation["_w"]
            ],
            "question": q["question"],
            "answer": a["answers"][0]["answer"]
        })

    return examples


# Load validation set
val_examples = get_split_examples("val")

print("=" * 70)
print("VALIDATION SET")
print("=" * 70)
print(f"Validation examples: {len(val_examples):,}")
print(f"Expected:            3,261")

assert len(val_examples) == 3261

print("\nFirst validation example:")
print("-" * 70)

print(format_for_gemma(val_examples[0]))
print(f"\nTarget answer: {val_examples[0]['answer']}")

print("\n✅ STEP 11 PASSED — VALIDATION SET READY.")

VALIDATION SET
Validation examples: 3,261
Expected:            3,261

First validation example:
----------------------------------------------------------------------
You are a spatial reasoning assistant.

Situation:
I am wiping a table that has a trash can next to it, while having a large table behind me with lots of chairs.

Agent position:
[-1.612321909455232, 3.8766019062927524, 0]

Agent rotation:
[0, 0, 0.9436221923009414, -0.33102440725287985]

Question:
Which direction should I toss a used napkin?

Answer:

Target answer: right

✅ STEP 11 PASSED — VALIDATION SET READY.


In [ ]:
# STEP 12 — Load Gemma 2B in 4-bit

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True,
)

print("=" * 70)
print("MODEL LOADED")
print("=" * 70)
print(f"Model: {MODEL_NAME}")
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Max sequence length: 512")
print(f"4-bit quantization: True")
print(f"Device: {model.device}")

print("\n✅ STEP 12 PASSED — GEMMA 2B LOADED.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


MODEL LOADED
Model: unsloth/gemma-2-2b-it-bnb-4bit
Tokenizer: GemmaTokenizer
Max sequence length: 512
4-bit quantization: True
Device: cuda:0

✅ STEP 12 PASSED — GEMMA 2B LOADED.


In [ ]:
# STEP 13 — Zero-shot generation sanity check

from transformers import TextStreamer

FastLanguageModel.for_inference(model)

def generate_answer(example):
    prompt = format_for_gemma(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer


# Test 5 validation examples
print("=" * 70)
print("ZERO-SHOT BASELINE — 5 EXAMPLES")
print("=" * 70)

for i in range(5):
    example = val_examples[i]
    prediction = generate_answer(example)

    print(f"\nExample {i + 1}")
    print("-" * 70)
    print(f"Question:  {example['question']}")
    print(f"Expected:  {example['answer']}")
    print(f"Predicted: {prediction}")

print("\n✅ STEP 13 COMPLETE — ZERO-SHOT GENERATION TESTED.")

ZERO-SHOT BASELINE — 5 EXAMPLES


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 1
----------------------------------------------------------------------
Question:  Which direction should I toss a used napkin?
Expected:  right
Predicted: 

Example 2
----------------------------------------------------------------------
Question:  Is the amount of cabinet I am facing odd or even?
Expected:  odd
Predicted: 


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 3
----------------------------------------------------------------------
Question:  What is on the right side of the soap dispenser in front of me?
Expected:  mirror
Predicted: 

Example 4
----------------------------------------------------------------------
Question:  How many drawers are in the cabinet in front of me?
Expected:  two
Predicted: 

Example 5
----------------------------------------------------------------------
Question:  What color is the door that I am facing?
Expected:  white
Predicted: 

✅ STEP 13 COMPLETE — ZERO-SHOT GENERATION TESTED.


In [ ]:
# STEP 13A — Native Gemma chat-template test

def build_chat_prompt(example):
    user_message = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_chat_answer(example):
    prompt = build_chat_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prompt, answer


# Test ONE example first
example = val_examples[0]

prompt, prediction = generate_chat_answer(example)

print("=" * 70)
print("NATIVE GEMMA CHAT TEMPLATE TEST")
print("=" * 70)

print("\nFormatted prompt:")
print("-" * 70)
print(prompt)

print("\nExpected answer:")
print(example["answer"])

print("\nModel prediction:")
print(prediction)

Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NATIVE GEMMA CHAT TEMPLATE TEST

Formatted prompt:
----------------------------------------------------------------------
<bos><start_of_turn>user
You are a spatial reasoning assistant.

Situation:
I am wiping a table that has a trash can next to it, while having a large table behind me with lots of chairs.

Agent position:
[-1.612321909455232, 3.8766019062927524, 0]

Agent rotation:
[0, 0, 0.9436221923009414, -0.33102440725287985]

Question:
Which direction should I toss a used napkin?<end_of_turn>
<start_of_turn>model


Expected answer:
right

Model prediction:



In [ ]:
# STEP 13B — Diagnose empty generation

example = val_examples[0]
prompt = build_chat_prompt(example)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

print("=" * 70)
print("GENERATION DIAGNOSTICS")
print("=" * 70)

print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print(f"Attention mask tokens: {inputs['attention_mask'].sum().item()}")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True
    )

generated_tokens = outputs.sequences[0][inputs["input_ids"].shape[1]:]

print(f"\nGenerated token count: {len(generated_tokens)}")
print(f"Generated token IDs: {generated_tokens.tolist()}")

if len(generated_tokens) > 0:
    print("\nDecoded generated text:")
    print(repr(
        tokenizer.decode(
            generated_tokens,
            skip_special_tokens=False
        )
    ))

print("\nSpecial token IDs:")
print(f"EOS: {tokenizer.eos_token_id}")
print(f"PAD: {tokenizer.pad_token_id}")
print(f"BOS: {tokenizer.bos_token_id}")

Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATION DIAGNOSTICS
Input tokens: 159
Attention mask tokens: 159

Generated token count: 1
Generated token IDs: [1]

Decoded generated text:
'<eos>'

Special token IDs:
EOS: 1
PAD: 0
BOS: 2


In [ ]:
# STEP 13C — Clean generation configuration

model.generation_config.max_length = None
model.generation_config.max_new_tokens = 16
model.generation_config.do_sample = False

print("=" * 70)
print("GENERATION CONFIGURATION")
print("=" * 70)
print(f"max_length:      {model.generation_config.max_length}")
print(f"max_new_tokens:  {model.generation_config.max_new_tokens}")
print(f"do_sample:       {model.generation_config.do_sample}")
print(f"eos_token_id:    {model.generation_config.eos_token_id}")
print(f"pad_token_id:    {model.generation_config.pad_token_id}")

print("\n✅ STEP 13C PASSED — GENERATION CONFIG CLEANED.")

GENERATION CONFIGURATION
max_length:      None
max_new_tokens:  16
do_sample:       False
eos_token_id:    [1, 107]
pad_token_id:    0

✅ STEP 13C PASSED — GENERATION CONFIG CLEANED.


In [ ]:
# STEP 13D — Retest generation after configuration cleanup

example = val_examples[0]

prompt = build_chat_prompt(example)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        return_dict_in_generate=True
    )

generated_tokens = outputs.sequences[0][inputs["input_ids"].shape[1]:]

print("=" * 70)
print("POST-CONFIGURATION GENERATION TEST")
print("=" * 70)

print(f"Expected answer: {example['answer']}")
print(f"Generated IDs:   {generated_tokens.tolist()}")

print(
    f"Decoded output:  "
    f"{repr(tokenizer.decode(generated_tokens, skip_special_tokens=False))}"
)

print(
    f"Clean output:    "
    f"{repr(tokenizer.decode(generated_tokens, skip_special_tokens=True).strip())}"
)

POST-CONFIGURATION GENERATION TEST
Expected answer: right
Generated IDs:   [1]
Decoded output:  '<eos>'
Clean output:    ''


In [ ]:
# STEP 13E — Inspect first-token prediction

example = val_examples[0]
prompt = build_chat_prompt(example)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

# Logits for the final input position
next_token_logits = outputs.logits[:, -1, :]

# Get the 10 most likely next tokens
top_values, top_indices = torch.topk(next_token_logits, k=10, dim=-1)

print("=" * 70)
print("FIRST-TOKEN PREDICTION DIAGNOSTIC")
print("=" * 70)

print(f"Prompt tokens: {inputs['input_ids'].shape[1]}")

print("\nTop 10 predicted next tokens:")
print("-" * 70)

for rank, (token_id, logit) in enumerate(
    zip(top_indices[0], top_values[0]), start=1
):
    token_id = token_id.item()
    logit = logit.item()

    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False
    )

    print(
        f"{rank:2d}. "
        f"ID={token_id:<6d} "
        f"logit={logit:8.3f} "
        f"text={repr(token_text)}"
    )

print("\nEOS IDs:")
print(model.generation_config.eos_token_id)

print("\nExpected answer:")
print(example["answer"])

`use_return_dict` is deprecated! Use `return_dict` instead!


FIRST-TOKEN PREDICTION DIAGNOSTIC
Prompt tokens: 159

Top 10 predicted next tokens:
----------------------------------------------------------------------
 1. ID=141    logit=  30.000 text='    '
 2. ID=140    logit=  30.000 text='   '
 3. ID=111    logit=  30.000 text='\n\n\n\n'
 4. ID=139    logit=  30.000 text='  '
 5. ID=108    logit=  30.000 text='\n'
 6. ID=1      logit=  30.000 text='<eos>'
 7. ID=109    logit=  30.000 text='\n\n'
 8. ID=110    logit=  30.000 text='\n\n\n'
 9. ID=199    logit=  30.000 text='<strong>'
10. ID=476    logit=  30.000 text=' a'

EOS IDs:
[1, 107]

Expected answer:
right


In [ ]:
# STEP 13F — Check model numerical health

print("=" * 70)
print("MODEL NUMERICAL HEALTH CHECK")
print("=" * 70)

# Check a representative parameter tensor
param = next(model.parameters())

print(f"Parameter dtype: {param.dtype}")
print(f"Parameter device: {param.device}")

print(f"\nParameter statistics:")
print(f"  min:  {param.float().min().item():.6f}")
print(f"  max:  {param.float().max().item():.6f}")
print(f"  mean: {param.float().mean().item():.6f}")
print(f"  std:  {param.float().std().item():.6f}")

print(f"\nParameter contains NaN: {torch.isnan(param.float()).any().item()}")
print(f"Parameter contains Inf: {torch.isinf(param.float()).any().item()}")

# Check the logits from our validation prompt
example = val_examples[0]
prompt = build_chat_prompt(example)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :].float()

print("\nFinal-position logits:")
print(f"  min:  {logits.min().item():.6f}")
print(f"  max:  {logits.max().item():.6f}")
print(f"  mean: {logits.mean().item():.6f}")
print(f"  std:  {logits.std().item():.6f}")

print(f"\nLogits contain NaN: {torch.isnan(logits).any().item()}")
print(f"Logits contain Inf: {torch.isinf(logits).any().item()}")

# Count unique rounded logit values
rounded = torch.round(logits * 1000) / 1000
unique_values = torch.unique(rounded)

print(f"\nUnique logits (rounded to 3 decimals): {len(unique_values):,}")

print("\n✅ STEP 13F COMPLETE")

MODEL NUMERICAL HEALTH CHECK
Parameter dtype: torch.float16
Parameter device: cuda:0

Parameter statistics:
  min:  -2.093750
  max:  2.750000
  mean: 0.000242
  std:  0.037288

Parameter contains NaN: False
Parameter contains Inf: False

Final-position logits:
  min:  -30.000000
  max:  30.000000
  mean: -23.179102
  std:  11.208129

Logits contain NaN: False
Logits contain Inf: False

Unique logits (rounded to 3 decimals): 9,056

✅ STEP 13F COMPLETE


In [ ]:
# STEP 13G — Plain-language inference sanity check

test_messages = [
    {
        "role": "user",
        "content": "What is 2 + 2? Answer with only the number."
    }
]

test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

test_inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

print("=" * 70)
print("PLAIN GEMMA INFERENCE TEST")
print("=" * 70)

print("\nPrompt:")
print(test_prompt)

with torch.no_grad():
    test_outputs = model.generate(
        **test_inputs,
        max_new_tokens=16,
        do_sample=False
    )

test_generated = test_outputs[0][test_inputs["input_ids"].shape[1]:]

print("\nGenerated token IDs:")
print(test_generated.tolist())

print("\nRaw decoded output:")
print(repr(
    tokenizer.decode(
        test_generated,
        skip_special_tokens=False
    )
))

print("\nClean output:")
print(repr(
    tokenizer.decode(
        test_generated,
        skip_special_tokens=True
    ).strip()
))

PLAIN GEMMA INFERENCE TEST

Prompt:
<bos><start_of_turn>user
What is 2 + 2? Answer with only the number.<end_of_turn>
<start_of_turn>model


Generated token IDs:
[1]

Raw decoded output:
'<eos>'

Clean output:
''


In [ ]:
# STEP 14 — Compare against the standard Gemma checkpoint

import gc
import torch

print("=" * 70)
print("STEP 14 — LOADING REFERENCE GEMMA CHECKPOINT")
print("=" * 70)

# Release the current model
del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print(
    f"Free GPU memory before loading: "
    f"{torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB"
)

from transformers import AutoTokenizer, AutoModelForCausalLM

REFERENCE_MODEL = "google/gemma-2-2b-it"

ref_tokenizer = AutoTokenizer.from_pretrained(
    REFERENCE_MODEL
)

ref_model = AutoModelForCausalLM.from_pretrained(
    REFERENCE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\n" + "=" * 70)
print("REFERENCE MODEL LOADED")
print("=" * 70)
print(f"Model: {REFERENCE_MODEL}")
print(f"Tokenizer: {ref_tokenizer.__class__.__name__}")
print(f"Device: {ref_model.device}")

print("\n✅ Reference checkpoint loaded.")

STEP 14 — LOADING REFERENCE GEMMA CHECKPOINT
Free GPU memory before loading: 13.07 GB


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6aa19626-46f08f9a677dc0c455a799f8;d5b33fbe-fca9-4b17-91bd-99ab7cd4afcc)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# STEP 14A — Load ungated reference model

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

REFERENCE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

ref_tokenizer = AutoTokenizer.from_pretrained(
    REFERENCE_MODEL
)

ref_model = AutoModelForCausalLM.from_pretrained(
    REFERENCE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("=" * 70)
print("UNGATED REFERENCE MODEL LOADED")
print("=" * 70)
print(f"Model: {REFERENCE_MODEL}")
print(f"Tokenizer: {ref_tokenizer.__class__.__name__}")
print(f"Device: {ref_model.device}")

print("\n✅ STEP 14A PASSED")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

UNGATED REFERENCE MODEL LOADED
Model: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer: Qwen2Tokenizer
Device: cuda:0

✅ STEP 14A PASSED


In [ ]:
# STEP 14B — Reference model inference sanity check

messages = [
    {
        "role": "user",
        "content": "What is 2 + 2? Answer with only the number."
    }
]

prompt = ref_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = ref_tokenizer(
    prompt,
    return_tensors="pt"
).to(ref_model.device)

with torch.no_grad():
    outputs = ref_model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print("=" * 70)
print("REFERENCE MODEL INFERENCE TEST")
print("=" * 70)

print(f"\nGenerated token IDs:")
print(generated_tokens.tolist())

print("\nRaw output:")
print(repr(
    ref_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )
))

print("\nClean output:")
print(repr(
    ref_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()
))

REFERENCE MODEL INFERENCE TEST

Generated token IDs:
[19, 151645]

Raw output:
'4<|im_end|>'

Clean output:
'4'


In [ ]:
# STEP 15A — Release reference model and prepare for Gemma reload

import gc
import torch

del ref_model
del ref_tokenizer

gc.collect()
torch.cuda.empty_cache()

print("=" * 70)
print("GPU CLEANUP")
print("=" * 70)
print(
    f"Free GPU memory: "
    f"{torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB"
)

print("\n✅ STEP 15A PASSED")

GPU CLEANUP
Free GPU memory: 13.17 GB

✅ STEP 15A PASSED


In [ ]:
# STEP 15B — Reload Gemma and inspect configuration

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True,
)

print("=" * 70)
print("GEMMA CONFIGURATION INSPECTION")
print("=" * 70)

print(f"Model: {MODEL_NAME}")
print(f"Model class: {model.__class__.__name__}")
print(f"Tokenizer class: {tokenizer.__class__.__name__}")

print("\nTokenizer special tokens:")
print(f"  BOS token:     {repr(tokenizer.bos_token)}")
print(f"  BOS ID:        {tokenizer.bos_token_id}")
print(f"  EOS token:     {repr(tokenizer.eos_token)}")
print(f"  EOS ID:        {tokenizer.eos_token_id}")
print(f"  PAD token:     {repr(tokenizer.pad_token)}")
print(f"  PAD ID:        {tokenizer.pad_token_id}")

print("\nGeneration configuration:")
print(f"  max_length:        {model.generation_config.max_length}")
print(f"  max_new_tokens:    {model.generation_config.max_new_tokens}")
print(f"  eos_token_id:      {model.generation_config.eos_token_id}")
print(f"  pad_token_id:      {model.generation_config.pad_token_id}")
print(f"  bos_token_id:      {model.generation_config.bos_token_id}")

print("\nModel configuration:")
print(f"  vocab_size:        {model.config.vocab_size}")
print(f"  hidden_size:       {model.config.hidden_size}")
print(f"  num_layers:        {model.config.num_hidden_layers}")

print("\nGPU:")
print(f"  Device:            {model.device}")
print(
    f"  Free memory:       "
    f"{torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB"
)

print("\n✅ STEP 15B COMPLETE")

==((====))==  Unsloth 2026.9.4: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


GEMMA CONFIGURATION INSPECTION
Model: unsloth/gemma-2-2b-it-bnb-4bit
Model class: Gemma2ForCausalLM
Tokenizer class: GemmaTokenizer

Tokenizer special tokens:
  BOS token:     '<bos>'
  BOS ID:        2
  EOS token:     '<eos>'
  EOS ID:        1
  PAD token:     '<pad>'
  PAD ID:        0

Generation configuration:
  max_length:        8192
  max_new_tokens:    None
  eos_token_id:      [1, 107]
  pad_token_id:      0
  bos_token_id:      2

Model configuration:
  vocab_size:        256000
  hidden_size:       2304
  num_layers:        26

GPU:
  Device:            cuda:0
  Free memory:       11.12 GB

✅ STEP 15B COMPLETE


In [ ]:
# STEP 15C — Direct probability check

example = val_examples[0]

messages = [
    {
        "role": "user",
        "content": (
            "You are a spatial reasoning assistant.\n\n"
            f"Situation:\n{example['situation']}\n\n"
            f"Agent position:\n{example['position']}\n\n"
            f"Agent rotation:\n{example['rotation']}\n\n"
            f"Question:\n{example['question']}"
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True
    )

logits = outputs.logits[0, -1].float()

# Convert logits to probabilities
probs = torch.softmax(logits, dim=-1)

# Top 10 probabilities
top_probs, top_ids = torch.topk(probs, k=10)

print("=" * 70)
print("DIRECT NEXT-TOKEN PROBABILITY CHECK")
print("=" * 70)

print(f"Input tokens: {inputs['input_ids'].shape[1]}")

print("\nTop 10 next-token predictions:")
print("-" * 70)

for rank, (token_id, probability) in enumerate(
    zip(top_ids, top_probs), start=1
):
    token_id = token_id.item()
    probability = probability.item()

    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False
    )

    print(
        f"{rank:2d}. "
        f"ID={token_id:<6d} "
        f"prob={probability:.8f} "
        f"text={repr(token_text)}"
    )

print("\nEOS probability:")
print(f"  EOS (1):   {probs[1].item():.8f}")
print(f"  EOS (107): {probs[107].item():.8f}")

print("\nExpected answer:")
print(example["answer"])

DIRECT NEXT-TOKEN PROBABILITY CHECK
Input tokens: 159

Top 10 next-token predictions:
----------------------------------------------------------------------
 1. ID=141    prob=0.00148028 text='    '
 2. ID=140    prob=0.00148028 text='   '
 3. ID=111    prob=0.00148028 text='\n\n\n\n'
 4. ID=139    prob=0.00148028 text='  '
 5. ID=108    prob=0.00148028 text='\n'
 6. ID=1      prob=0.00148028 text='<eos>'
 7. ID=109    prob=0.00148028 text='\n\n'
 8. ID=110    prob=0.00148028 text='\n\n\n'
 9. ID=199    prob=0.00148028 text='<strong>'
10. ID=476    prob=0.00148028 text=' a'

EOS probability:
  EOS (1):   0.00148028
  EOS (107): 0.00000000

Expected answer:
right


In [ ]:
# STEP 16 — Raw logits argmax test

example = val_examples[0]

messages = [
    {
        "role": "user",
        "content": (
            "You are a spatial reasoning assistant.\n\n"
            f"Situation:\n{example['situation']}\n\n"
            f"Agent position:\n{example['position']}\n\n"
            f"Agent rotation:\n{example['rotation']}\n\n"
            f"Question:\n{example['question']}"
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True
    )

logits = outputs.logits[0, -1].float()

best_token_id = torch.argmax(logits).item()
best_logit = logits[best_token_id].item()

best_token = tokenizer.decode(
    [best_token_id],
    skip_special_tokens=False
)

print("=" * 70)
print("RAW LOGITS ARGMAX TEST")
print("=" * 70)

print(f"Best token ID: {best_token_id}")
print(f"Best token text: {repr(best_token)}")
print(f"Best logit: {best_logit}")

print("\nEOS comparison:")
print(f"EOS 1 logit:   {logits[1].item()}")
print(f"EOS 107 logit: {logits[107].item()}")

print("\nDifference:")
print(
    f"Best token - EOS(1): "
    f"{best_logit - logits[1].item():.6f}"
)

print("\nExpected answer:")
print(example["answer"])

RAW LOGITS ARGMAX TEST
Best token ID: 1
Best token text: '<eos>'
Best logit: 30.0

EOS comparison:
EOS 1 logit:   30.0
EOS 107 logit: -30.0

Difference:
Best token - EOS(1): 0.000000

Expected answer:
right


In [ ]:
from huggingface_hub import login

login()

In [ ]:
# STEP 18 — Load official Gemma 2 2B Instruct

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

OFFICIAL_MODEL = "google/gemma-2-2b-it"

print("=" * 70)
print("LOADING OFFICIAL GEMMA 2 2B INSTRUCT")
print("=" * 70)

official_tokenizer = AutoTokenizer.from_pretrained(
    OFFICIAL_MODEL
)

official_model = AutoModelForCausalLM.from_pretrained(
    OFFICIAL_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

print("\n" + "=" * 70)
print("OFFICIAL GEMMA LOADED")
print("=" * 70)

print(f"Model: {OFFICIAL_MODEL}")
print(f"Tokenizer: {official_tokenizer.__class__.__name__}")
print(f"Model class: {official_model.__class__.__name__}")
print(f"Device: {official_model.device}")
print(f"Vocab size: {official_model.config.vocab_size}")

print("\nTokenizer:")
print(f"  BOS: {repr(official_tokenizer.bos_token)} "
      f"(ID {official_tokenizer.bos_token_id})")
print(f"  EOS: {repr(official_tokenizer.eos_token)} "
      f"(ID {official_tokenizer.eos_token_id})")
print(f"  PAD: {repr(official_tokenizer.pad_token)} "
      f"(ID {official_tokenizer.pad_token_id})")

print("\nGeneration config:")
print(f"  max_length: {official_model.generation_config.max_length}")
print(f"  eos_token_id: {official_model.generation_config.eos_token_id}")
print(f"  pad_token_id: {official_model.generation_config.pad_token_id}")

print("\nGPU memory:")
print(
    f"  Free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB"
)

print("\n✅ STEP 18 PASSED — OFFICIAL GEMMA LOADED.")

LOADING OFFICIAL GEMMA 2 2B INSTRUCT


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]


OFFICIAL GEMMA LOADED
Model: google/gemma-2-2b-it
Tokenizer: GemmaTokenizer
Model class: Gemma2ForCausalLM
Device: cuda:0
Vocab size: 256000

Tokenizer:
  BOS: '<bos>' (ID 2)
  EOS: '<eos>' (ID 1)
  PAD: '<pad>' (ID 0)

Generation config:
  max_length: None
  eos_token_id: [1, 107]
  pad_token_id: 0

GPU memory:
  Free: 6.11 GB

✅ STEP 18 PASSED — OFFICIAL GEMMA LOADED.


In [ ]:
# STEP 18A — Official Gemma inference sanity check

messages = [
    {
        "role": "user",
        "content": "What is 2 + 2? Answer with only the number."
    }
]

prompt = official_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = official_tokenizer(
    prompt,
    return_tensors="pt"
).to(official_model.device)

with torch.no_grad():
    outputs = official_model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print("=" * 70)
print("OFFICIAL GEMMA INFERENCE TEST")
print("=" * 70)

print("\nGenerated token IDs:")
print(generated_tokens.tolist())

print("\nRaw output:")
print(repr(
    official_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )
))

print("\nClean output:")
print(repr(
    official_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()
))

AttributeError: 'Gemma2Model' object has no attribute 'max_seq_length'

In [ ]:
# STEP 20 — Clean Gemma inference test
# IMPORTANT: Do NOT import Unsloth in this runtime.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b-it"

print("=" * 70)
print("LOADING OFFICIAL GEMMA — CLEAN TRANSFORMERS RUNTIME")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

print("\nModel loaded.")
print(f"Model class: {model.__class__.__name__}")
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Device: {model.device}")

# ------------------------------------------------------------
# Simple inference test
# ------------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": "What is 2 + 2? Answer with only the number."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

raw_output = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=False
)

clean_output = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("\n" + "=" * 70)
print("CLEAN GEMMA INFERENCE TEST")
print("=" * 70)

print(f"Generated token IDs: {generated_tokens.tolist()}")
print(f"Raw output:          {repr(raw_output)}")
print(f"Clean output:        {repr(clean_output)}")

print("\n" + "=" * 70)

if clean_output:
    print("✅ STEP 20 PASSED — OFFICIAL GEMMA GENERATES TEXT.")
else:
    print("❌ STEP 20 FAILED — GEMMA STILL RETURNS EMPTY OUTPUT.")

LOADING OFFICIAL GEMMA — CLEAN TRANSFORMERS RUNTIME


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


Model loaded.
Model class: Gemma2ForCausalLM
Tokenizer: GemmaTokenizer
Device: cuda:0

CLEAN GEMMA INFERENCE TEST
Generated token IDs: [235310, 235248, 108, 107]
Raw output:          '4 \n<end_of_turn>'
Clean output:        '4'

✅ STEP 20 PASSED — OFFICIAL GEMMA GENERATES TEXT.


In [ ]:
# STEP 21 — Check standard 4-bit quantization support

import bitsandbytes as bnb

print("=" * 70)
print("BITSANDBYTES CHECK")
print("=" * 70)

print(f"bitsandbytes version: {bnb.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")

print("\nCUDA device:")
print(torch.cuda.get_device_name(0))

print("\nGPU memory:")
free_memory, total_memory = torch.cuda.mem_get_info()
print(f"Free:  {free_memory / 1024**3:.2f} GB")
print(f"Total: {total_memory / 1024**3:.2f} GB")

print("\n✅ STEP 21 PASSED — STANDARD 4-BIT SUPPORT CHECK COMPLETE.")

BITSANDBYTES CHECK
bitsandbytes version: 0.50.2
CUDA available:       True

CUDA device:
Tesla T4

GPU memory:
Free:  9.54 GB
Total: 14.56 GB

✅ STEP 21 PASSED — STANDARD 4-BIT SUPPORT CHECK COMPLETE.


In [ ]:
# STEP 22 — Load official Gemma in standard 4-bit

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Release the FP16 model
del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("=" * 70)
print("LOADING OFFICIAL GEMMA IN STANDARD 4-BIT")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("\n" + "=" * 70)
print("4-BIT GEMMA LOADED")
print("=" * 70)

print(f"Model class: {model.__class__.__name__}")
print(f"Tokenizer:   {tokenizer.__class__.__name__}")
print(f"Device:      {model.device}")
print(f"4-bit:       {getattr(model, "is_loaded_in_4bit", "unknown")}")

free_memory, total_memory = torch.cuda.mem_get_info()

print(f"\nGPU memory:")
print(f"Free:  {free_memory / 1024**3:.2f} GB")
print(f"Total: {total_memory / 1024**3:.2f} GB")

print("\n✅ STEP 22 PASSED — OFFICIAL GEMMA LOADED IN STANDARD 4-BIT.")

LOADING OFFICIAL GEMMA IN STANDARD 4-BIT


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


4-BIT GEMMA LOADED
Model class: Gemma2ForCausalLM
Tokenizer:   GemmaTokenizer
Device:      cuda:0
4-bit:       True

GPU memory:
Free:  12.30 GB
Total: 14.56 GB

✅ STEP 22 PASSED — OFFICIAL GEMMA LOADED IN STANDARD 4-BIT.


In [ ]:
# STEP 23 — Attach LoRA adapters

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

print("=" * 70)
print("LORA CONFIGURATION")
print("=" * 70)

print(f"Rank (r):       {lora_config.r}")
print(f"Alpha:          {lora_config.lora_alpha}")
print(f"Dropout:        {lora_config.lora_dropout}")
print(f"Target modules: {lora_config.target_modules}")

print("\nTrainable parameters:")
model.print_trainable_parameters()

print("\n✅ STEP 23 PASSED — LORA ADAPTERS ATTACHED.")

LORA CONFIGURATION
Rank (r):       16
Alpha:          32
Dropout:        0.05
Target modules: {'gate_proj', 'v_proj', 'down_proj', 'o_proj', 'k_proj', 'q_proj', 'up_proj'}

Trainable parameters:
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881

✅ STEP 23 PASSED — LORA ADAPTERS ATTACHED.


In [ ]:
# STEP 24 — Restore dataset loader and inspect training record

import os
import json

DATA_DIR = "/content/egospatial"


def load_json(filename):
    path = os.path.join(DATA_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_split_examples(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    assert len(questions) == len(annotations)

    examples = []

    for q, a in zip(questions, annotations):
        position = a["position"]
        rotation = a["rotation"]

        examples.append({
            "scene_id": q["scene_id"],
            "situation": q["situation"],
            "position": [
                position["x"],
                position["y"],
                position["z"]
            ],
            "rotation": [
                rotation["_x"],
                rotation["_y"],
                rotation["_z"],
                rotation["_w"]
            ],
            "question": q["question"],
            "answer": a["answers"][0]["answer"]
        })

    return examples


def make_training_text(example):
    prompt = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    answer = example["answer"]

    return prompt, answer


# Load training examples
train_examples = get_split_examples("train")

print("=" * 70)
print("TRAINING DATA RESTORED")
print("=" * 70)
print(f"Training examples: {len(train_examples):,}")
print("Expected:          26,623")

assert len(train_examples) == 26623

# Inspect first example
train_example = train_examples[0]
prompt_text, answer_text = make_training_text(train_example)

print("\n" + "=" * 70)
print("TRAINING RECORD INSPECTION")
print("=" * 70)

print("\nPROMPT:")
print("-" * 70)
print(prompt_text)

print("\nTARGET ANSWER:")
print("-" * 70)
print(repr(answer_text))

# Tokenize separately
prompt_tokens = tokenizer(
    prompt_text,
    add_special_tokens=True
)["input_ids"]

answer_tokens = tokenizer(
    answer_text,
    add_special_tokens=False
)["input_ids"]

print("\nTOKEN COUNTS:")
print("-" * 70)
print(f"Prompt tokens: {len(prompt_tokens)}")
print(f"Answer tokens: {len(answer_tokens)}")
print(f"Total:         {len(prompt_tokens) + len(answer_tokens)}")

print("\nAnswer token IDs:")
print(answer_tokens)

print("\nDecoded answer:")
print(
    repr(
        tokenizer.decode(
            answer_tokens,
            skip_special_tokens=False
        )
    )
)

print("\n✅ STEP 24 PASSED — TRAINING DATA RESTORED AND RECORD INSPECTED.")

TRAINING DATA RESTORED
Training examples: 26,623
Expected:          26,623

TRAINING RECORD INSPECTION

PROMPT:
----------------------------------------------------------------------
You are a spatial reasoning assistant.

Situation:
I am facing a window and there is a desk on my right and a chair behind me.

Agent position:
[-0.9651003385573296, -1.2417634435553606, 0]

Agent rotation:
[0, 0, 0.09983341664682724, 0.9950041652780182]

Question:
What color is the desk to my right?

TARGET ANSWER:
----------------------------------------------------------------------
'brown'

TOKEN COUNTS:
----------------------------------------------------------------------
Prompt tokens: 144
Answer tokens: 1
Total:         145

Answer token IDs:
[24797]

Decoded answer:
'brown'

✅ STEP 24 PASSED — TRAINING DATA RESTORED AND RECORD INSPECTED.


In [ ]:
# STEP 25 — Build response-only supervised labels

IGNORE_INDEX = -100


def build_training_tokens(example):
    user_content = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]

    # Full conversation, including the answer
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Prompt only, ending immediately before the model response
    prompt_messages = [
        {
            "role": "user",
            "content": user_content
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    prompt_length = len(prompt_ids)

    # Everything before the answer is ignored.
    labels = (
        [IGNORE_INDEX] * prompt_length
        + full_ids[prompt_length:]
    )

    assert len(full_ids) == len(labels)
    assert any(label != IGNORE_INDEX for label in labels)

    return {
        "input_ids": full_ids,
        "labels": labels,
        "prompt_length": prompt_length,
        "answer": example["answer"],
    }


# Inspect one real training example
sample = build_training_tokens(train_examples[0])

print("=" * 70)
print("RESPONSE-ONLY LABEL INSPECTION")
print("=" * 70)

print(f"Answer:              {sample['answer']}")
print(f"Total tokens:        {len(sample['input_ids'])}")
print(f"Prompt tokens:       {sample['prompt_length']}")
print(
    f"Supervised tokens:   "
    f"{sum(x != IGNORE_INDEX for x in sample['labels'])}"
)

print("\nLast input tokens:")
print(sample["input_ids"][-10:])

print("\nLast labels:")
print(sample["labels"][-10:])

# Decode only the supervised portion
answer_token_ids = [
    token_id
    for token_id, label in zip(
        sample["input_ids"],
        sample["labels"]
    )
    if label != IGNORE_INDEX
]

print("\nDecoded supervised target:")
print(
    repr(
        tokenizer.decode(
            answer_token_ids,
            skip_special_tokens=False
        )
    )
)

print("\nFirst 20 labels:")
print(sample["labels"][:20])

print("\nIgnored labels in prompt:")
print(
    sum(label == IGNORE_INDEX for label in sample["labels"])
)

print("\n" + "=" * 70)
print("CHECKS")
print("=" * 70)

assert all(
    label == IGNORE_INDEX
    for label in sample["labels"][:sample["prompt_length"]]
)

assert sample["answer"] in tokenizer.decode(
    answer_token_ids,
    skip_special_tokens=True
)

print("✓ Prompt labels are masked")
print("✓ Answer labels are active")
print("✓ Target answer is recoverable")
print("\n✅ STEP 25 PASSED — RESPONSE-ONLY LABELING VERIFIED.")

RESPONSE-ONLY LABEL INSPECTION
Answer:              brown
Total tokens:        155
Prompt tokens:       152
Supervised tokens:   3

Last input tokens:
[1833, 235336, 107, 108, 106, 2516, 108, 24797, 107, 108]

Last labels:
[-100, -100, -100, -100, -100, -100, -100, 24797, 107, 108]

Decoded supervised target:
'brown<end_of_turn>\n'

First 20 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Ignored labels in prompt:
152

CHECKS
✓ Prompt labels are masked
✓ Answer labels are active
✓ Target answer is recoverable

✅ STEP 25 PASSED — RESPONSE-ONLY LABELING VERIFIED.


In [ ]:
# STEP 26 — Self-contained training sequence length audit

import os
import json
import torch

DATA_DIR = "/content/egospatial"


def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def get_split_examples(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    assert len(questions) == len(annotations)

    examples = []

    for q, a in zip(questions, annotations):
        position = a["position"]
        rotation = a["rotation"]

        examples.append({
            "scene_id": q["scene_id"],
            "situation": q["situation"],
            "position": [
                position["x"],
                position["y"],
                position["z"]
            ],
            "rotation": [
                rotation["_x"],
                rotation["_y"],
                rotation["_z"],
                rotation["_w"]
            ],
            "question": q["question"],
            "answer": a["answers"][0]["answer"]
        })

    return examples


def build_training_tokens(example):
    user_content = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    return full_ids


# Load training set
train_examples = get_split_examples("train")

print("=" * 70)
print("TRAINING SEQUENCE LENGTH AUDIT")
print("=" * 70)
print(f"Training examples loaded: {len(train_examples):,}")

assert len(train_examples) == 26623

# Measure tokenized lengths
lengths = []

for example in train_examples:
    token_ids = build_training_tokens(example)
    lengths.append(len(token_ids))

lengths_tensor = torch.tensor(lengths, dtype=torch.float32)

print(f"\nMinimum tokens: {min(lengths):,}")
print(f"Maximum tokens: {max(lengths):,}")
print(f"Mean tokens:    {sum(lengths) / len(lengths):.2f}")

print("\nPercentiles:")

for percentile in [50, 90, 95, 99, 99.5]:
    value = torch.quantile(
        lengths_tensor,
        percentile / 100
    ).item()

    print(f"  {percentile:>5}%: {value:.0f} tokens")

print("\nPotential truncation:")

for limit in [256, 384, 512]:
    count = sum(length > limit for length in lengths)

    print(
        f"  > {limit} tokens: "
        f"{count:,} "
        f"({count / len(lengths) * 100:.2f}%)"
    )

print("\n" + "=" * 70)
print("CHECK")
print("=" * 70)

assert len(lengths) == 26623

print("✓ All 26,623 examples analyzed")
print("\n✅ STEP 26 PASSED — SEQUENCE LENGTH AUDIT COMPLETE.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/egospatial/v1_balanced_questions_train_scannetv2.json'

In [ ]:
# STEP 25A — Restore EgoSpatial dataset files after runtime restart

import os
import requests

DATA_DIR = "/content/egospatial"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Platinum04/EgoSpatial-Dataset/main"
)

files = [
    "v1_balanced_questions_train_scannetv2.json",
    "v1_balanced_questions_val_scannetv2.json",
    "v1_balanced_questions_test_scannetv2.json",
    "v1_balanced_sqa_annotations_train_scannetv2.json",
    "v1_balanced_sqa_annotations_val_scannetv2.json",
    "v1_balanced_sqa_annotations_test_scannetv2.json",
]

print("=" * 70)
print("RESTORING EGOSPATIAL DATASET")
print("=" * 70)

for filename in files:
    url = f"{BASE_URL}/{filename}"
    output_path = os.path.join(DATA_DIR, filename)

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    print(
        f"✓ {filename} — "
        f"{len(response.content):,} bytes"
    )

print("\n" + "=" * 70)
print("VERIFYING FILES")
print("=" * 70)

for filename in files:
    path = os.path.join(DATA_DIR, filename)

    assert os.path.exists(path)
    assert os.path.getsize(path) > 0

    print(f"✓ {filename}")

print("\n✅ STEP 25A PASSED — ALL 6 DATASET FILES RESTORED.")

RESTORING EGOSPATIAL DATASET
✓ v1_balanced_questions_train_scannetv2.json — 11,140,002 bytes
✓ v1_balanced_questions_val_scannetv2.json — 1,385,508 bytes
✓ v1_balanced_questions_test_scannetv2.json — 1,460,185 bytes
✓ v1_balanced_sqa_annotations_train_scannetv2.json — 9,093,541 bytes
✓ v1_balanced_sqa_annotations_val_scannetv2.json — 1,114,948 bytes
✓ v1_balanced_sqa_annotations_test_scannetv2.json — 1,200,381 bytes

VERIFYING FILES
✓ v1_balanced_questions_train_scannetv2.json
✓ v1_balanced_questions_val_scannetv2.json
✓ v1_balanced_questions_test_scannetv2.json
✓ v1_balanced_sqa_annotations_train_scannetv2.json
✓ v1_balanced_sqa_annotations_val_scannetv2.json
✓ v1_balanced_sqa_annotations_test_scannetv2.json

✅ STEP 25A PASSED — ALL 6 DATASET FILES RESTORED.


In [ ]:
# STEP 26 — Training sequence length audit

import os
import json
import torch

DATA_DIR = "/content/egospatial"


def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def get_split_examples(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    assert len(questions) == len(annotations)

    examples = []

    for q, a in zip(questions, annotations):
        position = a["position"]
        rotation = a["rotation"]

        examples.append({
            "scene_id": q["scene_id"],
            "situation": q["situation"],
            "position": [
                position["x"],
                position["y"],
                position["z"]
            ],
            "rotation": [
                rotation["_x"],
                rotation["_y"],
                rotation["_z"],
                rotation["_w"]
            ],
            "question": q["question"],
            "answer": a["answers"][0]["answer"]
        })

    return examples


def build_training_tokens(example):
    user_content = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]


# Load training data
train_examples = get_split_examples("train")

print("=" * 70)
print("TRAINING SEQUENCE LENGTH AUDIT")
print("=" * 70)
print(f"Training examples loaded: {len(train_examples):,}")

assert len(train_examples) == 26623

# Measure every example
lengths = []

for example in train_examples:
    lengths.append(
        len(build_training_tokens(example))
    )

lengths_tensor = torch.tensor(
    lengths,
    dtype=torch.float32
)

print(f"\nMinimum tokens: {min(lengths):,}")
print(f"Maximum tokens: {max(lengths):,}")
print(f"Mean tokens:    {sum(lengths) / len(lengths):.2f}")

print("\nPercentiles:")

for percentile in [50, 90, 95, 99, 99.5]:
    value = torch.quantile(
        lengths_tensor,
        percentile / 100
    ).item()

    print(f"  {percentile:>5}%: {value:.0f} tokens")

print("\nPotential truncation:")

for limit in [256, 384, 512]:
    count = sum(length > limit for length in lengths)

    print(
        f"  > {limit} tokens: "
        f"{count:,} "
        f"({count / len(lengths) * 100:.2f}%)"
    )

print("\n" + "=" * 70)
print("CHECK")
print("=" * 70)

assert len(lengths) == 26623

print("✓ All 26,623 training examples analyzed")
print("\n✅ STEP 26 PASSED — SEQUENCE LENGTH AUDIT COMPLETE.")

TRAINING SEQUENCE LENGTH AUDIT
Training examples loaded: 26,623


NameError: name 'tokenizer' is not defined

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 70)
print("TOKENIZER RESTORED")
print("=" * 70)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Chat template available:", tokenizer.chat_template is not None)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6aa1a7bc-5ff2d1af42c7a7450a3e51aa;7fc22716-36f3-48fd-840f-94e05a3cce60)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami
from transformers import AutoTokenizer

print("Hugging Face user:", whoami()["name"])

MODEL_NAME = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 70)
print("TOKENIZER RESTORED")
print("=" * 70)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Chat template available:", tokenizer.chat_template is not None)

Hugging Face user: Platinum04


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

TOKENIZER RESTORED
Tokenizer: GemmaTokenizer
Vocab size: 256000
Chat template available: True


In [ ]:
import os
import json
import torch

DATA_DIR = "/content/egospatial"


def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def get_split_examples(split):
    questions = load_json(
        f"v1_balanced_questions_{split}_scannetv2.json"
    )["questions"]

    annotations = load_json(
        f"v1_balanced_sqa_annotations_{split}_scannetv2.json"
    )["annotations"]

    assert len(questions) == len(annotations)

    examples = []

    for q, a in zip(questions, annotations):
        position = a["position"]
        rotation = a["rotation"]

        examples.append({
            "scene_id": q["scene_id"],
            "situation": q["situation"],
            "position": [
                position["x"],
                position["y"],
                position["z"]
            ],
            "rotation": [
                rotation["_x"],
                rotation["_y"],
                rotation["_z"],
                rotation["_w"]
            ],
            "question": q["question"],
            "answer": a["answers"][0]["answer"]
        })

    return examples


def build_training_tokens(example):
    user_content = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]


train_examples = get_split_examples("train")

print("=" * 70)
print("TRAINING SEQUENCE LENGTH AUDIT")
print("=" * 70)
print(f"Training examples loaded: {len(train_examples):,}")

lengths = []

for example in train_examples:
    lengths.append(
        len(build_training_tokens(example))
    )

lengths_tensor = torch.tensor(
    lengths,
    dtype=torch.float32
)

print()
print(f"Minimum: {int(lengths_tensor.min())}")
print(f"Maximum: {int(lengths_tensor.max())}")
print(f"Mean:    {lengths_tensor.mean().item():.2f}")

for p in [50, 90, 95, 99, 99.5]:
    value = torch.quantile(
        lengths_tensor,
        p / 100
    ).item()

    print(f"P{p}:     {value:.1f}")

print()
print("Examples exceeding common sequence lengths:")

for limit in [256, 384, 512]:
    count = int((lengths_tensor > limit).sum())

    print(
        f"> {limit}: {count:,} "
        f"({count / len(lengths) * 100:.2f}%)"
    )

assert len(lengths) == 26623

print()
print("SEQUENCE LENGTH AUDIT PASSED")

TRAINING SEQUENCE LENGTH AUDIT
Training examples loaded: 26,623

Minimum: 103
Maximum: 193
Mean:    153.60
P50:     154.0
P90:     165.0
P95:     168.0
P99:     176.0
P99.5:     179.0

Examples exceeding common sequence lengths:
> 256: 0 (0.00%)
> 384: 0 (0.00%)
> 512: 0 (0.00%)

SEQUENCE LENGTH AUDIT PASSED


In [ ]:
import os
import json
import re
import torch

# ============================================================
# STEP 27 — CLEAN ZERO-SHOT BASELINE
# ============================================================

DATA_DIR = "/content/egospatial"
MODEL_NAME = "google/gemma-2-2b-it"

# ------------------------------------------------------------
# 1. Load validation data
# ------------------------------------------------------------

def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


questions = load_json(
    "v1_balanced_questions_val_scannetv2.json"
)["questions"]

annotations = load_json(
    "v1_balanced_sqa_annotations_val_scannetv2.json"
)["annotations"]

assert len(questions) == len(annotations)

val_examples = []

for q, a in zip(questions, annotations):

    position = a["position"]
    rotation = a["rotation"]

    val_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ],
        "question": q["question"],
        "answer": a["answers"][0]["answer"]
    })

print("=" * 70)
print("STEP 27 — ZERO-SHOT BASELINE")
print("=" * 70)
print(f"Validation examples available: {len(val_examples):,}")


# ------------------------------------------------------------
# 2. Prediction function
# ------------------------------------------------------------

def build_user_content(example):

    return (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )


def normalize_answer(text):
    text = text.strip().lower()

    # Remove Gemma turn/control markers if generated
    text = text.replace("<end_of_turn>", "")
    text = text.replace("<eos>", "")

    # Keep only the first non-empty line
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines:
        text = lines[0]

    # Remove simple surrounding punctuation
    text = text.strip(" \t\n\r.,!?;:\"'")

    return text


@torch.inference_mode()
def predict(example):

    messages = [
        {
            "role": "user",
            "content": build_user_content(example)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = outputs[0][inputs.shape[-1]:]

    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )

    prediction = normalize_answer(raw_output)

    return prediction, raw_output


# ------------------------------------------------------------
# 3. Run 100-example baseline
# ------------------------------------------------------------

N = min(100, len(val_examples))

correct = 0
results = []

for i in range(N):

    example = val_examples[i]

    prediction, raw_output = predict(example)

    expected = normalize_answer(example["answer"])

    is_correct = prediction == expected

    if is_correct:
        correct += 1

    results.append({
        "index": i,
        "scene_id": example["scene_id"],
        "question": example["question"],
        "expected": expected,
        "prediction": prediction,
        "correct": is_correct,
        "raw_output": raw_output
    })

    if i < 10:
        print()
        print("-" * 70)
        print(f"Example {i + 1}")
        print("Question :", example["question"])
        print("Expected :", expected)
        print("Predicted:", prediction)
        print("Correct  :", is_correct)


# ------------------------------------------------------------
# 4. Report baseline accuracy
# ------------------------------------------------------------

accuracy = correct / N

print()
print("=" * 70)
print("ZERO-SHOT BASELINE RESULT")
print("=" * 70)
print(f"Examples evaluated: {N}")
print(f"Correct:            {correct}")
print(f"Accuracy:           {accuracy * 100:.2f}%")
print("=" * 70)

FileNotFoundError: [Errno 2] No such file or directory: '/content/egospatial/v1_balanced_questions_val_scannetv2.json'

In [ ]:
import os
import requests

DATA_DIR = "/content/egospatial"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/Platinum04/EgoSpatial-Dataset/main"

FILES = [
    "v1_balanced_questions_train_scannetv2.json",
    "v1_balanced_questions_val_scannetv2.json",
    "v1_balanced_questions_test_scannetv2.json",
    "v1_balanced_sqa_annotations_train_scannetv2.json",
    "v1_balanced_sqa_annotations_val_scannetv2.json",
    "v1_balanced_sqa_annotations_test_scannetv2.json",
]

print("=" * 70)
print("RESTORING EGOSPATIAL DATASET")
print("=" * 70)

for filename in FILES:
    url = f"{BASE_URL}/{filename}"
    path = os.path.join(DATA_DIR, filename)

    response = requests.get(url)
    response.raise_for_status()

    with open(path, "wb") as f:
        f.write(response.content)

    print(f"✓ {filename} — {len(response.content):,} bytes")

print()
print("All six files restored.")

RESTORING EGOSPATIAL DATASET
✓ v1_balanced_questions_train_scannetv2.json — 11,140,002 bytes
✓ v1_balanced_questions_val_scannetv2.json — 1,385,508 bytes
✓ v1_balanced_questions_test_scannetv2.json — 1,460,185 bytes
✓ v1_balanced_sqa_annotations_train_scannetv2.json — 9,093,541 bytes
✓ v1_balanced_sqa_annotations_val_scannetv2.json — 1,114,948 bytes
✓ v1_balanced_sqa_annotations_test_scannetv2.json — 1,200,381 bytes

All six files restored.


In [ ]:
import json
import os

DATA_DIR = "/content/egospatial"

with open(
    os.path.join(DATA_DIR, "v1_balanced_questions_val_scannetv2.json"),
    "r",
    encoding="utf-8"
) as f:
    val_questions = json.load(f)["questions"]

with open(
    os.path.join(DATA_DIR, "v1_balanced_sqa_annotations_val_scannetv2.json"),
    "r",
    encoding="utf-8"
) as f:
    val_annotations = json.load(f)["annotations"]

print("=" * 70)
print("VALIDATION DATA CHECK")
print("=" * 70)
print(f"Questions:    {len(val_questions):,}")
print(f"Annotations:  {len(val_annotations):,}")

assert len(val_questions) == 3261
assert len(val_annotations) == 3261

print("✓ Validation dataset verified")

VALIDATION DATA CHECK
Questions:    3,261
Annotations:  3,261
✓ Validation dataset verified


In [ ]:
print(type(model).__name__)
print(type(tokenizer).__name__)

NameError: name 'model' is not defined

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-2-2b-it"

print("=" * 70)
print("LOADING OFFICIAL GEMMA 2B — 4-BIT")
print("=" * 70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print()
print("Model:", type(model).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Device:", model.device)
print(
    "4-bit:",
    any(
        getattr(module, "is_loaded_in_4bit", False)
        for module in model.modules()
    )
)

print()
print("MODEL LOAD COMPLETE")

LOADING OFFICIAL GEMMA 2B — 4-BIT


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6aa22859-4cd2de841ff67da45a8710c7;e49174df-7773-42ab-ab07-fe8332ce8b13)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
from huggingface_hub import login, whoami, get_token

print("=" * 70)
print("HUGGING FACE LOGIN")
print("=" * 70)

login()

token = get_token()

print()
print("Token available:", token is not None)

if token is not None:
    print("Token length:", len(token))
    print("Authenticated as:", whoami()["name"])
    print()
    print("✓ HUGGING FACE AUTHENTICATION PASSED")
else:
    print("✗ Token was not found")

HUGGING FACE LOGIN



Token available: True
Token length: 825
Authenticated as: Platinum04

✓ HUGGING FACE AUTHENTICATION PASSED


In [ ]:
import torch
from huggingface_hub import get_token
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "google/gemma-2-2b-it"
HF_TOKEN = get_token()

print("=" * 70)
print("LOADING OFFICIAL GEMMA 2B — AUTHENTICATED 4-BIT")
print("=" * 70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print()
print("Model:", type(model).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Device:", model.device)

is_4bit = any(
    getattr(module, "is_loaded_in_4bit", False)
    for module in model.modules()
)

print("4-bit:", is_4bit)

print()
print("MODEL LOAD COMPLETE")

LOADING OFFICIAL GEMMA 2B — AUTHENTICATED 4-BIT


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import sys
import torch
import bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available

print("=" * 70)
print("BITSANDBYTES ENVIRONMENT CHECK")
print("=" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("bitsandbytes:", bnb.__version__)
print("Transformers sees bitsandbytes:", is_bitsandbytes_available())

print()
print("bitsandbytes location:")
print(bnb.__file__)

ModuleNotFoundError: No module named 'bitsandbytes'

In [ ]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.5 MB/s eta 0:00:00


In [ ]:
import bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available

print("=" * 70)
print("BITSANDBYTES VERIFICATION")
print("=" * 70)

print("bitsandbytes version:", bnb.__version__)
print("Transformers sees bitsandbytes:", is_bitsandbytes_available())

assert is_bitsandbytes_available()

print()
print("✓ BITSANDBYTES READY")

BITSANDBYTES VERIFICATION
bitsandbytes version: 0.50.2
Transformers sees bitsandbytes: False


AssertionError: 

In [ ]:
import importlib
import transformers.utils.import_utils as import_utils

print("=" * 70)
print("REFRESHING TRANSFORMERS PACKAGE DETECTION")
print("=" * 70)

importlib.reload(import_utils)

print("bitsandbytes available:",
      import_utils.is_bitsandbytes_available())

assert import_utils.is_bitsandbytes_available()

print()
print("✓ TRANSFORMERS NOW RECOGNIZES BITSANDBYTES")

REFRESHING TRANSFORMERS PACKAGE DETECTION
bitsandbytes available: True

✓ TRANSFORMERS NOW RECOGNIZES BITSANDBYTES


In [ ]:
import torch
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-2-2b-it"
HF_TOKEN = get_token()

print("=" * 70)
print("LOADING OFFICIAL GEMMA 2B — 4-BIT")
print("=" * 70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print()
print("Model:", type(model).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Device:", model.device)

print()
print("MODEL LOAD COMPLETE")

LOADING OFFICIAL GEMMA 2B — 4-BIT


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import torch
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b-it"
HF_TOKEN = get_token()

print("=" * 70)
print("LOADING OFFICIAL GEMMA 2B — CLEAN FP16 BASELINE")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print()
print("Model:", type(model).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Device:", model.device)
print("Dtype:", next(model.parameters()).dtype)

print()
print("✓ CLEAN FP16 GEMMA LOADED")

LOADING OFFICIAL GEMMA 2B — CLEAN FP16 BASELINE


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]


Model: Gemma2ForCausalLM
Tokenizer: GemmaTokenizer
Device: cuda:0
Dtype: torch.float16

✓ CLEAN FP16 GEMMA LOADED


In [ ]:
import os
import json
import torch

# ============================================================
# STEP 27D — ZERO-SHOT BASELINE
# ============================================================

DATA_DIR = "/content/egospatial"

# ------------------------------------------------------------
# Load validation data
# ------------------------------------------------------------

def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


questions = load_json(
    "v1_balanced_questions_val_scannetv2.json"
)["questions"]

annotations = load_json(
    "v1_balanced_sqa_annotations_val_scannetv2.json"
)["annotations"]

assert len(questions) == len(annotations)

val_examples = []

for q, a in zip(questions, annotations):
    position = a["position"]
    rotation = a["rotation"]

    val_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ],
        "question": q["question"],
        "answer": a["answers"][0]["answer"]
    })

print("=" * 70)
print("STEP 27D — ZERO-SHOT BASELINE")
print("=" * 70)
print(f"Validation examples: {len(val_examples):,}")


# ------------------------------------------------------------
# Build prompt
# ------------------------------------------------------------

def build_user_content(example):
    return (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )


# ------------------------------------------------------------
# Normalize answers
# ------------------------------------------------------------

def normalize_answer(text):
    text = text.strip().lower()

    text = text.replace("<end_of_turn>", "")
    text = text.replace("<eos>", "")

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines:
        text = lines[0]

    text = text.strip(" \t\n\r.,!?;:\"'")

    return text


# ------------------------------------------------------------
# Generate prediction
# ------------------------------------------------------------

@torch.inference_mode()
def predict(example):

    messages = [
        {
            "role": "user",
            "content": build_user_content(example)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = outputs[0][inputs.shape[-1]:]

    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )

    prediction = normalize_answer(raw_output)

    return prediction, raw_output


# ------------------------------------------------------------
# Evaluate first 100 validation examples
# ------------------------------------------------------------

N = min(100, len(val_examples))

correct = 0
results = []

for i in range(N):

    example = val_examples[i]

    prediction, raw_output = predict(example)

    expected = normalize_answer(example["answer"])

    is_correct = prediction == expected

    if is_correct:
        correct += 1

    results.append({
        "index": i,
        "scene_id": example["scene_id"],
        "question": example["question"],
        "expected": expected,
        "prediction": prediction,
        "correct": is_correct,
        "raw_output": raw_output
    })

    if i < 10:
        print()
        print("-" * 70)
        print(f"Example {i + 1}")
        print("Question :", example["question"])
        print("Expected :", expected)
        print("Predicted:", prediction)
        print("Correct  :", is_correct)


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

accuracy = correct / N

print()
print("=" * 70)
print("ZERO-SHOT BASELINE RESULT")
print("=" * 70)
print(f"Examples evaluated: {N}")
print(f"Correct:            {correct}")
print(f"Accuracy:           {accuracy * 100:.2f}%")
print("=" * 70)

STEP 27D — ZERO-SHOT BASELINE
Validation examples: 3,261


AttributeError: 

In [ ]:
@torch.inference_mode()
def predict(example):

    messages = [
        {
            "role": "user",
            "content": build_user_content(example)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move tensors to the model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
        if torch.is_tensor(value)
    }

    input_length = inputs["input_ids"].shape[-1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = outputs[0][input_length:]

    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )

    prediction = normalize_answer(raw_output)

    return prediction, raw_output


print("✓ Prediction function fixed")

✓ Prediction function fixed


In [ ]:
example = val_examples[0]

prediction, raw_output = predict(example)

print("=" * 70)
print("SINGLE EXAMPLE BASELINE TEST")
print("=" * 70)
print("Question :", example["question"])
print("Expected :", normalize_answer(example["answer"]))
print("Predicted:", prediction)
print("Raw output:", repr(raw_output))
print("=" * 70)

SINGLE EXAMPLE BASELINE TEST
Question : Which direction should I toss a used napkin?
Expected : right
Predicted: to figure out which direction to toss the napkin, i need to consider a few
Raw output: 'To figure out which direction to toss the napkin, I need to consider a few'


In [ ]:
def build_user_content(example):
    return (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}\n\n"
        "Answer with only the answer. Do not explain your reasoning."
    )


print("✓ Inference prompt updated")

✓ Inference prompt updated


In [ ]:
example = val_examples[0]

prediction, raw_output = predict(example)

print("=" * 70)
print("SINGLE EXAMPLE — ANSWER-ONLY TEST")
print("=" * 70)
print("Question :", example["question"])
print("Expected :", normalize_answer(example["answer"]))
print("Predicted:", prediction)
print("Raw output:", repr(raw_output))
print("=" * 70)

SINGLE EXAMPLE — ANSWER-ONLY TEST
Question : Which direction should I toss a used napkin?
Expected : right
Predicted: towards the trash can
Raw output: 'Towards the trash can. \n<end_of_turn>'


In [ ]:
# ============================================================
# STEP 27D-4 — 100-EXAMPLE ZERO-SHOT BASELINE
# ============================================================

N = min(100, len(val_examples))

correct = 0
results = []

for i in range(N):

    example = val_examples[i]

    prediction, raw_output = predict(example)

    expected = normalize_answer(example["answer"])

    is_correct = prediction == expected

    if is_correct:
        correct += 1

    results.append({
        "index": i,
        "scene_id": example["scene_id"],
        "question": example["question"],
        "expected": expected,
        "prediction": prediction,
        "correct": is_correct,
        "raw_output": raw_output
    })

    if i < 10:
        print()
        print("-" * 70)
        print(f"Example {i + 1}")
        print("Question :", example["question"])
        print("Expected :", expected)
        print("Predicted:", prediction)
        print("Correct  :", is_correct)


accuracy = correct / N

print()
print("=" * 70)
print("ZERO-SHOT BASELINE RESULT")
print("=" * 70)
print(f"Examples evaluated: {N}")
print(f"Correct:            {correct}")
print(f"Accuracy:           {accuracy * 100:.2f}%")
print("=" * 70)


----------------------------------------------------------------------
Example 1
Question : Which direction should I toss a used napkin?
Expected : right
Predicted: towards the trash can
Correct  : False

----------------------------------------------------------------------
Example 2
Question : Is the amount of cabinet I am facing odd or even?
Expected : odd
Predicted: even
Correct  : False

----------------------------------------------------------------------
Example 3
Question : What is on the right side of the soap dispenser in front of me?
Expected : mirror
Predicted: hand
Correct  : False

----------------------------------------------------------------------
Example 4
Question : How many drawers are in the cabinet in front of me?
Expected : two
Predicted: 3
Correct  : False

----------------------------------------------------------------------
Example 5
Question : What color is the door that I am facing?
Expected : white
Predicted: red
Correct  : False

----------------------

In [ ]:
from collections import Counter, defaultdict

print("=" * 70)
print("STEP 28 — ZERO-SHOT BASELINE ERROR ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Overall statistics
# ------------------------------------------------------------

total = len(results)
correct = sum(r["correct"] for r in results)

print(f"Total examples : {total}")
print(f"Correct        : {correct}")
print(f"Incorrect      : {total - correct}")
print(f"Accuracy       : {correct / total * 100:.2f}%")

# ------------------------------------------------------------
# 2. Expected-answer distribution
# ------------------------------------------------------------

expected_counts = Counter(
    r["expected"]
    for r in results
)

print()
print("-" * 70)
print("MOST COMMON EXPECTED ANSWERS")
print("-" * 70)

for answer, count in expected_counts.most_common(20):
    print(f"{answer:<20} {count}")

# ------------------------------------------------------------
# 3. Prediction distribution
# ------------------------------------------------------------

prediction_counts = Counter(
    r["prediction"]
    for r in results
)

print()
print("-" * 70)
print("MOST COMMON MODEL PREDICTIONS")
print("-" * 70)

for answer, count in prediction_counts.most_common(20):
    print(f"{answer:<30} {count}")

# ------------------------------------------------------------
# 4. Exact-match pairs
# ------------------------------------------------------------

pairs = Counter(
    (r["expected"], r["prediction"])
    for r in results
)

print()
print("-" * 70)
print("MOST COMMON EXPECTED → PREDICTED PAIRS")
print("-" * 70)

for (expected, prediction), count in pairs.most_common(20):
    status = "✓" if expected == prediction else "✗"
    print(
        f"{status} {expected:<20} → {prediction:<30} {count}"
    )

# ------------------------------------------------------------
# 5. Correct-answer breakdown
# ------------------------------------------------------------

print()
print("-" * 70)
print("CORRECT ANSWERS")
print("-" * 70)

correct_answers = Counter(
    r["expected"]
    for r in results
    if r["correct"]
)

for answer, count in correct_answers.most_common():
    print(f"{answer:<20} {count}")

print()
print("=" * 70)
print("ERROR ANALYSIS COMPLETE")
print("=" * 70)

STEP 28 — ZERO-SHOT BASELINE ERROR ANALYSIS
Total examples : 100
Correct        : 19
Incorrect      : 81
Accuracy       : 19.00%

----------------------------------------------------------------------
MOST COMMON EXPECTED ANSWERS
----------------------------------------------------------------------
yes                  13
right                7
one                  7
two                  6
no                   6
odd                  4
black                4
table                3
rectangular          3
blue                 2
cabinet              2
open                 2
window               2
mirror               1
white                1
backside             1
on                   1
red black            1
three                1
picture              1

----------------------------------------------------------------------
MOST COMMON MODEL PREDICTIONS
----------------------------------------------------------------------
no                             12
1                              

In [ ]:
import os
import json
import torch

# ============================================================
# STEP 29 — BUILD + AUDIT TRAINING DATASET
# ============================================================

DATA_DIR = "/content/egospatial"
MAX_SEQ_LENGTH = 256

# ------------------------------------------------------------
# 1. Load training data
# ------------------------------------------------------------

def load_json(filename):
    with open(
        os.path.join(DATA_DIR, filename),
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


questions = load_json(
    "v1_balanced_questions_train_scannetv2.json"
)["questions"]

annotations = load_json(
    "v1_balanced_sqa_annotations_train_scannetv2.json"
)["annotations"]

assert len(questions) == len(annotations) == 26623

train_examples = []

for q, a in zip(questions, annotations):

    position = a["position"]
    rotation = a["rotation"]

    answer = a["answers"][0]["answer"]

    assert isinstance(answer, str)
    assert answer.strip() != ""

    train_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ],
        "question": q["question"],
        "answer": answer
    })


print("=" * 70)
print("STEP 29 — TRAINING DATASET BUILD")
print("=" * 70)
print(f"Training examples: {len(train_examples):,}")


# ------------------------------------------------------------
# 2. Build Gemma conversation
# ------------------------------------------------------------

def build_messages(example):

    user_content = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{example['situation']}\n\n"
        f"Agent position:\n{example['position']}\n\n"
        f"Agent rotation:\n{example['rotation']}\n\n"
        f"Question:\n{example['question']}"
    )

    return [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]


# ------------------------------------------------------------
# 3. Build full token sequence
# ------------------------------------------------------------

def build_training_item(example):

    messages = build_messages(example)

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    # User-only conversation.
    # add_generation_prompt=True gives us exactly the
    # portion that should NOT contribute to the loss.
    prompt_messages = [
        {
            "role": "user",
            "content": messages[0]["content"]
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    input_ids = full_ids[:MAX_SEQ_LENGTH]

    # Response-only labels
    labels = [-100] * len(input_ids)

    prompt_length = len(prompt_ids)

    # If the prompt itself exceeds the maximum, this example
    # cannot contain a supervised answer.
    if prompt_length < len(input_ids):

        for i in range(prompt_length, len(input_ids)):
            labels[i] = input_ids[i]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_length": prompt_length,
        "full_length": len(full_ids)
    }


# ------------------------------------------------------------
# 4. Build ALL training items
# ------------------------------------------------------------

training_items = []

for i, example in enumerate(train_examples):

    item = build_training_item(example)

    training_items.append(item)

    if (i + 1) % 5000 == 0:
        print(f"Processed: {i + 1:,} / {len(train_examples):,}")


print()
print("All training examples tokenized.")


# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

assert len(training_items) == 26623

sequence_lengths = [
    len(item["input_ids"])
    for item in training_items
]

prompt_lengths = [
    item["prompt_length"]
    for item in training_items
]

full_lengths = [
    item["full_length"]
    for item in training_items
]

# No sequence may exceed MAX_SEQ_LENGTH
assert max(sequence_lengths) <= MAX_SEQ_LENGTH

# No example should have a prompt that consumes the whole sequence
no_supervision = [
    i
    for i, item in enumerate(training_items)
    if not any(label != -100 for label in item["labels"])
]

assert len(no_supervision) == 0

# Every example must contain supervised answer tokens
supervised_counts = [
    sum(label != -100 for label in item["labels"])
    for item in training_items
]

assert min(supervised_counts) > 0

# Attention mask must match input IDs
for item in training_items[:100]:
    assert len(item["input_ids"]) == len(item["attention_mask"])
    assert len(item["input_ids"]) == len(item["labels"])


# ------------------------------------------------------------
# 6. Report statistics
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING DATASET INTEGRITY REPORT")
print("=" * 70)

print(f"Examples:                 {len(training_items):,}")
print(f"MAX_SEQ_LENGTH:           {MAX_SEQ_LENGTH}")
print(f"Minimum sequence length:  {min(sequence_lengths)}")
print(f"Maximum sequence length:  {max(sequence_lengths)}")
print(f"Average sequence length:  {sum(sequence_lengths)/len(sequence_lengths):.2f}")

print()
print(f"Minimum prompt length:    {min(prompt_lengths)}")
print(f"Maximum prompt length:    {max(prompt_lengths)}")
print(f"Average prompt length:    {sum(prompt_lengths)/len(prompt_lengths):.2f}")

print()
print(f"Minimum supervised tokens: {min(supervised_counts)}")
print(f"Maximum supervised tokens: {max(supervised_counts)}")
print(f"Average supervised tokens: {sum(supervised_counts)/len(supervised_counts):.2f}")

print()
print(f"Examples with no supervision: {len(no_supervision)}")

# ------------------------------------------------------------
# 7. Inspect one complete example
# ------------------------------------------------------------

example_index = 0
example = train_examples[example_index]
item = training_items[example_index]

print()
print("=" * 70)
print("SAMPLE TRAINING ITEM")
print("=" * 70)

print("Scene ID:")
print(example["scene_id"])

print()
print("Question:")
print(example["question"])

print()
print("Expected answer:")
print(repr(example["answer"]))

print()
print("Full decoded sequence:")
print(
    tokenizer.decode(
        item["input_ids"],
        skip_special_tokens=False
    )
)

print()
print("Supervised target:")
supervised_ids = [
    token_id
    for token_id, label in zip(
        item["input_ids"],
        item["labels"]
    )
    if label != -100
]

print(
    repr(
        tokenizer.decode(
            supervised_ids,
            skip_special_tokens=False
        )
    )
)

print()
print("=" * 70)
print("STEP 29 COMPLETE")
print("=" * 70)

STEP 29 — TRAINING DATASET BUILD
Training examples: 26,623
Processed: 5,000 / 26,623
Processed: 10,000 / 26,623
Processed: 15,000 / 26,623
Processed: 20,000 / 26,623
Processed: 25,000 / 26,623

All training examples tokenized.

TRAINING DATASET INTEGRITY REPORT
Examples:                 26,623
MAX_SEQ_LENGTH:           256
Minimum sequence length:  103
Maximum sequence length:  193
Average sequence length:  153.60

Minimum prompt length:    100
Maximum prompt length:    190
Average prompt length:    150.43

Minimum supervised tokens: 3
Maximum supervised tokens: 19
Average supervised tokens: 3.18

Examples with no supervision: 0

SAMPLE TRAINING ITEM
Scene ID:
scene0380_00

Question:
What color is the desk to my right?

Expected answer:
'brown'

Full decoded sequence:
<bos><start_of_turn>user
You are a spatial reasoning assistant.

Situation:
I am facing a window and there is a desk on my right and a chair behind me.

Agent position:
[-0.9651003385573296, -1.2417634435553606, 0]

Agent

In [ ]:
print("=" * 70)
print("STEP 30 — MODEL CHECK BEFORE LoRA")
print("=" * 70)

print("Model class:", type(model).__name__)
print("Tokenizer class:", type(tokenizer).__name__)
print("Model device:", model.device)
print("Model dtype:", next(model.parameters()).dtype)

print()
print("PEFT already attached:",
      hasattr(model, "peft_config"))

assert type(model).__name__ == "Gemma2ForCausalLM"
assert next(model.parameters()).dtype == torch.float16
assert not hasattr(model, "peft_config")

print()
print("✓ CLEAN BASE MODEL CONFIRMED")

STEP 30 — MODEL CHECK BEFORE LoRA
Model class: Gemma2ForCausalLM
Tokenizer class: GemmaTokenizer
Model device: cuda:0
Model dtype: torch.float16

PEFT already attached: False

✓ CLEAN BASE MODEL CONFIRMED


In [ ]:
from peft import LoraConfig, get_peft_model

# ============================================================
# STEP 31 — ATTACH LoRA
# ============================================================

print("=" * 70)
print("STEP 31 — ATTACHING LoRA")
print("=" * 70)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

print()
print("Model class:", type(model).__name__)
print("PEFT attached:", hasattr(model, "peft_config"))

assert hasattr(model, "peft_config")

print()
print("✓ LoRA ATTACHED SUCCESSFULLY")

STEP 31 — ATTACHING LoRA


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.7 MB/s eta 0:00:00


In [ ]:
import torchao

print("=" * 70)
print("TORCHAO VERSION CHECK")
print("=" * 70)
print("torchao version:", torchao.__version__)

version_parts = tuple(
    int(x)
    for x in torchao.__version__.split(".")[:2]
)

assert version_parts >= (0, 16)

print("✓ TORCHAO VERSION IS COMPATIBLE")

TORCHAO VERSION CHECK
torchao version: 0.18.0
✓ TORCHAO VERSION IS COMPATIBLE


In [ ]:
from peft import LoraConfig, get_peft_model

print("=" * 70)
print("STEP 31B — ATTACHING LoRA")
print("=" * 70)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

print()
print("Model class:", type(model).__name__)
print("PEFT attached:", hasattr(model, "peft_config"))

assert hasattr(model, "peft_config")

print()
print("✓ LoRA ATTACHED SUCCESSFULLY")

STEP 31B — ATTACHING LoRA
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881

Model class: PeftModelForCausalLM
PEFT attached: True

✓ LoRA ATTACHED SUCCESSFULLY


In [ ]:
import inspect
import transformers
import peft
import torch

print("=" * 70)
print("STEP 32 — TRAINING ENVIRONMENT CHECK")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

print()
print("Model:", type(model).__name__)
print("Trainable parameters:")

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"  Trainable: {trainable:,}")
print(f"  Total:     {total:,}")
print(f"  Ratio:     {100 * trainable / total:.4f}%")

print()
print("Sample training item:")
sample = training_items[0]

print("  input_ids:", len(sample["input_ids"]))
print("  labels:", len(sample["labels"]))
print("  attention_mask:", len(sample["attention_mask"]))

supervised = sum(
    label != -100
    for label in sample["labels"]
)

ignored = sum(
    label == -100
    for label in sample["labels"]
)

print("  Ignored labels:", ignored)
print("  Supervised labels:", supervised)

assert len(sample["input_ids"]) == len(sample["labels"])
assert len(sample["input_ids"]) == len(sample["attention_mask"])
assert supervised > 0
assert ignored > 0

print()
print("✓ TRAINING INPUT STRUCTURE VERIFIED")

STEP 32 — TRAINING ENVIRONMENT CHECK
Transformers: 5.16.1
PEFT: 0.20.0
PyTorch: 2.11.0+cu128
GPU: Tesla T4

Model: PeftModelForCausalLM
Trainable parameters:
  Trainable: 20,766,720
  Total:     2,635,108,608
  Ratio:     0.7881%

Sample training item:
  input_ids: 155
  labels: 155
  attention_mask: 155
  Ignored labels: 152
  Supervised labels: 3

✓ TRAINING INPUT STRUCTURE VERIFIED


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer

# ============================================================
# STEP 33 — TRAINING SMOKE TEST
# ============================================================

print("=" * 70)
print("STEP 33 — TRAINING SMOKE TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. Minimal Dataset wrapper
# ------------------------------------------------------------

class EgoSpatialTorchDataset(Dataset):

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        return {
            "input_ids": torch.tensor(
                item["input_ids"],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                item["attention_mask"],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                item["labels"],
                dtype=torch.long
            )
        }


# Use only 8 examples for the smoke test
smoke_dataset = EgoSpatialTorchDataset(
    training_items[:8]
)

print("Smoke-test examples:", len(smoke_dataset))


# ------------------------------------------------------------
# 2. Custom collator
# ------------------------------------------------------------

def smoke_collator(features):

    return {
        "input_ids": torch.stack(
            [f["input_ids"] for f in features]
        ),
        "attention_mask": torch.stack(
            [f["attention_mask"] for f in features]
        ),
        "labels": torch.stack(
            [f["labels"] for f in features]
        )
    }


# ------------------------------------------------------------
# 3. Conservative T4 training arguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="/content/egospatial_smoke_test",

    max_steps=2,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,

    learning_rate=1e-4,
    warmup_steps=0,

    logging_steps=1,

    fp16=True,
    bf16=False,

    optim="adamw_torch",

    save_strategy="no",
    report_to="none",

    remove_unused_columns=False,

    gradient_checkpointing=False,

    dataloader_num_workers=0
)


# ------------------------------------------------------------
# 4. Create Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=smoke_dataset,
    data_collator=smoke_collator
)


# ------------------------------------------------------------
# 5. Run smoke test
# ------------------------------------------------------------

print()
print("Starting 2-step training smoke test...")
print()

train_result = trainer.train()


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("SMOKE TEST COMPLETE")
print("=" * 70)

print("Global steps:", trainer.state.global_step)

if train_result.training_loss is not None:
    print(
        "Training loss:",
        round(train_result.training_loss, 4)
    )

print()
print("✓ TRAINING PIPELINE EXECUTED")

STEP 33 — TRAINING SMOKE TEST
Smoke-test examples: 8

Starting 2-step training smoke test...



RuntimeError: stack expects each tensor to be equal size, but got [120] at entry 0 and [150] at entry 1

In [ ]:
def smoke_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:
        pad_len = max_len - len(f["input_ids"])

        input_ids.append(
            torch.cat([
                f["input_ids"],
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])
        )

        attention_masks.append(
            torch.cat([
                f["attention_mask"],
                torch.zeros(
                    pad_len,
                    dtype=torch.long
                )
            ])
        )

        labels.append(
            torch.cat([
                f["labels"],
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels)
    }

print("✓ Dynamic-padding collator ready")

✓ Dynamic-padding collator ready


In [ ]:
training_args = TrainingArguments(

_IncompleteInputError: incomplete input (1951541638.py, line 1)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/egospatial_smoke_test",
    max_steps=2,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,

    learning_rate=1e-4,
    warmup_steps=0,

    logging_steps=1,

    fp16=True,
    bf16=False,

    optim="adamw_torch",

    save_strategy="no",
    report_to="none",

    remove_unused_columns=False,

    gradient_checkpointing=False,

    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=smoke_dataset,
    data_collator=smoke_collator
)

print("Starting 2-step training smoke test...")
train_result = trainer.train()

print()
print("=" * 70)
print("SMOKE TEST RESULT")
print("=" * 70)

print("Global steps:", trainer.state.global_step)

if train_result.training_loss is not None:
    print("Training loss:", round(train_result.training_loss, 4))

print("✓ TRAINING PIPELINE EXECUTED")

Starting 2-step training smoke test...


Step,Training Loss
1,15.207809
2,13.239855



SMOKE TEST RESULT
Global steps: 2
Training loss: 14.2238
✓ TRAINING PIPELINE EXECUTED


In [ ]:
# ============================================================
# STEP 34 — RESET TO FRESH MODEL + FRESH LoRA
# ============================================================

import gc
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

print("=" * 70)
print("STEP 34 — FRESH MODEL RESET")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"

# ------------------------------------------------------------
# 1. Remove the smoke-test model/trainer from memory
# ------------------------------------------------------------

try:
    del trainer
except:
    pass

try:
    del model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("✓ Previous model cleared")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# ------------------------------------------------------------
# 2. Reload the official pretrained Gemma
# ------------------------------------------------------------

print()
print("Loading clean pretrained Gemma...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

print("✓ Clean pretrained Gemma loaded")


# ------------------------------------------------------------
# 3. Attach BRAND-NEW LoRA adapters
# ------------------------------------------------------------

print()
print("Attaching fresh LoRA adapters...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✓ Fresh LoRA attached")


# ------------------------------------------------------------
# 4. Report trainable parameters
# ------------------------------------------------------------

print()
model.print_trainable_parameters()


# ------------------------------------------------------------
# 5. Verify model state
# ------------------------------------------------------------

print()
print("=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print("Model class:", type(model).__name__)
print("Base model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

print()
print("Expected:")
print("  Trainable parameters ≈ 20.77M")
print("  Trainable ratio ≈ 0.79%")

print()
print("✓ FRESH MODEL READY")

STEP 34 — FRESH MODEL RESET
✓ Previous model cleared
GPU memory allocated: 4.96 GB
GPU memory reserved:  4.98 GB

Loading clean pretrained Gemma...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

✓ Clean pretrained Gemma loaded

Attaching fresh LoRA adapters...
✓ Fresh LoRA attached

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881

MODEL VERIFICATION
Model class: PeftModelForCausalLM
Base model: google/gemma-2-2b-it
Device: cuda:0
Dtype: torch.float16

Expected:
  Trainable parameters ≈ 20.77M
  Trainable ratio ≈ 0.79%

✓ FRESH MODEL READY


In [ ]:
# ============================================================
# STEP 35 — REAL DATA BATCH SANITY CHECK
# ============================================================

import torch
from torch.utils.data import DataLoader

print("=" * 70)
print("STEP 35 — REAL DATA BATCH SANITY CHECK")
print("=" * 70)

# Use the already-created training dataset and collator
sanity_loader = DataLoader(
    smoke_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=smoke_collator,
    num_workers=0
)

batch = next(iter(sanity_loader))

print("Batch shapes:")
print("  input_ids:     ", batch["input_ids"].shape)
print("  attention_mask: ", batch["attention_mask"].shape)
print("  labels:         ", batch["labels"].shape)

print()
print("Running one forward pass...")

# Move batch to the model device
batch = {
    key: value.to(model.device)
    for key, value in batch.items()
}

with torch.no_grad():
    outputs = model(**batch)

loss = outputs.loss

print()
print("=" * 70)
print("SANITY CHECK RESULT")
print("=" * 70)

print("Loss:", round(loss.item(), 4))
print("Loss finite:", torch.isfinite(loss).item())

print()
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print()
print("✓ REAL DATA FORWARD PASS SUCCESSFUL")

STEP 35 — REAL DATA BATCH SANITY CHECK
Batch shapes:
  input_ids:      torch.Size([2, 171])
  attention_mask:  torch.Size([2, 171])
  labels:          torch.Size([2, 171])

Running one forward pass...

SANITY CHECK RESULT
Loss: 12.5253
Loss finite: True

GPU memory allocated: 10.11 GB

✓ REAL DATA FORWARD PASS SUCCESSFUL


In [ ]:
# ============================================================
# STEP 36A — VERIFY FULL TRAINING DATASET
# ============================================================

print("=" * 70)
print("STEP 36A — FULL TRAINING DATASET VERIFICATION")
print("=" * 70)

full_train_dataset = EgoSpatialTorchDataset(training_items)

print("Dataset length:", len(full_train_dataset))
print("Expected length: 26623")

assert len(full_train_dataset) == 26623, (
    f"Expected 26623 examples, got {len(full_train_dataset)}"
)

# Inspect first and last examples
first_item = full_train_dataset[0]
last_item = full_train_dataset[-1]

print()
print("First example:")
print("  input_ids:", len(first_item["input_ids"]))
print("  labels:", len(first_item["labels"]))

print()
print("Last example:")
print("  input_ids:", len(last_item["input_ids"]))
print("  labels:", len(last_item["labels"]))

print()
print("✓ FULL TRAINING DATASET VERIFIED")

STEP 36A — FULL TRAINING DATASET VERIFICATION
Dataset length: 26623
Expected length: 26623

First example:
  input_ids: 155
  labels: 155

Last example:
  input_ids: 156
  labels: 156

✓ FULL TRAINING DATASET VERIFIED


In [ ]:
# ============================================================
# STEP 36B — REAL EGOSPATIAL-GEMMA TRAINING
# ============================================================

from transformers import TrainingArguments, Trainer

print("=" * 70)
print("STEP 36B — REAL EGOSPATIAL-GEMMA TRAINING")
print("=" * 70)

print("Training examples:", len(full_train_dataset))
print("Epochs: 1")
print("Batch size: 2")
print("Gradient accumulation: 4")
print("Effective batch size: 8")
print("Learning rate: 1e-4")
print("Warmup steps: 100")
print("FP16: True")
print()

training_args = TrainingArguments(
    output_dir="/content/egospatial_gemma_v1",

    # Training duration
    num_train_epochs=1,

    # Batch configuration
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Optimization
    learning_rate=1e-4,
    warmup_steps=100,
    optim="adamw_torch",

    # Precision
    fp16=True,
    bf16=False,

    # Logging
    logging_steps=50,
    logging_first_step=True,
    report_to="none",

    # Memory / dataloader
    gradient_checkpointing=False,
    dataloader_num_workers=0,

    # Dataset handling
    remove_unused_columns=False,

    # Checkpointing
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    # Reproducibility
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_train_dataset,
    data_collator=smoke_collator,
)

print("✓ Trainer configured")
print()
print("Starting full training...")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Global steps:", trainer.state.global_step)

if train_result.training_loss is not None:
    print(
        "Final training loss:",
        round(train_result.training_loss, 4)
    )

print("✓ EGOSPATIAL-GEMMA V1 TRAINING FINISHED")

STEP 36B — REAL EGOSPATIAL-GEMMA TRAINING
Training examples: 26623
Epochs: 1
Batch size: 2
Gradient accumulation: 4
Effective batch size: 8
Learning rate: 1e-4
Warmup steps: 100
FP16: True

✓ Trainer configured

Starting full training...


Step,Training Loss
1,14.422605
50,5.197249
100,0.778276
150,0.795091
200,0.724277
250,0.719351
300,0.607217
350,0.657008
400,0.643619
450,0.649763


In [ ]:
import torch
import os

print("=" * 70)
print("GPU / CHECKPOINT RECOVERY CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

print()
print("Training checkpoint directories:")

if os.path.exists("/content/egospatial_gemma_v1"):
    for name in sorted(os.listdir("/content/egospatial_gemma_v1")):
        if name.startswith("checkpoint"):
            print(" ", name)
else:
    print("  /content/egospatial_gemma_v1 does not exist")

GPU / CHECKPOINT RECOVERY CHECK
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Training checkpoint directories:
  /content/egospatial_gemma_v1 does not exist


In [ ]:
# ============================================================
# STEP 36D — CHECK IN-MEMORY TRAINING STATE
# ============================================================

import os
import torch

print("=" * 70)
print("STEP 36D — IN-MEMORY TRAINING STATE")
print("=" * 70)

# Check trainer
if "trainer" in globals():
    print("Trainer object: EXISTS")

    try:
        print("Trainer global step:", trainer.state.global_step)
    except Exception as e:
        print("Could not read global step:", e)

    try:
        print("Trainer epoch:", trainer.state.epoch)
    except Exception as e:
        print("Could not read epoch:", e)

    try:
        print("Optimizer exists:", trainer.optimizer is not None)
    except Exception as e:
        print("Optimizer check failed:", e)

else:
    print("Trainer object: NOT FOUND")

print()

# Check model
if "model" in globals():
    print("Model object: EXISTS")
    print("Model class:", type(model).__name__)

    try:
        print("Device:", next(model.parameters()).device)
    except Exception as e:
        print("Device check failed:", e)

else:
    print("Model object: NOT FOUND")

print()

# Check output directory
print("Output directory exists:",
      os.path.exists("/content/egospatial_gemma_v1"))

print()

# GPU state
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

print()
print("=" * 70)

STEP 36D — IN-MEMORY TRAINING STATE
Trainer object: NOT FOUND

Model object: NOT FOUND

Output directory exists: False

CUDA available: True
GPU: Tesla T4
Allocated: 0.0 GB



In [ ]:
# ============================================================
# STEP 36E — PERSISTENT CHECKPOINT STORAGE
# ============================================================

from google.colab import drive
import os

print("=" * 70)
print("STEP 36E — GOOGLE DRIVE CHECKPOINT STORAGE")
print("=" * 70)

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/EgoSpatial_Gemma_v1"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print()
print("Checkpoint directory:")
print(CHECKPOINT_DIR)

print()
print("✓ PERSISTENT CHECKPOINT STORAGE READY")

STEP 36E — GOOGLE DRIVE CHECKPOINT STORAGE


ValueError: mount failed

In [ ]:
# ============================================================
# STEP 36F — REPAIR GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive
import os

print("=" * 70)
print("STEP 36F — GOOGLE DRIVE MOUNT RETRY")
print("=" * 70)

try:
    drive.mount(
        "/content/drive",
        force_remount=True,
        timeout_ms=120000
    )

    print()
    print("Drive mount completed.")

except Exception as e:
    print()
    print("Drive mount error:")
    print(type(e).__name__, ":", e)

print()
print("Mount point exists:", os.path.exists("/content/drive"))

if os.path.exists("/content/drive"):
    print("Drive contents:")
    print(os.listdir("/content/drive")[:10])

STEP 36F — GOOGLE DRIVE MOUNT RETRY

Drive mount error:
MessageError : Error: credential propagation was unsuccessful

Mount point exists: False


In [ ]:
# ============================================================
# STEP 36G — VERIFY HUGGING FACE PERSISTENT STORAGE
# ============================================================

from huggingface_hub import HfApi

print("=" * 70)
print("STEP 36G — HUGGING FACE AUTHENTICATION")
print("=" * 70)

api = HfApi()

try:
    user_info = api.whoami()

    print("Hugging Face username:", user_info["name"])
    print()
    print("✓ HUGGING FACE AUTHENTICATION VERIFIED")

except Exception as e:
    print()
    print("Hugging Face authentication failed:")
    print(type(e).__name__, ":", e)

STEP 36G — HUGGING FACE AUTHENTICATION
Hugging Face username: Platinum04

✓ HUGGING FACE AUTHENTICATION VERIFIED


In [ ]:
# ============================================================
# STEP 36H — CREATE PRIVATE CHECKPOINT REPOSITORY
# ============================================================

from huggingface_hub import HfApi

print("=" * 70)
print("STEP 36H — HUGGING FACE CHECKPOINT REPOSITORY")
print("=" * 70)

api = HfApi()

REPO_ID = "Platinum04/EgoSpatial-Gemma-v1"

try:
    api.create_repo(
        repo_id=REPO_ID,
        repo_type="model",
        private=True,
        exist_ok=True
    )

    print()
    print("Repository:", REPO_ID)
    print("Visibility: PRIVATE")
    print()
    print("✓ CHECKPOINT REPOSITORY READY")

except Exception as e:
    print()
    print("Repository creation failed:")
    print(type(e).__name__, ":", e)

STEP 36H — HUGGING FACE CHECKPOINT REPOSITORY

Repository: Platinum04/EgoSpatial-Gemma-v1
Visibility: PRIVATE

✓ CHECKPOINT REPOSITORY READY


In [ ]:
# ============================================================
# STEP 36I — REBUILD FRESH GEMMA + LoRA
# ============================================================

import gc
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

print("=" * 70)
print("STEP 36I — FRESH MODEL REBUILD")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"

# ------------------------------------------------------------
# Clear any residual objects
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

print("GPU memory before loading:")
print(
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB allocated"
)

# ------------------------------------------------------------
# Load official Gemma
# ------------------------------------------------------------

print()
print("Loading official Gemma 2B...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

print("✓ Clean pretrained Gemma loaded")

# ------------------------------------------------------------
# Fresh LoRA
# ------------------------------------------------------------

print()
print("Attaching fresh LoRA...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✓ Fresh LoRA attached")

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print()
model.print_trainable_parameters()

print()
print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

print()
print(
    "GPU memory after loading:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print()
print("=" * 70)
print("✓ FRESH MODEL READY FOR TRAINING")
print("=" * 70)

STEP 36I — FRESH MODEL REBUILD
GPU memory before loading:
0.0 GB allocated

Loading official Gemma 2B...


NameError: name 'HF_TOKEN' is not defined

In [ ]:
import torch
import os

print("=" * 70)
print("EGOSPATIAL-GEMMA — CURRENT RUNTIME CHECK")
print("=" * 70)

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print()
print("HF_TOKEN defined:", "HF_TOKEN" in globals())
print("MODEL defined:", "model" in globals())
print("TOKENIZER defined:", "tokenizer" in globals())
print("TRAINING DATA defined:", "training_items" in globals())
print("FULL DATASET defined:", "full_train_dataset" in globals())
print("COLLATOR defined:", "smoke_collator" in globals())

print()
print("Checkpoint directory:")
print("/content/drive/MyDrive/EgoSpatial_Gemma_v1")
print("Drive mounted:", os.path.exists("/content/drive"))

print()
print("=" * 70)

EGOSPATIAL-GEMMA — CURRENT RUNTIME CHECK

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

HF_TOKEN defined: False
MODEL defined: False
TOKENIZER defined: False
TRAINING DATA defined: False
FULL DATASET defined: False
COLLATOR defined: False

Checkpoint directory:
/content/drive/MyDrive/EgoSpatial_Gemma_v1
Drive mounted: False



In [ ]:
# ============================================================
# STEP 37 — RESTORE HUGGING FACE AUTHENTICATION
# ============================================================

from huggingface_hub import login, HfApi

print("=" * 70)
print("STEP 37 — HUGGING FACE AUTHENTICATION")
print("=" * 70)

login()

api = HfApi()

user_info = api.whoami()

HF_TOKEN = api.token

print()
print("Hugging Face username:", user_info["name"])
print("Token available:", HF_TOKEN is not None)

print()
print("✓ HUGGING FACE AUTHENTICATION RESTORED")

STEP 37 — HUGGING FACE AUTHENTICATION

Hugging Face username: Platinum04
Token available: False

✓ HUGGING FACE AUTHENTICATION RESTORED


In [ ]:
# ============================================================
# STEP 37A — VERIFY HUGGING FACE REPOSITORY ACCESS
# ============================================================

from huggingface_hub import HfApi

print("=" * 70)
print("STEP 37A — HUGGING FACE REPOSITORY ACCESS")
print("=" * 70)

api = HfApi()

REPO_ID = "Platinum04/EgoSpatial-Gemma-v1"

try:
    repo_info = api.model_info(REPO_ID)

    print()
    print("Repository:", repo_info.id)
    print("Private:", repo_info.private)

    print()
    print("✓ PRIVATE REPOSITORY ACCESS VERIFIED")

except Exception as e:
    print()
    print("Repository access failed:")
    print(type(e).__name__, ":", e)

STEP 37A — HUGGING FACE REPOSITORY ACCESS

Repository: Platinum04/EgoSpatial-Gemma-v1
Private: True

✓ PRIVATE REPOSITORY ACCESS VERIFIED


In [ ]:
# ============================================================
# STEP 38A — DOWNLOAD EGOSPATIAL TRAINING DATA
# ============================================================

import os
import requests

print("=" * 70)
print("STEP 38A — DOWNLOAD EGOSPATIAL TRAINING DATA")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"
os.makedirs(DATA_DIR, exist_ok=True)

files_to_download = {
    "v1_balanced_questions_train_scannetv2.json":
        "https://raw.githubusercontent.com/Platinum04/EgoSpatial-Dataset/main/v1_balanced_questions_train_scannetv2.json",

    "v1_balanced_sqa_annotations_train_scannetv2.json":
        "https://raw.githubusercontent.com/Platinum04/EgoSpatial-Dataset/main/v1_balanced_sqa_annotations_train_scannetv2.json"
}

for filename, url in files_to_download.items():

    destination = os.path.join(DATA_DIR, filename)

    print()
    print("Downloading:", filename)

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(destination, "wb") as f:
        f.write(response.content)

    size_mb = os.path.getsize(destination) / (1024 * 1024)

    print(
        "✓ Downloaded:",
        round(size_mb, 2),
        "MB"
    )

print()
print("=" * 70)
print("DOWNLOADED FILES")
print("=" * 70)

for filename in files_to_download:
    path = os.path.join(DATA_DIR, filename)

    print(
        filename,
        "→",
        round(os.path.getsize(path) / (1024 * 1024), 2),
        "MB"
    )

print()
print("✓ TRAINING FILES READY")

STEP 38A — DOWNLOAD EGOSPATIAL TRAINING DATA

Downloading: v1_balanced_questions_train_scannetv2.json
✓ Downloaded: 10.62 MB

Downloading: v1_balanced_sqa_annotations_train_scannetv2.json
✓ Downloaded: 8.67 MB

DOWNLOADED FILES
v1_balanced_questions_train_scannetv2.json → 10.62 MB
v1_balanced_sqa_annotations_train_scannetv2.json → 8.67 MB

✓ TRAINING FILES READY


In [ ]:
# ============================================================
# STEP 38B — REBUILD RAW TRAINING EXAMPLES
# ============================================================

import json
import os

print("=" * 70)
print("STEP 38B — REBUILD RAW TRAINING EXAMPLES")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"

QUESTIONS_PATH = os.path.join(
    DATA_DIR,
    "v1_balanced_questions_train_scannetv2.json"
)

ANNOTATIONS_PATH = os.path.join(
    DATA_DIR,
    "v1_balanced_sqa_annotations_train_scannetv2.json"
)

# ------------------------------------------------------------
# Load files
# ------------------------------------------------------------

with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    questions = json.load(f)

with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print("Questions loaded:", len(questions))
print("Annotations loaded:", len(annotations))

# ------------------------------------------------------------
# Build annotation lookup
# ------------------------------------------------------------

annotation_by_id = {
    item["question_id"]: item
    for item in annotations
}

# ------------------------------------------------------------
# Reconstruct examples
# ------------------------------------------------------------

raw_examples = []

for q in questions:

    qid = q["question_id"]

    if qid not in annotation_by_id:
        raise ValueError(
            f"Missing annotation for question ID: {qid}"
        )

    ann = annotation_by_id[qid]

    answers = ann.get("answers", [])

    if not answers:
        raise ValueError(
            f"No answer found for question ID: {qid}"
        )

    answer = answers[0]["answer"]

    raw_examples.append({
        "question_id": qid,
        "scene_id": q.get("scene_id"),
        "situation": q["situation"],
        "agent_position": q.get("agent_position"),
        "agent_rotation": q.get("agent_rotation"),
        "question": q["question"],
        "answer": answer
    })

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(raw_examples) == 26623

question_ids = [x["question_id"] for x in raw_examples]

assert len(question_ids) == len(set(question_ids))

scene_ids = [x["scene_id"] for x in raw_examples]

assert all(scene_id is not None for scene_id in scene_ids)

print()
print("Raw examples:", len(raw_examples))
print("Unique question IDs:", len(set(question_ids)))
print("Unique scenes:", len(set(scene_ids)))

# ------------------------------------------------------------
# Sample
# ------------------------------------------------------------

print()
print("=" * 70)
print("SAMPLE")
print("=" * 70)

sample = raw_examples[0]

for key, value in sample.items():
    print(f"{key}: {value}")

print()
print("=" * 70)
print("✓ RAW TRAINING DATA REBUILT AND VERIFIED")
print("=" * 70)

STEP 38B — REBUILD RAW TRAINING EXAMPLES
Questions loaded: 6
Annotations loaded: 5


TypeError: string indices must be integers, not 'str'

In [ ]:
# ============================================================
# STEP 38C — INSPECT DATASET JSON STRUCTURE
# ============================================================

import json
import os

print("=" * 70)
print("STEP 38C — INSPECT DATASET JSON STRUCTURE")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"

files_to_check = [
    "v1_balanced_questions_train_scannetv2.json",
    "v1_balanced_sqa_annotations_train_scannetv2.json"
]

for filename in files_to_check:

    path = os.path.join(DATA_DIR, filename)

    print()
    print("-" * 70)
    print(filename)
    print("-" * 70)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level Python type:", type(data).__name__)

    if isinstance(data, dict):

        print("Number of top-level keys:", len(data))
        print("Top-level keys:")

        for key in data.keys():
            print("  -", repr(key))

        print()
        print("Value types:")

        for key, value in data.items():
            print(
                " ",
                repr(key),
                "→",
                type(value).__name__,
                "length:",
                len(value) if hasattr(value, "__len__") else "N/A"
            )

            if isinstance(value, list) and len(value) > 0:
                print("    First item type:", type(value[0]).__name__)
                print("    First item:", value[0])

    elif isinstance(data, list):

        print("Number of items:", len(data))

        if len(data) > 0:
            print("First item type:", type(data[0]).__name__)
            print("First item:", data[0])

    else:
        print("Unexpected structure:", repr(data))

print()
print("=" * 70)
print("✓ JSON STRUCTURE INSPECTION COMPLETE")
print("=" * 70)

STEP 38C — INSPECT DATASET JSON STRUCTURE

----------------------------------------------------------------------
v1_balanced_questions_train_scannetv2.json
----------------------------------------------------------------------
Top-level Python type: dict
Number of top-level keys: 6
Top-level keys:
  - 'info'
  - 'license'
  - 'data_type'
  - 'data_subtype'
  - 'task_type'
  - 'questions'

Value types:
  'info' → dict length: 6
  'license' → dict length: 2
  'data_type' → str length: 7
  'data_subtype' → str length: 2
  'task_type' → str length: 27
  'questions' → list length: 26623
    First item type: dict
    First item: {'scene_id': 'scene0380_00', 'situation': 'I am facing a window and there is a desk on my right and a chair behind me.', 'alternative_situation': ['I stand looking out of the window in thought and a radiator is right in front of me.', 'I am looking outside through the window behind the desk.'], 'question': 'What color is the desk to my right?', 'question_id': 220602

In [ ]:
# ============================================================
# STEP 38D — CORRECT RAW DATASET RECONSTRUCTION
# ============================================================

import json
import os

print("=" * 70)
print("STEP 38D — CORRECT RAW DATASET RECONSTRUCTION")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"

QUESTIONS_PATH = os.path.join(
    DATA_DIR,
    "v1_balanced_questions_train_scannetv2.json"
)

ANNOTATIONS_PATH = os.path.join(
    DATA_DIR,
    "v1_balanced_sqa_annotations_train_scannetv2.json"
)

# ------------------------------------------------------------
# Load JSON wrappers
# ------------------------------------------------------------

with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations_data = json.load(f)

# ------------------------------------------------------------
# Extract actual arrays
# ------------------------------------------------------------

questions = questions_data["questions"]
annotations = annotations_data["annotations"]

print("Questions:", len(questions))
print("Annotations:", len(annotations))

assert len(questions) == 26623
assert len(annotations) == 26623

# ------------------------------------------------------------
# Build annotation lookup
# ------------------------------------------------------------

annotation_by_id = {
    item["question_id"]: item
    for item in annotations
}

# ------------------------------------------------------------
# Reconstruct examples
# ------------------------------------------------------------

raw_examples = []

for q in questions:

    qid = q["question_id"]

    if qid not in annotation_by_id:
        raise ValueError(
            f"Missing annotation for question ID: {qid}"
        )

    ann = annotation_by_id[qid]

    answers = ann["answers"]

    if not answers:
        raise ValueError(
            f"No answer for question ID: {qid}"
        )

    answer = answers[0]["answer"]

    # --------------------------------------------------------
    # Pose comes from annotation
    # --------------------------------------------------------

    position = ann["position"]
    rotation = ann["rotation"]

    agent_position = [
        position["x"],
        position["y"],
        position["z"]
    ]

    agent_rotation = [
        rotation["_x"],
        rotation["_y"],
        rotation["_z"],
        rotation["_w"]
    ]

    raw_examples.append({
        "question_id": qid,
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "alternative_situation": q.get(
            "alternative_situation",
            []
        ),
        "agent_position": agent_position,
        "agent_rotation": agent_rotation,
        "question": q["question"],
        "answer": answer
    })

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(raw_examples) == 26623

question_ids = [
    x["question_id"]
    for x in raw_examples
]

assert len(question_ids) == len(set(question_ids))

annotation_ids = [
    x["question_id"]
    for x in annotations
]

assert question_ids == annotation_ids, (
    "Question and annotation ordering/IDs do not match."
)

scene_ids = [
    x["scene_id"]
    for x in raw_examples
]

assert all(scene_id is not None for scene_id in scene_ids)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print()
print("=" * 70)
print("DATASET INTEGRITY")
print("=" * 70)

print("Raw examples:", len(raw_examples))
print("Unique question IDs:", len(set(question_ids)))
print("Unique scenes:", len(set(scene_ids)))

print()
print("First example:")
print("-" * 70)

for key, value in raw_examples[0].items():
    print(f"{key}: {value}")

print()
print("Last example:")
print("-" * 70)

for key, value in raw_examples[-1].items():
    print(f"{key}: {value}")

print()
print("=" * 70)
print("✓ RAW TRAINING DATA REBUILT CORRECTLY")
print("=" * 70)

STEP 38D — CORRECT RAW DATASET RECONSTRUCTION
Questions: 26623
Annotations: 26623

DATASET INTEGRITY
Raw examples: 26623
Unique question IDs: 26623
Unique scenes: 518

First example:
----------------------------------------------------------------------
question_id: 220602000000
scene_id: scene0380_00
situation: I am facing a window and there is a desk on my right and a chair behind me.
alternative_situation: ['I stand looking out of the window in thought and a radiator is right in front of me.', 'I am looking outside through the window behind the desk.']
agent_position: [-0.9651003385573296, -1.2417634435553606, 0]
agent_rotation: [0, 0, 0.09983341664682724, 0.9950041652780182]
question: What color is the desk to my right?
answer: brown

Last example:
----------------------------------------------------------------------
question_id: 220602033402
scene_id: scene0258_00
situation: I am sitting on a brown chair while there is a brown chair on my left and wall is at my back.
alternative_

In [ ]:
# ============================================================
# STEP 39 — TOKENIZE EGOSPATIAL TRAINING DATA
# ============================================================

import torch
from transformers import AutoTokenizer

print("=" * 70)
print("STEP 39 — GEMMA TOKENIZATION")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"
MAX_SEQ_LENGTH = 256

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer:", type(tokenizer).__name__)
print("Pad token:", repr(tokenizer.pad_token))
print("EOS token:", repr(tokenizer.eos_token))

# Gemma normally has a valid EOS token but no separate pad token.
# Use EOS as padding for batching.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Effective pad token:", repr(tokenizer.pad_token))

# ------------------------------------------------------------
# Build training examples
# ------------------------------------------------------------

training_items = []

for i, item in enumerate(raw_examples):

    user_text = (
        "You are a spatial reasoning assistant.\n\n"
        f"Situation:\n{item['situation']}\n\n"
        f"Agent position:\n{item['agent_position']}\n\n"
        f"Agent rotation:\n{item['agent_rotation']}\n\n"
        f"Question:\n{item['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_text
        },
        {
            "role": "model",
            "content": item["answer"]
        }
    ]

    # --------------------------------------------------------
    # Full conversational sequence
    # --------------------------------------------------------

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # --------------------------------------------------------
    # User-only portion
    # --------------------------------------------------------

    user_messages = [
        {
            "role": "user",
            "content": user_text
        }
    ]

    user_text_formatted = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    prompt_tokens = tokenizer(
        user_text_formatted,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    prompt_length = len(prompt_tokens["input_ids"])

    # --------------------------------------------------------
    # Response-only labels
    # --------------------------------------------------------

    labels = [-100] * len(input_ids)

    for j in range(prompt_length, len(input_ids)):
        labels[j] = input_ids[j]

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    supervised_count = sum(
        1 for x in labels if x != -100
    )

    if supervised_count == 0:
        raise ValueError(
            f"Example {i} has zero supervised tokens."
        )

    training_items.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    })

# ------------------------------------------------------------
# Dataset statistics
# ------------------------------------------------------------

sequence_lengths = [
    len(x["input_ids"])
    for x in training_items
]

prompt_lengths = [
    sum(1 for x in item["labels"] if x == -100)
    for item in training_items
]

supervised_lengths = [
    sum(1 for x in item["labels"] if x != -100)
    for item in training_items
]

print()
print("=" * 70)
print("TOKENIZATION STATISTICS")
print("=" * 70)

print("Training items:", len(training_items))

print(
    "Sequence length:",
    "min =", min(sequence_lengths),
    "max =", max(sequence_lengths),
    "avg =", round(sum(sequence_lengths) / len(sequence_lengths), 2)
)

print(
    "Prompt tokens:",
    "min =", min(prompt_lengths),
    "max =", max(prompt_lengths),
    "avg =", round(sum(prompt_lengths) / len(prompt_lengths), 2)
)

print(
    "Supervised tokens:",
    "min =", min(supervised_lengths),
    "max =", max(supervised_lengths),
    "avg =", round(sum(supervised_lengths) / len(supervised_lengths), 2)
)

print(
    "Zero-supervision examples:",
    sum(1 for x in supervised_lengths if x == 0)
)

# ------------------------------------------------------------
# Verify expected size
# ------------------------------------------------------------

assert len(training_items) == 26623

assert all(
    len(x["input_ids"]) == len(x["labels"])
    for x in training_items
)

assert all(
    x > 0
    for x in supervised_lengths
)

print()
print("=" * 70)
print("SAMPLE DECODE")
print("=" * 70)

sample_ids = training_items[0]["input_ids"]

print(
    tokenizer.decode(
        sample_ids,
        skip_special_tokens=False
    )
)

print()
print("Supervised target:")
print(
    tokenizer.decode(
        [
            token
            for token in training_items[0]["labels"]
            if token != -100
        ],
        skip_special_tokens=False
    )
)

print()
print("=" * 70)
print("✓ TOKENIZATION COMPLETE")
print("=" * 70)

STEP 39 — GEMMA TOKENIZATION


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Tokenizer: GemmaTokenizer
Pad token: '<pad>'
EOS token: '<eos>'
Effective pad token: '<pad>'

TOKENIZATION STATISTICS
Training items: 26623
Sequence length: min = 103 max = 193 avg = 153.6
Prompt tokens: min = 100 max = 190 avg = 150.43
Supervised tokens: min = 3 max = 19 avg = 3.18
Zero-supervision examples: 0

SAMPLE DECODE
<bos><start_of_turn>user
You are a spatial reasoning assistant.

Situation:
I am facing a window and there is a desk on my right and a chair behind me.

Agent position:
[-0.9651003385573296, -1.2417634435553606, 0]

Agent rotation:
[0, 0, 0.09983341664682724, 0.9950041652780182]

Question:
What color is the desk to my right?<end_of_turn>
<start_of_turn>model
brown<end_of_turn>


Supervised target:
brown<end_of_turn>


✓ TOKENIZATION COMPLETE


In [ ]:
# ============================================================
# STEP 40 — LOAD FRESH GEMMA + LoRA
# ============================================================

import gc
import torch

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

print("=" * 70)
print("STEP 40 — FRESH GEMMA + LoRA")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"

# ------------------------------------------------------------
# Clean GPU memory
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

print("GPU memory before model:")
print(
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB allocated"
)

# ------------------------------------------------------------
# Load official pretrained Gemma
# ------------------------------------------------------------

print()
print("Loading official Gemma 2B...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

print("✓ Official pretrained Gemma loaded")

# ------------------------------------------------------------
# Attach fresh LoRA
# ------------------------------------------------------------

print()
print("Attaching fresh LoRA adapters...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✓ Fresh LoRA attached")

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print()
model.print_trainable_parameters()

print()
print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

print()
print(
    "GPU memory after model:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print()
print("=" * 70)
print("✓ FRESH MODEL READY")
print("=" * 70)

STEP 40 — FRESH GEMMA + LoRA
GPU memory before model:
0.0 GB allocated

Loading official Gemma 2B...


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Official pretrained Gemma loaded

Attaching fresh LoRA adapters...


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 49.5 MB/s eta 0:00:00


In [ ]:
import torchao
print("torchao version:", torchao.__version__)

torchao version: 0.18.0


In [ ]:
from peft import LoraConfig, get_peft_model

print("Attaching fresh LoRA adapters...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✓ Fresh LoRA attached")
model.print_trainable_parameters()

Attaching fresh LoRA adapters...
✓ Fresh LoRA attached
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


In [ ]:
from transformers import TrainingArguments
import inspect

print("=" * 70)
print("STEP 41 — CHECKPOINT / HUB SUPPORT")
print("=" * 70)

sig = inspect.signature(TrainingArguments.__init__)
params = sig.parameters

for name in [
    "save_strategy",
    "save_steps",
    "save_total_limit",
    "push_to_hub",
    "hub_model_id",
    "hub_strategy",
    "hub_always_push",
    "resume_from_checkpoint",
    "logging_steps",
]:
    if name in params:
        print(f"✓ {name}: supported")
    else:
        print(f"✗ {name}: NOT supported")

print("\nTransformers version:", __import__("transformers").__version__)

STEP 41 — CHECKPOINT / HUB SUPPORT
✓ save_strategy: supported
✓ save_steps: supported
✓ save_total_limit: supported
✓ push_to_hub: supported
✓ hub_model_id: supported
✓ hub_strategy: supported
✓ hub_always_push: supported
✓ resume_from_checkpoint: supported
✓ logging_steps: supported

Transformers version: 5.16.1


In [ ]:
from transformers import TrainingArguments

print("=" * 70)
print("STEP 42 — PERSISTENT TRAINING CONFIGURATION")
print("=" * 70)

OUTPUT_DIR = "/content/egospatial_gemma_v1"
REPO_ID = "Platinum04/EgoSpatial-Gemma-v1"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,

    # Warmup
    warmup_steps=100,

    # Precision
    fp16=True,

    # Memory / performance
    gradient_checkpointing=False,

    # Optimizer
    optim="adamw_torch",

    # Logging
    logging_steps=50,

    # Checkpointing
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    # Hugging Face Hub persistence
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_strategy="checkpoint",
    hub_always_push=True,

    # Reproducibility
    seed=42,
    data_seed=42,

    # Misc
    report_to="none",
    remove_unused_columns=False,
)

print("✓ TrainingArguments created")
print()
print("Output directory :", OUTPUT_DIR)
print("Hub repository   :", REPO_ID)
print("Epochs           :", training_args.num_train_epochs)
print("Effective batch  :", 2 * 4)
print("Learning rate    :", training_args.learning_rate)
print("Save every       :", training_args.save_steps, "steps")
print("Keep local       :", training_args.save_total_limit, "checkpoints")
print("Hub strategy     :", training_args.hub_strategy)
print("Always push      :", training_args.hub_always_push)
print("FP16             :", training_args.fp16)

STEP 42 — PERSISTENT TRAINING CONFIGURATION
✓ TrainingArguments created

Output directory : /content/egospatial_gemma_v1
Hub repository   : Platinum04/EgoSpatial-Gemma-v1
Epochs           : 1
Effective batch  : 8
Learning rate    : 0.0001
Save every       : 250 steps
Keep local       : 2 checkpoints
Hub strategy     : HubStrategy.CHECKPOINT
Always push      : True
FP16             : True


In [ ]:
from transformers import Trainer

print("=" * 70)
print("STEP 43 — TRAINER PRE-FLIGHT")
print("=" * 70)

# Dynamic padding collator
def training_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:
        pad_len = max_len - len(f["input_ids"])

        input_ids.append(
            torch.cat([
                f["input_ids"],
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])
        )

        attention_masks.append(
            torch.cat([
                f["attention_mask"],
                torch.zeros(pad_len, dtype=torch.long)
            ])
        )

        labels.append(
            torch.cat([
                f["labels"],
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels),
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=training_collator,
)

print("✓ Trainer created")
print()

print("Model type       :", type(model).__name__)
print("Dataset size     :", len(train_dataset))
print("Batch size       :", training_args.per_device_train_batch_size)
print("Grad accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch  :", (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
))
print("Total epochs     :", training_args.num_train_epochs)
print("Save interval    :", training_args.save_steps)
print("Hub repository   :", training_args.hub_model_id)

print()

# Verify one real batch
batch = next(iter(trainer.get_train_dataloader()))

print("Batch input shape :", tuple(batch["input_ids"].shape))
print("Batch label shape :", tuple(batch["labels"].shape))

# Verify supervised tokens exist
supervised_tokens = (batch["labels"] != -100).sum().item()
print("Supervised tokens :", supervised_tokens)

print()

# Verify model is ready
print("Model device      :", next(model.parameters()).device)
print("Model dtype       :", next(model.parameters()).dtype)
print("GPU allocated     :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

assert len(train_dataset) == 26623
assert supervised_tokens > 0
assert next(model.parameters()).is_cuda
assert next(model.parameters()).dtype == torch.float16

print()
print("✓ ALL PRE-FLIGHT CHECKS PASSED")
print()
print("READY FOR FULL TRAINING")

STEP 43 — TRAINER PRE-FLIGHT


NameError: name 'train_dataset' is not defined

In [ ]:
from datasets import Dataset

print("=" * 70)
print("STEP 43A — RECOVER TOKENIZED TRAINING DATASET")
print("=" * 70)

# Check whether the tokenized data variable already exists
if "tokenized_data" in globals():
    train_dataset = Dataset.from_list(tokenized_data)

elif "tokenized_train" in globals():
    train_dataset = tokenized_train

elif "train_data" in globals():
    train_dataset = Dataset.from_list(train_data)

else:
    raise NameError(
        "No tokenized dataset variable was found in memory. "
        "We need to rebuild it from the downloaded JSON files."
    )

print("✓ train_dataset recovered")
print("Number of examples:", len(train_dataset))
print("Columns:", train_dataset.column_names)

assert len(train_dataset) == 26623

print("✓ Dataset size verified: 26,623 examples")

STEP 43A — RECOVER TOKENIZED TRAINING DATASET


NameError: No tokenized dataset variable was found in memory. We need to rebuild it from the downloaded JSON files.

In [ ]:
from datasets import Dataset

print("=" * 70)
print("STEP 43B — REBUILD TOKENIZED TRAINING DATASET")
print("=" * 70)

# ------------------------------------------------------------------
# 1. Load the already-downloaded JSON files
# ------------------------------------------------------------------

QUESTIONS_PATH = "/content/egospatial_data/v1_balanced_questions_train_scannetv2.json"
ANNOTATIONS_PATH = "/content/egospatial_data/v1_balanced_sqa_annotations_train_scannetv2.json"

with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations_data = json.load(f)

questions = questions_data["questions"]
annotations = annotations_data["annotations"]

print("Questions   :", len(questions))
print("Annotations :", len(annotations))

# ------------------------------------------------------------------
# 2. Reconstruct the raw examples
# ------------------------------------------------------------------

annotations_by_id = {
    item["question_id"]: item
    for item in annotations
}

raw_examples = []

for q in questions:
    ann = annotations_by_id[q["question_id"]]

    position = ann["position"]
    rotation = ann["rotation"]

    answer = ann["answers"][0]["answer"]

    raw_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "question": q["question"],
        "answer": answer,
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ]
    })

print("Raw examples:", len(raw_examples))

assert len(raw_examples) == 26623

# ------------------------------------------------------------------
# 3. Recreate the exact Gemma chat-format examples
# ------------------------------------------------------------------

tokenized_examples = []

for item in raw_examples:

    user_text = f"""You are a spatial reasoning assistant.

Situation:
{item["situation"]}

Agent position:
{item["position"]}

Agent rotation:
{item["rotation"]}

Question:
{item["question"]}"""

    messages = [
        {
            "role": "user",
            "content": user_text
        },
        {
            "role": "model",
            "content": item["answer"]
        }
    ]

    # Full conversational sequence
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # User-only prompt
    prompt_text = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True
    )

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=256
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=256
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    prompt_length = len(prompt_tokens["input_ids"])

    labels = [-100] * prompt_length + input_ids[prompt_length:]

    # Safety check
    labels = labels[:len(input_ids)]

    tokenized_examples.append({
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    })

train_dataset = tokenized_examples

print("✓ Tokenization complete")
print("Training examples:", len(train_dataset))

# ------------------------------------------------------------------
# 4. Verify the exact properties we established previously
# ------------------------------------------------------------------

seq_lengths = [len(x["input_ids"]) for x in train_dataset]

supervised_lengths = [
    int((x["labels"] != -100).sum())
    for x in train_dataset
]

print()
print("Sequence length:")
print("  min :", min(seq_lengths))
print("  max :", max(seq_lengths))
print("  avg :", round(sum(seq_lengths) / len(seq_lengths), 2))

print()
print("Supervised tokens:")
print("  min :", min(supervised_lengths))
print("  max :", max(supervised_lengths))
print("  avg :", round(sum(supervised_lengths) / len(supervised_lengths), 2))

print()
print("Zero-supervision examples:",
      sum(x == 0 for x in supervised_lengths))

assert len(train_dataset) == 26623
assert min(seq_lengths) == 103
assert max(seq_lengths) == 193
assert min(supervised_lengths) == 3
assert max(supervised_lengths) == 19
assert sum(x == 0 for x in supervised_lengths) == 0

print()
print("✓ DATASET REBUILD VERIFIED")

STEP 43B — REBUILD TOKENIZED TRAINING DATASET
Questions   : 26623
Annotations : 26623
Raw examples: 26623
✓ Tokenization complete
Training examples: 26623

Sequence length:
  min : 103
  max : 193
  avg : 153.6

Supervised tokens:
  min : 3
  max : 19
  avg : 3.18

Zero-supervision examples: 0

✓ DATASET REBUILD VERIFIED


In [ ]:
print("=" * 70)
print("STEP 43C — DATASET REBUILD STATUS")
print("=" * 70)

print("tokenized_examples exists:",
      "tokenized_examples" in globals())

print("train_dataset exists:",
      "train_dataset" in globals())

if "tokenized_examples" in globals():
    print("Tokenized examples:", len(tokenized_examples))

if "train_dataset" in globals():
    print("Train dataset:", len(train_dataset))

print()
print("Tokenizer:", type(tokenizer).__name__)
print("MAX_SEQ_LENGTH:", 256)

STEP 43C — DATASET REBUILD STATUS
tokenized_examples exists: True
train_dataset exists: True
Tokenized examples: 26623
Train dataset: 26623

Tokenizer: GemmaTokenizer
MAX_SEQ_LENGTH: 256


In [ ]:
from transformers import Trainer

print("=" * 70)
print("STEP 43D — FINAL TRAINER PRE-FLIGHT")
print("=" * 70)

def training_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:
        pad_len = max_len - len(f["input_ids"])

        input_ids.append(
            torch.cat([
                f["input_ids"],
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])
        )

        attention_masks.append(
            torch.cat([
                f["attention_mask"],
                torch.zeros(pad_len, dtype=torch.long)
            ])
        )

        labels.append(
            torch.cat([
                f["labels"],
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels),
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=training_collator,
)

print("✓ Trainer created")
print()

# ------------------------------------------------------------
# Inspect one REAL training batch
# ------------------------------------------------------------

batch = next(iter(trainer.get_train_dataloader()))

print("Batch input shape :", tuple(batch["input_ids"].shape))
print("Batch label shape :", tuple(batch["labels"].shape))

supervised_tokens = (batch["labels"] != -100).sum().item()

print("Supervised tokens :", supervised_tokens)

# ------------------------------------------------------------
# Model checks
# ------------------------------------------------------------

param = next(model.parameters())

print()
print("Model type        :", type(model).__name__)
print("Model device      :", param.device)
print("Model dtype       :", param.dtype)
print(
    "GPU allocated     :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

# ------------------------------------------------------------
# Dataset / training checks
# ------------------------------------------------------------

print()
print("Dataset size      :", len(train_dataset))
print("Batch size        :", training_args.per_device_train_batch_size)
print("Grad accumulation :", training_args.gradient_accumulation_steps)
print(
    "Effective batch   :",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("Learning rate     :", training_args.learning_rate)
print("Save every        :", training_args.save_steps)
print("Hub repository    :", training_args.hub_model_id)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(train_dataset) == 26623
assert batch["input_ids"].shape[0] == 2
assert batch["labels"].shape == batch["input_ids"].shape
assert supervised_tokens > 0
assert param.is_cuda
assert param.dtype == torch.float16

print()
print("=" * 70)
print("✓ ALL FINAL PRE-FLIGHT CHECKS PASSED")
print("=" * 70)
print()
print("The training run is ready.")

STEP 43D — FINAL TRAINER PRE-FLIGHT
✓ Trainer created

Batch input shape : (2, 162)
Batch label shape : (2, 162)
Supervised tokens : 6

Model type        : PeftModelForCausalLM
Model device      : cuda:0
Model dtype       : torch.float16
GPU allocated     : 4.95 GB

Dataset size      : 26623
Batch size        : 2
Grad accumulation : 4
Effective batch   : 8
Learning rate     : 0.0001
Save every        : 250
Hub repository    : Platinum04/EgoSpatial-Gemma-v1

✓ ALL FINAL PRE-FLIGHT CHECKS PASSED

The training run is ready.


In [ ]:
print("=" * 70)
print("STEP 44 — STARTING FULL EGO-SPATIAL GEMMA TRAINING")
print("=" * 70)

print()
print("Dataset        :", len(train_dataset))
print("Epochs         :", training_args.num_train_epochs)
print("Effective batch:", 8)
print("Learning rate  :", training_args.learning_rate)
print("Total steps    :", trainer.num_training_steps)
print("Checkpoint     :", "every 250 steps")
print("Hub repo       :", training_args.hub_model_id)
print()

print("Starting training...")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("✓ TRAINING COMPLETED")
print("=" * 70)

print("Final training loss:", train_result.training_loss)
print("Global steps       :", trainer.state.global_step)

STEP 44 — STARTING FULL EGO-SPATIAL GEMMA TRAINING

Dataset        : 26623
Epochs         : 1
Effective batch: 8
Learning rate  : 0.0001


AttributeError: 'Trainer' object has no attribute 'num_training_steps'

In [ ]:
print("=" * 70)
print("STEP 44A — STARTING FULL EGO-SPATIAL GEMMA TRAINING")
print("=" * 70)

print()
print("Dataset        :", len(train_dataset))
print("Epochs         :", training_args.num_train_epochs)
print("Effective batch:", 8)
print("Learning rate  :", training_args.learning_rate)
print("Checkpoint     :", "every 250 steps")
print("Hub repo       :", training_args.hub_model_id)
print()

print("Starting training...")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("✓ TRAINING COMPLETED")
print("=" * 70)

print("Final training loss:", train_result.training_loss)
print("Global steps       :", trainer.state.global_step)

STEP 44A — STARTING FULL EGO-SPATIAL GEMMA TRAINING

Dataset        : 26623
Epochs         : 1
Effective batch: 8
Learning rate  : 0.0001
Checkpoint     : every 250 steps
Hub repo       : Platinum04/EgoSpatial-Gemma-v1

Starting training...


Step,Training Loss
50,5.413882
100,0.779909
150,0.791308
200,0.725863
250,0.725307
300,0.606733
350,0.651413
400,0.645801
450,0.648201
500,0.630859



✓ TRAINING COMPLETED
Final training loss: 0.658006216184451
Global steps       : 3328


In [ ]:
print("=" * 70)
print("STEP 45 — SAVE FINAL EGO-SPATIAL GEMMA")
print("=" * 70)

FINAL_DIR = "/content/egospatial_gemma_v1/final"

print("Saving final model...")
trainer.save_model(FINAL_DIR)

print("✓ Final model saved locally")

print()
print("Pushing final model to Hugging Face Hub...")

trainer.push_to_hub(
    commit_message="Final EgoSpatial-Gemma v1 after 1 epoch on EgoSpatial training set"
)

print()
print("=" * 70)
print("✓ FINAL MODEL SAVED AND PUSHED")
print("=" * 70)

print("Local path :", FINAL_DIR)
print("Hub repo   :", training_args.hub_model_id)

STEP 45 — SAVE FINAL EGO-SPATIAL GEMMA
Saving final model...
✓ Final model saved locally

Pushing final model to Hugging Face Hub...


No files have been modified since last commit. Skipping to prevent empty commit.



✓ FINAL MODEL SAVED AND PUSHED
Local path : /content/egospatial_gemma_v1/final
Hub repo   : Platinum04/EgoSpatial-Gemma-v1


In [ ]:
print("=" * 70)
print("STEP 46 — FINAL MODEL SANITY TEST")
print("=" * 70)

import torch

model.eval()

# Use the first validation example we already reconstructed
test_item = val_raw_examples[0]

user_text = f"""You are a spatial reasoning assistant.

Situation:
{test_item["situation"]}

Agent position:
{test_item["position"]}

Agent rotation:
{test_item["rotation"]}

Question:
{test_item["question"]}"""

messages = [
    {
        "role": "user",
        "content": user_text
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print()
print("Question :", test_item["question"])
print("Expected :", test_item["answer"])
print("Predicted:", repr(prediction))

print()
print("=" * 70)
print("✓ SANITY TEST COMPLETE")
print("=" * 70)

STEP 46 — FINAL MODEL SANITY TEST


NameError: name 'val_raw_examples' is not defined

In [ ]:
import json

print("=" * 70)
print("STEP 46A — REBUILD VALIDATION EXAMPLES")
print("=" * 70)

VAL_QUESTIONS_PATH = (
    "/content/egospatial_data/"
    "v1_balanced_questions_val_scannetv2.json"
)

VAL_ANNOTATIONS_PATH = (
    "/content/egospatial_data/"
    "v1_balanced_sqa_annotations_val_scannetv2.json"
)

# ------------------------------------------------------------
# 1. Load validation JSON
# ------------------------------------------------------------

with open(VAL_QUESTIONS_PATH, "r", encoding="utf-8") as f:
    val_questions_data = json.load(f)

with open(VAL_ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    val_annotations_data = json.load(f)

val_questions = val_questions_data["questions"]
val_annotations = val_annotations_data["annotations"]

print("Validation questions   :", len(val_questions))
print("Validation annotations :", len(val_annotations))

# ------------------------------------------------------------
# 2. Index annotations by question ID
# ------------------------------------------------------------

val_annotations_by_id = {
    item["question_id"]: item
    for item in val_annotations
}

# ------------------------------------------------------------
# 3. Reconstruct validation examples
# ------------------------------------------------------------

val_raw_examples = []

for q in val_questions:

    ann = val_annotations_by_id[q["question_id"]]

    position = ann["position"]
    rotation = ann["rotation"]

    answer = ann["answers"][0]["answer"]

    val_raw_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "question": q["question"],
        "answer": answer,
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ]
    })

print("Validation examples  :", len(val_raw_examples))

# ------------------------------------------------------------
# 4. Integrity checks
# ------------------------------------------------------------

assert len(val_questions) == 3261
assert len(val_annotations) == 3261
assert len(val_raw_examples) == 3261

question_ids = [q["question_id"] for q in val_questions]
annotation_ids = [a["question_id"] for a in val_annotations]

assert question_ids == annotation_ids

print("Unique scenes        :",
      len(set(x["scene_id"] for x in val_raw_examples)))

print()
print("First validation example:")
print("Scene    :", val_raw_examples[0]["scene_id"])
print("Question :", val_raw_examples[0]["question"])
print("Answer   :", val_raw_examples[0]["answer"])
print("Position :", val_raw_examples[0]["position"])
print("Rotation :", val_raw_examples[0]["rotation"])

print()
print("=" * 70)
print("✓ VALIDATION SET REBUILT")
print("=" * 70)

STEP 46A — REBUILD VALIDATION EXAMPLES


FileNotFoundError: [Errno 2] No such file or directory: '/content/egospatial_data/v1_balanced_questions_val_scannetv2.json'

In [ ]:
import os
import urllib.request

print("=" * 70)
print("STEP 46B — DOWNLOAD VALIDATION DATA")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Platinum04/EgoSpatial-Dataset/main/"
)

val_files = [
    "v1_balanced_questions_val_scannetv2.json",
    "v1_balanced_sqa_annotations_val_scannetv2.json",
]

for filename in val_files:
    path = os.path.join(DATA_DIR, filename)

    print(f"\nDownloading: {filename}")

    urllib.request.urlretrieve(
        BASE_URL + filename,
        path
    )

    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"✓ Saved: {path}")
    print(f"  Size: {size_mb:.2f} MB")

print()
print("=" * 70)
print("✓ VALIDATION FILES DOWNLOADED")
print("=" * 70)

STEP 46B — DOWNLOAD VALIDATION DATA

Downloading: v1_balanced_questions_val_scannetv2.json
✓ Saved: /content/egospatial_data/v1_balanced_questions_val_scannetv2.json
  Size: 1.32 MB

Downloading: v1_balanced_sqa_annotations_val_scannetv2.json
✓ Saved: /content/egospatial_data/v1_balanced_sqa_annotations_val_scannetv2.json
  Size: 1.06 MB

✓ VALIDATION FILES DOWNLOADED


In [ ]:
import json

print("=" * 70)
print("STEP 46C — REBUILD VALIDATION EXAMPLES")
print("=" * 70)

VAL_QUESTIONS_PATH = (
    "/content/egospatial_data/"
    "v1_balanced_questions_val_scannetv2.json"
)

VAL_ANNOTATIONS_PATH = (
    "/content/egospatial_data/"
    "v1_balanced_sqa_annotations_val_scannetv2.json"
)

with open(VAL_QUESTIONS_PATH, "r", encoding="utf-8") as f:
    val_questions_data = json.load(f)

with open(VAL_ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    val_annotations_data = json.load(f)

val_questions = val_questions_data["questions"]
val_annotations = val_annotations_data["annotations"]

print("Validation questions   :", len(val_questions))
print("Validation annotations :", len(val_annotations))

val_annotations_by_id = {
    item["question_id"]: item
    for item in val_annotations
}

val_raw_examples = []

for q in val_questions:
    ann = val_annotations_by_id[q["question_id"]]

    position = ann["position"]
    rotation = ann["rotation"]

    val_raw_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "question": q["question"],
        "answer": ann["answers"][0]["answer"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ]
    })

print("Validation examples  :", len(val_raw_examples))

# Integrity checks
assert len(val_questions) == 3261
assert len(val_annotations) == 3261
assert len(val_raw_examples) == 3261

assert (
    [q["question_id"] for q in val_questions]
    ==
    [a["question_id"] for a in val_annotations]
)

print("Unique scenes        :",
      len(set(x["scene_id"] for x in val_raw_examples)))

print()
print("First validation example:")
print("Scene    :", val_raw_examples[0]["scene_id"])
print("Question :", val_raw_examples[0]["question"])
print("Answer   :", val_raw_examples[0]["answer"])
print("Position :", val_raw_examples[0]["position"])
print("Rotation :", val_raw_examples[0]["rotation"])

print()
print("=" * 70)
print("✓ VALIDATION SET REBUILT")
print("=" * 70)

STEP 46C — REBUILD VALIDATION EXAMPLES


FileNotFoundError: [Errno 2] No such file or directory: '/content/egospatial_data/v1_balanced_questions_val_scannetv2.json'

In [ ]:
import os
import torch

print("=" * 70)
print("STEP 46D — RUNTIME RECOVERY CHECK")
print("=" * 70)

print()
print("GPU available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

print()
print("MODEL IN MEMORY       :", "model" in globals())
print("TOKENIZER IN MEMORY   :", "tokenizer" in globals())
print("TRAINER IN MEMORY     :", "trainer" in globals())
print("TRAIN_DATASET MEMORY  :", "train_dataset" in globals())

print()
print("DATA DIRECTORY EXISTS :", os.path.exists("/content/egospatial_data"))

if os.path.exists("/content/egospatial_data"):
    print("Files currently in /content/egospatial_data:")
    for filename in os.listdir("/content/egospatial_data"):
        print("  -", filename)

print()
print("=" * 70)
print("✓ RECOVERY CHECK COMPLETE")
print("=" * 70)

STEP 46D — RUNTIME RECOVERY CHECK

GPU available : True
GPU           : Tesla T4

MODEL IN MEMORY       : False
TOKENIZER IN MEMORY   : False
TRAINER IN MEMORY     : False
TRAIN_DATASET MEMORY  : False

DATA DIRECTORY EXISTS : False

✓ RECOVERY CHECK COMPLETE


In [ ]:
from huggingface_hub import HfApi

print("=" * 70)
print("STEP 47 — INSPECT PERSISTED EGO-SPATIAL MODEL")
print("=" * 70)

REPO_ID = "Platinum04/EgoSpatial-Gemma-v1"

api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

print("Repository:", REPO_ID)
print()
print("Files persisted on Hugging Face:")
print("-" * 70)

for f in files:
    print(f)

print()
print("=" * 70)
print("TOTAL FILES:", len(files))
print("=" * 70)

STEP 47 — INSPECT PERSISTED EGO-SPATIAL MODEL


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6aa2883a-767087ec4b65783d0826a437;02094467-7852-4f9f-8e55-44c1e23c92b1)

Repository Not Found for url: https://huggingface.co/api/models/Platinum04/EgoSpatial-Gemma-v1/tree/main?recursive=true&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
from huggingface_hub import login, whoami

print("=" * 70)
print("STEP 47A — HUGGING FACE RE-AUTHENTICATION")
print("=" * 70)

login()

info = whoami()

print()
print("✓ Hugging Face authentication successful")
print("Username:", info["name"])

assert info["name"] == "Platinum04"

print("✓ Correct Hugging Face account confirmed")

STEP 47A — HUGGING FACE RE-AUTHENTICATION



✓ Hugging Face authentication successful
Username: Platinum04
✓ Correct Hugging Face account confirmed


In [ ]:
from huggingface_hub import HfApi

print("=" * 70)
print("STEP 47B — INSPECT PERSISTED EGO-SPATIAL MODEL")
print("=" * 70)

REPO_ID = "Platinum04/EgoSpatial-Gemma-v1"

api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

print("Repository:", REPO_ID)
print()
print("Files persisted on Hugging Face:")
print("-" * 70)

for f in files:
    print(f)

print()
print("=" * 70)
print("TOTAL FILES:", len(files))
print("=" * 70)

STEP 47B — INSPECT PERSISTED EGO-SPATIAL MODEL
Repository: Platinum04/EgoSpatial-Gemma-v1

Files persisted on Hugging Face:
----------------------------------------------------------------------
.gitattributes
README.md
adapter_config.json
adapter_model.safetensors
final/README.md
final/adapter_config.json
final/adapter_model.safetensors
final/training_args.bin
last-checkpoint/README.md
last-checkpoint/adapter_config.json
last-checkpoint/adapter_model.safetensors
last-checkpoint/optimizer.pt
last-checkpoint/rng_state.pth
last-checkpoint/scaler.pt
last-checkpoint/scheduler.pt
last-checkpoint/trainer_state.json
last-checkpoint/training_args.bin
training_args.bin

TOTAL FILES: 18


In [ ]:
import gc
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("=" * 70)
print("STEP 48 — LOAD TRAINED EGO-SPATIAL GEMMA")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"
ADAPTER_REPO = "Platinum04/EgoSpatial-Gemma-v1"

gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading base Gemma...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

print("✓ Base Gemma loaded")

print()
print("Loading trained LoRA adapter from Hugging Face...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
    is_trainable=False
)

model.eval()

print("✓ Trained LoRA adapter loaded")

print()
print("Model type :", type(model).__name__)
print("Device     :", next(model.parameters()).device)
print("Dtype      :", next(model.parameters()).dtype)

print()
print("=" * 70)
print("✓ TRAINED MODEL READY FOR EVALUATION")
print("=" * 70)

STEP 48 — LOAD TRAINED EGO-SPATIAL GEMMA
Loading tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Base Gemma loaded

Loading trained LoRA adapter from Hugging Face...


adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.8 MB/s eta 0:00:00


In [ ]:
import torchao
print("torchao version:", torchao.__version__)

torchao version: 0.18.0


In [ ]:
import torch
import os

print("=" * 70)
print("STEP 48C — POST-RESTART RUNTIME CHECK")
print("=" * 70)

print()
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print(
        "GPU memory     :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

print()
print("Model in memory     :", "model" in globals())
print("Base model in memory:", "base_model" in globals())
print("Tokenizer in memory :", "tokenizer" in globals())
print("Validation data     :", "val_raw_examples" in globals())

print()
print("/content/egospatial_data exists:",
      os.path.exists("/content/egospatial_data"))

print()
print("=" * 70)
print("✓ RUNTIME CHECK COMPLETE")
print("=" * 70)

STEP 48C — POST-RESTART RUNTIME CHECK

CUDA available : True
GPU            : Tesla T4
GPU memory     : 14.56 GB

Model in memory     : False
Base model in memory: False
Tokenizer in memory : False
Validation data     : False

/content/egospatial_data exists: False

✓ RUNTIME CHECK COMPLETE


In [ ]:
import subprocess
import sys

print("=" * 70)
print("STEP 48D — RESTORE EVALUATION ENVIRONMENT")
print("=" * 70)

# Fix PEFT/torchao compatibility in this fresh Colab runtime
print("Installing compatible torchao...")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "torchao>=0.16.0"
])

print("✓ torchao installation complete")

# Verify torchao
import torchao

print("torchao version:", torchao.__version__)
assert tuple(map(int, torchao.__version__.split(".")[:2])) >= (0, 16)

# Re-authenticate Hugging Face
print()
print("Authenticating Hugging Face...")

from huggingface_hub import login, whoami

login()

info = whoami()

print("✓ Logged in as:", info["name"])
assert info["name"] == "Platinum04"

print()
print("=" * 70)
print("✓ EVALUATION ENVIRONMENT READY")
print("=" * 70)

STEP 48D — RESTORE EVALUATION ENVIRONMENT
Installing compatible torchao...


✓ torchao installation complete
torchao version: 0.18.0

Authenticating Hugging Face...


✓ Logged in as: Platinum04

✓ EVALUATION ENVIRONMENT READY


In [ ]:
import gc
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("=" * 70)
print("STEP 48E — LOAD TRAINED EGO-SPATIAL GEMMA")
print("=" * 70)

MODEL_NAME = "google/gemma-2-2b-it"
ADAPTER_REPO = "Platinum04/EgoSpatial-Gemma-v1"

gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading base Gemma 2B...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

print("✓ Base Gemma loaded")

print()
print("Loading trained LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
    is_trainable=False
)

model.eval()

print("✓ Trained LoRA adapter loaded")

print()
print("Model type :", type(model).__name__)
print("Device     :", next(model.parameters()).device)
print("Dtype      :", next(model.parameters()).dtype)

print()
print(
    "GPU memory :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print()
print("=" * 70)
print("✓ TRAINED EGO-SPATIAL MODEL READY")
print("=" * 70)

STEP 48E — LOAD TRAINED EGO-SPATIAL GEMMA
Loading tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

✓ Base Gemma loaded

Loading trained LoRA adapter...


adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 83.1MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✓ Trained LoRA adapter loaded

Model type : PeftModelForCausalLM
Device     : cuda:0
Dtype      : torch.float16

GPU memory : 4.95 GB

✓ TRAINED EGO-SPATIAL MODEL READY


In [ ]:
import os
import json
import urllib.request

print("=" * 70)
print("STEP 49 — RESTORE VALIDATION DATA")
print("=" * 70)

DATA_DIR = "/content/egospatial_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Platinum04/EgoSpatial-Dataset/main/"
)

files = [
    "v1_balanced_questions_val_scannetv2.json",
    "v1_balanced_sqa_annotations_val_scannetv2.json",
]

for filename in files:
    path = os.path.join(DATA_DIR, filename)

    print(f"Downloading {filename}...")

    urllib.request.urlretrieve(
        BASE_URL + filename,
        path
    )

    print(
        "✓",
        filename,
        f"({os.path.getsize(path) / 1024**2:.2f} MB)"
    )

# Load
with open(
    os.path.join(DATA_DIR, files[0]),
    "r",
    encoding="utf-8"
) as f:
    val_questions_data = json.load(f)

with open(
    os.path.join(DATA_DIR, files[1]),
    "r",
    encoding="utf-8"
) as f:
    val_annotations_data = json.load(f)

val_questions = val_questions_data["questions"]
val_annotations = val_annotations_data["annotations"]

print()
print("Validation questions   :", len(val_questions))
print("Validation annotations :", len(val_annotations))

assert len(val_questions) == 3261
assert len(val_annotations) == 3261

print()
print("=" * 70)
print("✓ VALIDATION DATA RESTORED")
print("=" * 70)

STEP 49 — RESTORE VALIDATION DATA
✓ v1_balanced_questions_val_scannetv2.json (1.32 MB)
✓ v1_balanced_sqa_annotations_val_scannetv2.json (1.06 MB)

Validation questions   : 3261
Validation annotations : 3261

✓ VALIDATION DATA RESTORED


In [ ]:
print("=" * 70)
print("STEP 50 — BUILD VALIDATION EXAMPLES")
print("=" * 70)

# Match each question with its annotation using question_id
val_annotations_by_id = {
    item["question_id"]: item
    for item in val_annotations
}

val_raw_examples = []

for q in val_questions:
    ann = val_annotations_by_id[q["question_id"]]

    position = ann["position"]
    rotation = ann["rotation"]

    val_raw_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "question": q["question"],
        "answer": ann["answers"][0]["answer"],
        "position": [
            position["x"],
            position["y"],
            position["z"]
        ],
        "rotation": [
            rotation["_x"],
            rotation["_y"],
            rotation["_z"],
            rotation["_w"]
        ]
    })

print("Validation examples :", len(val_raw_examples))
print(
    "Unique scenes       :",
    len(set(x["scene_id"] for x in val_raw_examples))
)

# Integrity checks
assert len(val_raw_examples) == 3261
assert len(set(x["scene_id"] for x in val_raw_examples)) == 65

print()
print("First validation example:")
print("Scene    :", val_raw_examples[0]["scene_id"])
print("Situation:", val_raw_examples[0]["situation"])
print("Question :", val_raw_examples[0]["question"])
print("Answer   :", val_raw_examples[0]["answer"])
print("Position :", val_raw_examples[0]["position"])
print("Rotation :", val_raw_examples[0]["rotation"])

print()
print("=" * 70)
print("✓ VALIDATION EXAMPLES READY")
print("=" * 70)

STEP 50 — BUILD VALIDATION EXAMPLES
Validation examples : 3261
Unique scenes       : 65

First validation example:
Scene    : scene0249_00
Situation: I am wiping a table that has a trash can next to it, while having a large table behind me with lots of chairs.
Question : Which direction should I toss a used napkin?
Answer   : right
Position : [-1.612321909455232, 3.8766019062927524, 0]
Rotation : [0, 0, 0.9436221923009414, -0.33102440725287985]

✓ VALIDATION EXAMPLES READY


In [ ]:
print("=" * 70)
print("STEP 51 — TRAINED MODEL SANITY INFERENCE")
print("=" * 70)

import torch

model.eval()

item = val_raw_examples[0]

user_text = f"""You are a spatial reasoning assistant.

Situation:
{item["situation"]}

Agent position:
{item["position"]}

Agent rotation:
{item["rotation"]}

Question:
{item["question"]}"""

messages = [
    {
        "role": "user",
        "content": user_text
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print()
print("Question :", item["question"])
print("Expected :", item["answer"])
print("Predicted:", repr(prediction))

print()
print("=" * 70)

if prediction.lower() == item["answer"].lower():
    print("✓ SANITY TEST PASSED — EXACT MATCH")
else:
    print("⚠ SANITY TEST — PREDICTION DOES NOT MATCH")

print("=" * 70)

STEP 51 — TRAINED MODEL SANITY INFERENCE

Question : Which direction should I toss a used napkin?
Expected : right
Predicted: 'right\ntrash can'

⚠ SANITY TEST — PREDICTION DOES NOT MATCH


In [ ]:
print("=" * 70)
print("STEP 52 — INSPECT GENERATION TOKENS")
print("=" * 70)

print()
print("Raw generated text:")
print(repr(tokenizer.decode(
    generated_tokens,
    skip_special_tokens=False
)))

print()
print("Generated token IDs:")
print(generated_tokens.tolist())

print()
print("Generated tokens individually:")

for token_id in generated_tokens.tolist():
    token = tokenizer.decode(
        [token_id],
        skip_special_tokens=False
    )
    print(token_id, repr(token))

print()
print("=" * 70)
print("✓ GENERATION INSPECTION COMPLETE")
print("=" * 70)

STEP 52 — INSPECT GENERATION TOKENS

Raw generated text:
'right<end_of_turn>\ntrash can<end_of_turn>\n<end_of_turn>'

Generated token IDs:
[1331, 107, 108, 64038, 798, 107, 108, 107]

Generated tokens individually:
1331 'right'
107 '<end_of_turn>'
108 '\n'
64038 'trash'
798 ' can'
107 '<end_of_turn>'
108 '\n'
107 '<end_of_turn>'

✓ GENERATION INSPECTION COMPLETE


In [ ]:
print("=" * 70)
print("STEP 53 — VERIFY CONTROLLED ANSWER EXTRACTION")
print("=" * 70)

# Find the first end-of-turn token
eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")

generated_list = generated_tokens.tolist()

if eot_id in generated_list:
    eot_index = generated_list.index(eot_id)
    answer_tokens = generated_list[:eot_index]
else:
    answer_tokens = generated_list

extracted_answer = tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
).strip()

expected_answer = item["answer"].strip()

print()
print("Raw generation     :", repr(
    tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )
))

print("Extracted answer   :", repr(extracted_answer))
print("Expected answer    :", repr(expected_answer))

print()

if extracted_answer.lower() == expected_answer.lower():
    print("✓ CONTROLLED EXTRACTION = EXACT MATCH")
else:
    print("⚠ CONTROLLED EXTRACTION = MISMATCH")

print()
print("Evaluation rule:")
print("1. Generate model output")
print("2. Stop at FIRST <end_of_turn>")
print("3. Decode preceding tokens")
print("4. Strip whitespace")
print("5. Case-insensitive exact comparison")

print()
print("=" * 70)
print("✓ EXTRACTION RULE VERIFIED")
print("=" * 70)

STEP 53 — VERIFY CONTROLLED ANSWER EXTRACTION

Raw generation     : 'right<end_of_turn>\ntrash can<end_of_turn>\n<end_of_turn>'
Extracted answer   : 'right'
Expected answer    : 'right'

✓ CONTROLLED EXTRACTION = EXACT MATCH

Evaluation rule:
1. Generate model output
2. Stop at FIRST <end_of_turn>
3. Decode preceding tokens
4. Strip whitespace
5. Case-insensitive exact comparison

✓ EXTRACTION RULE VERIFIED


In [ ]:
import time
import torch

print("=" * 70)
print("STEP 54 — FULL EGO-SPATIAL VALIDATION")
print("=" * 70)

model.eval()

# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

MAX_NEW_TOKENS = 12
N = len(val_raw_examples)

eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")

val_results = []

correct = 0
start_time = time.time()

print()
print("Validation examples :", N)
print("Max new tokens      :", MAX_NEW_TOKENS)
print("Decoding            : deterministic")
print("Answer extraction   : first <end_of_turn>")
print()

# ------------------------------------------------------------
# Evaluation loop
# ------------------------------------------------------------

for i, item in enumerate(val_raw_examples):

    user_text = f"""You are a spatial reasoning assistant.

Situation:
{item["situation"]}

Agent position:
{item["position"]}

Agent rotation:
{item["rotation"]}

Question:
{item["question"]}"""

    messages = [
        {
            "role": "user",
            "content": user_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eot_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_list = generated_tokens.tolist()

    # --------------------------------------------------------
    # Extract only the first answer segment
    # --------------------------------------------------------

    if eot_id in generated_list:
        eot_index = generated_list.index(eot_id)
        answer_tokens = generated_list[:eot_index]
    else:
        answer_tokens = generated_list

    prediction = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

    expected = item["answer"].strip()

    is_correct = (
        prediction.lower() == expected.lower()
    )

    if is_correct:
        correct += 1

    val_results.append({
        "index": i,
        "scene_id": item["scene_id"],
        "question": item["question"],
        "expected": expected,
        "prediction": prediction,
        "correct": is_correct,
    })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 100 == 0 or i == 0:
        elapsed = time.time() - start_time
        accuracy = correct / (i + 1) * 100
        rate = (i + 1) / elapsed

        print(
            f"[{i+1:4d}/{N}] "
            f"Accuracy: {accuracy:6.2f}% | "
            f"Speed: {rate:.2f} ex/s | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

# ------------------------------------------------------------
# Final statistics
# ------------------------------------------------------------

elapsed = time.time() - start_time
accuracy = correct / N * 100

print()
print("=" * 70)
print("✓ FULL VALIDATION COMPLETE")
print("=" * 70)

print()
print("Total examples :", N)
print("Correct        :", correct)
print("Incorrect      :", N - correct)
print("Accuracy       :", f"{accuracy:.2f}%")
print("Time           :", f"{elapsed/60:.2f} minutes")
print("Speed          :", f"{N/elapsed:.2f} examples/sec")

print()
print("Previous zero-shot baseline: 19.00% (19/100)")
print("Fine-tuned validation      :", f"{accuracy:.2f}%")

print()
print("=" * 70)

STEP 54 — FULL EGO-SPATIAL VALIDATION

Validation examples : 3261
Max new tokens      : 12
Decoding            : deterministic
Answer extraction   : first <end_of_turn>

[   1/3261] Accuracy: 100.00% | Speed: 4.49 ex/s | Elapsed: 0.0 min
[ 100/3261] Accuracy:  49.00% | Speed: 4.99 ex/s | Elapsed: 0.3 min
[ 200/3261] Accuracy:  49.00% | Speed: 4.98 ex/s | Elapsed: 0.7 min
[ 300/3261] Accuracy:  49.33% | Speed: 4.94 ex/s | Elapsed: 1.0 min
[ 400/3261] Accuracy:  48.25% | Speed: 4.93 ex/s | Elapsed: 1.4 min
[ 500/3261] Accuracy:  49.20% | Speed: 4.98 ex/s | Elapsed: 1.7 min
[ 600/3261] Accuracy:  50.33% | Speed: 4.97 ex/s | Elapsed: 2.0 min
[ 700/3261] Accuracy:  50.71% | Speed: 4.96 ex/s | Elapsed: 2.4 min
[ 800/3261] Accuracy:  50.75% | Speed: 4.97 ex/s | Elapsed: 2.7 min
[ 900/3261] Accuracy:  51.00% | Speed: 4.96 ex/s | Elapsed: 3.0 min
[1000/3261] Accuracy:  51.10% | Speed: 4.97 ex/s | Elapsed: 3.4 min
[1100/3261] Accuracy:  51.00% | Speed: 4.98 ex/s | Elapsed: 3.7 min
[1200/3261] Ac

In [ ]:
from collections import Counter, defaultdict

print("=" * 70)
print("STEP 55 — VALIDATION PREDICTION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Ground-truth and prediction distributions
# ------------------------------------------------------------

ground_truth_counts = Counter(
    r["expected"].lower()
    for r in val_results
)

prediction_counts = Counter(
    r["prediction"].lower()
    for r in val_results
)

print()
print("TOP 20 GROUND-TRUTH ANSWERS")
print("-" * 70)

for answer, count in ground_truth_counts.most_common(20):
    print(f"{answer:25s} {count:5d}")

print()
print("TOP 20 MODEL PREDICTIONS")
print("-" * 70)

for answer, count in prediction_counts.most_common(20):
    print(f"{answer:25s} {count:5d}")

# ------------------------------------------------------------
# Accuracy by ground-truth answer
# ------------------------------------------------------------

stats = defaultdict(lambda: {
    "total": 0,
    "correct": 0
})

for r in val_results:
    answer = r["expected"].lower()

    stats[answer]["total"] += 1

    if r["correct"]:
        stats[answer]["correct"] += 1

print()
print("ACCURACY BY GROUND-TRUTH ANSWER")
print("-" * 70)

rows = []

for answer, s in stats.items():
    accuracy = 100 * s["correct"] / s["total"]

    rows.append((
        s["total"],
        answer,
        s["correct"],
        accuracy
    ))

for total, answer, correct_count, accuracy in sorted(
    rows,
    reverse=True
)[:30]:

    print(
        f"{answer:25s} "
        f"n={total:4d} "
        f"correct={correct_count:4d} "
        f"accuracy={accuracy:6.2f}%"
    )

# ------------------------------------------------------------
# Prediction concentration
# ------------------------------------------------------------

print()
print("PREDICTION CONCENTRATION")
print("-" * 70)

total_predictions = len(val_results)

for k in [5, 10, 20]:
    top_k_count = sum(
        count
        for _, count in prediction_counts.most_common(k)
    )

    percentage = 100 * top_k_count / total_predictions

    print(
        f"Top {k:2d} predicted answers cover "
        f"{top_k_count:4d}/{total_predictions} "
        f"({percentage:.2f}%) of predictions"
    )

print()
print("=" * 70)
print("✓ PREDICTION ANALYSIS COMPLETE")
print("=" * 70)

STEP 55 — VALIDATION PREDICTION ANALYSIS

TOP 20 GROUND-TRUTH ANSWERS
----------------------------------------------------------------------
no                          322
yes                         302
two                         184
right                       175
one                         152
left                        145
odd                         102
even                         83
brown                        77
table                        63
rectangular                  58
black                        55
white                        46
three                        43
four                         37
closed                       36
chair                        36
backward                     35
window                       33
forward                      29

TOP 20 MODEL PREDICTIONS
----------------------------------------------------------------------
yes                         410
two                         241
no                          228
left                      

In [ ]:
from collections import Counter, defaultdict

print("=" * 70)
print("STEP 55 — VALIDATION PREDICTION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Ground-truth and prediction distributions
# ------------------------------------------------------------

ground_truth_counts = Counter(
    r["expected"].lower()
    for r in val_results
)

prediction_counts = Counter(
    r["prediction"].lower()
    for r in val_results
)

print()
print("TOP 20 GROUND-TRUTH ANSWERS")
print("-" * 70)

for answer, count in ground_truth_counts.most_common(20):
    print(f"{answer:25s} {count:5d}")

print()
print("TOP 20 MODEL PREDICTIONS")
print("-" * 70)

for answer, count in prediction_counts.most_common(20):
    print(f"{answer:25s} {count:5d}")

# ------------------------------------------------------------
# Accuracy by ground-truth answer
# ------------------------------------------------------------

stats = defaultdict(lambda: {
    "total": 0,
    "correct": 0
})

for r in val_results:
    answer = r["expected"].lower()

    stats[answer]["total"] += 1

    if r["correct"]:
        stats[answer]["correct"] += 1

print()
print("ACCURACY BY GROUND-TRUTH ANSWER")
print("-" * 70)

rows = []

for answer, s in stats.items():
    accuracy = 100 * s["correct"] / s["total"]

    rows.append((
        s["total"],
        answer,
        s["correct"],
        accuracy
    ))

for total, answer, correct_count, accuracy in sorted(
    rows,
    reverse=True
)[:30]:

    print(
        f"{answer:25s} "
        f"n={total:4d} "
        f"correct={correct_count:4d} "
        f"accuracy={accuracy:6.2f}%"
    )

# ------------------------------------------------------------
# Prediction concentration
# ------------------------------------------------------------

print()
print("PREDICTION CONCENTRATION")
print("-" * 70)

total_predictions = len(val_results)

for k in [5, 10, 20]:
    top_k_count = sum(
        count
        for _, count in prediction_counts.most_common(k)
    )

    percentage = 100 * top_k_count / total_predictions

    print(
        f"Top {k:2d} predicted answers cover "
        f"{top_k_count:4d}/{total_predictions} "
        f"({percentage:.2f}%) of predictions"
    )

print()
print("=" * 70)
print("✓ PREDICTION ANALYSIS COMPLETE")
print("=" * 70)

STEP 55 — VALIDATION PREDICTION ANALYSIS

TOP 20 GROUND-TRUTH ANSWERS
----------------------------------------------------------------------
no                          322
yes                         302
two                         184
right                       175
one                         152
left                        145
odd                         102
even                         83
brown                        77
table                        63
rectangular                  58
black                        55
white                        46
three                        43
four                         37
closed                       36
chair                        36
backward                     35
window                       33
forward                      29

TOP 20 MODEL PREDICTIONS
----------------------------------------------------------------------
yes                         410
two                         241
no                          228
left                      

In [ ]:
from collections import Counter

print("=" * 70)
print("STEP 56 — STRUCTURED CONFUSION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def show_confusions(labels, title):
    print()
    print(title)
    print("-" * 70)

    relevant = [
        r for r in val_results
        if r["expected"].lower() in labels
    ]

    matrix = Counter(
        (r["expected"].lower(), r["prediction"].lower())
        for r in relevant
    )

    print(
        f"{'Expected':15s} "
        f"{'Predicted':15s} "
        f"{'Count':>6s}"
    )

    for (expected, predicted), count in matrix.most_common():
        print(
            f"{expected:15s} "
            f"{predicted:15s} "
            f"{count:6d}"
        )

# ------------------------------------------------------------
# Directional
# ------------------------------------------------------------

show_confusions(
    {
        "left",
        "right",
        "forward",
        "backward",
        "behind"
    },
    "DIRECTIONAL CONFUSIONS"
)

# ------------------------------------------------------------
# Binary
# ------------------------------------------------------------

show_confusions(
    {
        "yes",
        "no",
        "true",
        "false"
    },
    "YES / NO CONFUSIONS"
)

# ------------------------------------------------------------
# Counting
# ------------------------------------------------------------

show_confusions(
    {
        "zero",
        "one",
        "two",
        "three",
        "four",
        "five"
    },
    "COUNTING CONFUSIONS"
)

# ------------------------------------------------------------
# Parity
# ------------------------------------------------------------

show_confusions(
    {
        "odd",
        "even"
    },
    "ODD / EVEN CONFUSIONS"
)

print()
print("=" * 70)
print("✓ CONFUSION ANALYSIS COMPLETE")
print("=" * 70)

STEP 56 — STRUCTURED CONFUSION ANALYSIS

DIRECTIONAL CONFUSIONS
----------------------------------------------------------------------
Expected        Predicted        Count
right           right               89
right           left                82
left            left                76
left            right               63
backward        left                20
forward         left                12
backward        right               12
behind          behind              11
forward         right               10
forward         forward              6
behind          right                6
left            behind               4
right           behind               1
right           forward              1
behind          in front of me       1
forward         yes                  1
left            front                1
backward        trash can            1
right           yes                  1
behind          bed in eight o'clock      1
backward        forward              1
ri

In [ ]:
from collections import Counter

print("=" * 70)
print("STEP 57 — BASELINE COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# Ground-truth distribution
# ------------------------------------------------------------

gt_counts = Counter(
    r["expected"].lower()
    for r in val_results
)

N = len(val_results)

# ------------------------------------------------------------
# 1. Global majority baseline
# ------------------------------------------------------------

majority_answer, majority_count = gt_counts.most_common(1)[0]

majority_accuracy = 100 * majority_count / N

print()
print("GLOBAL MAJORITY BASELINE")
print("-" * 70)
print("Most common answer :", majority_answer)
print("Occurrences        :", majority_count)
print("Accuracy           :", f"{majority_accuracy:.2f}%")

# ------------------------------------------------------------
# 2. Model accuracy
# ------------------------------------------------------------

model_correct = sum(
    r["correct"]
    for r in val_results
)

model_accuracy = 100 * model_correct / N

# ------------------------------------------------------------
# 3. Number of unique answers
# ------------------------------------------------------------

unique_answers = len(gt_counts)

print()
print("DATASET STATISTICS")
print("-" * 70)
print("Validation examples:", N)
print("Unique answers     :", unique_answers)

# ------------------------------------------------------------
# 4. Random-uniform baseline
# ------------------------------------------------------------

random_uniform_accuracy = 100 / unique_answers

print()
print("RANDOM UNIFORM BASELINE")
print("-" * 70)
print(
    "Expected accuracy:",
    f"{random_uniform_accuracy:.4f}%"
)

# ------------------------------------------------------------
# 5. Compare
# ------------------------------------------------------------

print()
print("MODEL COMPARISON")
print("-" * 70)

print(
    f"Majority baseline : {majority_accuracy:.2f}%"
)

print(
    f"Fine-tuned model  : {model_accuracy:.2f}%"
)

print(
    f"Improvement over majority:",
    f"{model_accuracy - majority_accuracy:+.2f} percentage points"
)

print(
    f"Improvement over zero-shot:",
    f"{model_accuracy - 19.00:+.2f} percentage points"
)

print()
print("=" * 70)
print("✓ BASELINE COMPARISON COMPLETE")
print("=" * 70)

STEP 57 — BASELINE COMPARISON

GLOBAL MAJORITY BASELINE
----------------------------------------------------------------------
Most common answer : no
Occurrences        : 322
Accuracy           : 9.87%

DATASET STATISTICS
----------------------------------------------------------------------
Validation examples: 3261
Unique answers     : 396

RANDOM UNIFORM BASELINE
----------------------------------------------------------------------
Expected accuracy: 0.2525%

MODEL COMPARISON
----------------------------------------------------------------------
Majority baseline : 9.87%
Fine-tuned model  : 50.02%
Improvement over majority: +40.14 percentage points
Improvement over zero-shot: +31.02 percentage points

✓ BASELINE COMPARISON COMPLETE


In [ ]:
from collections import defaultdict, Counter

print("="*70)
print("STEP 58 — ANSWER-FAMILY PERFORMANCE ANALYSIS")
print("="*70)

# Group answers into broad semantic families
def answer_family(ans):
    a = ans.lower().strip()

    if a in {"yes", "no", "true", "false"}:
        return "YES / NO"

    if a in {
        "left", "right", "forward", "backward",
        "behind", "in front of me", "front", "back"
    }:
        return "DIRECTION"

    if a in {"zero", "one", "two", "three", "four", "five",
             "six", "seven", "eight", "nine", "ten"}:
        return "COUNTING"

    if a in {"odd", "even"}:
        return "ODD / EVEN"

    if a in {
        "red", "blue", "green", "yellow", "orange",
        "purple", "pink", "brown", "black", "white",
        "gray", "grey"
    }:
        return "COLOR"

    return "OBJECT / ATTRIBUTE"


# Calculate performance by family
family_stats = defaultdict(lambda: {
    "total": 0,
    "correct": 0,
    "gt_answers": Counter(),
    "pred_answers": Counter()
})

for r in val_results:
    family = answer_family(r["expected"])

    family_stats[family]["total"] += 1
    family_stats[family]["correct"] += int(r["correct"])
    family_stats[family]["gt_answers"][r["expected"].lower()] += 1
    family_stats[family]["pred_answers"][r["generated"].lower()] += 1


# Print results
print("\nPERFORMANCE BY ANSWER FAMILY")
print("-"*70)

for family, stats in sorted(
    family_stats.items(),
    key=lambda x: -x[1]["total"]
):
    accuracy = 100 * stats["correct"] / stats["total"]

    print(
        f"{family:20s} "
        f"{stats['correct']:4d}/{stats['total']:<4d} "
        f"= {accuracy:6.2f}%"
    )


# Show distribution
print("\nGROUND-TRUTH DISTRIBUTION BY FAMILY")
print("-"*70)

for family, stats in sorted(
    family_stats.items(),
    key=lambda x: -x[1]["total"]
):
    print(f"\n{family}:")

    for answer, count in stats["gt_answers"].most_common(10):
        print(f"  {answer:20s} {count}")


print("\n"+"="*70)
print("✓ STEP 58 COMPLETE")
print("="*70)

STEP 58 — ANSWER-FAMILY PERFORMANCE ANALYSIS


KeyError: 'generated'

In [ ]:
print("="*70)
print("STEP 58A — INSPECT VALIDATION RESULT FORMAT")
print("="*70)

print("\nNumber of results:", len(val_results))

print("\nKeys in first result:")
print(val_results[0].keys())

print("\nFirst result:")
print(val_results[0])

print("\n" + "="*70)
print("✓ STEP 58A COMPLETE")
print("="*70)

STEP 58A — INSPECT VALIDATION RESULT FORMAT

Number of results: 3261

Keys in first result:
dict_keys(['index', 'scene_id', 'question', 'expected', 'prediction', 'correct'])

First result:
{'index': 0, 'scene_id': 'scene0249_00', 'question': 'Which direction should I toss a used napkin?', 'expected': 'right', 'prediction': 'right', 'correct': True}

✓ STEP 58A COMPLETE


In [ ]:
from collections import defaultdict, Counter

print("="*70)
print("STEP 58B — ANSWER-FAMILY PERFORMANCE ANALYSIS")
print("="*70)

def answer_family(ans):
    a = ans.lower().strip()

    if a in {"yes", "no", "true", "false"}:
        return "YES / NO"

    if a in {
        "left", "right", "forward", "backward",
        "behind", "in front of me", "front", "back"
    }:
        return "DIRECTION"

    if a in {
        "zero", "one", "two", "three", "four",
        "five", "six", "seven", "eight", "nine", "ten"
    }:
        return "COUNTING"

    if a in {"odd", "even"}:
        return "ODD / EVEN"

    if a in {
        "red", "blue", "green", "yellow", "orange",
        "purple", "pink", "brown", "black", "white",
        "gray", "grey"
    }:
        return "COLOR"

    return "OBJECT / ATTRIBUTE"


family_stats = defaultdict(lambda: {
    "total": 0,
    "correct": 0,
    "gt_answers": Counter(),
    "pred_answers": Counter()
})

for r in val_results:
    family = answer_family(r["expected"])

    family_stats[family]["total"] += 1
    family_stats[family]["correct"] += int(r["correct"])
    family_stats[family]["gt_answers"][r["expected"].lower()] += 1
    family_stats[family]["pred_answers"][r["prediction"].lower()] += 1


print("\nPERFORMANCE BY ANSWER FAMILY")
print("-"*70)

for family, stats in sorted(
    family_stats.items(),
    key=lambda x: -x[1]["total"]
):
    accuracy = 100 * stats["correct"] / stats["total"]

    print(
        f"{family:20s} "
        f"{stats['correct']:4d}/{stats['total']:<4d} "
        f"= {accuracy:6.2f}%"
    )


print("\nGROUND-TRUTH DISTRIBUTION BY FAMILY")
print("-"*70)

for family, stats in sorted(
    family_stats.items(),
    key=lambda x: -x[1]["total"]
):
    print(f"\n{family}:")

    for answer, count in stats["gt_answers"].most_common(10):
        print(f"  {answer:20s} {count}")


print("\nMODEL PREDICTION DISTRIBUTION BY FAMILY")
print("-"*70)

for family, stats in sorted(
    family_stats.items(),
    key=lambda x: -x[1]["total"]
):
    print(f"\n{family}:")

    for answer, count in stats["pred_answers"].most_common(10):
        print(f"  {answer:20s} {count}")


print("\n"+"="*70)
print("✓ STEP 58B COMPLETE")
print("="*70)

STEP 58B — ANSWER-FAMILY PERFORMANCE ANALYSIS

PERFORMANCE BY ANSWER FAMILY
----------------------------------------------------------------------
OBJECT / ATTRIBUTE    536/1303 =  41.14%
YES / NO              445/649  =  68.57%
COUNTING              230/490  =  46.94%
DIRECTION             183/411  =  44.53%
COLOR                 124/223  =  55.61%
ODD / EVEN            113/185  =  61.08%

GROUND-TRUTH DISTRIBUTION BY FAMILY
----------------------------------------------------------------------

OBJECT / ATTRIBUTE:
  table                63
  rectangular          58
  closed               36
  chair                36
  window               33
  backpack             28
  open                 27
  refrigerator         26
  trash can            24
  cabinet              23

YES / NO:
  no                   322
  yes                  302
  true                 23
  false                2

COUNTING:
  two                  184
  one                  152
  three                43
  four     

In [ ]:
from collections import defaultdict

print("="*70)
print("STEP 59 — PER-ANSWER ACCURACY")
print("="*70)

answer_stats = defaultdict(lambda: {
    "total": 0,
    "correct": 0
})

for r in val_results:
    answer = r["expected"].lower().strip()
    answer_stats[answer]["total"] += 1
    answer_stats[answer]["correct"] += int(r["correct"])


results = []

for answer, stats in answer_stats.items():
    total = stats["total"]
    correct = stats["correct"]
    accuracy = 100 * correct / total

    results.append(
        (answer, total, correct, accuracy)
    )


# Sort by number of examples
print("\nMOST FREQUENT ANSWERS")
print("-"*70)

for answer, total, correct, accuracy in sorted(
    results,
    key=lambda x: -x[1]
)[:40]:

    print(
        f"{answer:25s} "
        f"{correct:4d}/{total:<4d} "
        f"= {accuracy:6.2f}%"
    )


print("\nLOWEST ACCURACY — ANSWERS WITH >= 10 EXAMPLES")
print("-"*70)

for answer, total, correct, accuracy in sorted(
    [x for x in results if x[1] >= 10],
    key=lambda x: x[3]
)[:30]:

    print(
        f"{answer:25s} "
        f"{correct:4d}/{total:<4d} "
        f"= {accuracy:6.2f}%"
    )


print("\nHIGHEST ACCURACY — ANSWERS WITH >= 10 EXAMPLES")
print("-"*70)

for answer, total, correct, accuracy in sorted(
    [x for x in results if x[1] >= 10],
    key=lambda x: -x[3]
)[:30]:

    print(
        f"{answer:25s} "
        f"{correct:4d}/{total:<4d} "
        f"= {accuracy:6.2f}%"
    )


print("\n"+"="*70)
print("✓ STEP 59 COMPLETE")
print("="*70)

STEP 59 — PER-ANSWER ACCURACY

MOST FREQUENT ANSWERS
----------------------------------------------------------------------
no                         174/322  =  54.04%
yes                        248/302  =  82.12%
two                        110/184  =  59.78%
right                       89/175  =  50.86%
one                        100/152  =  65.79%
left                        76/145  =  52.41%
odd                         53/102  =  51.96%
even                        60/83   =  72.29%
brown                       57/77   =  74.03%
table                       34/63   =  53.97%
rectangular                 51/58   =  87.93%
black                       20/55   =  36.36%
white                       38/46   =  82.61%
three                        3/43   =   6.98%
four                        15/37   =  40.54%
closed                      35/36   =  97.22%
chair                       20/36   =  55.56%
backward                     0/35   =   0.00%
window                      16/33   =  48.48%
fo

In [ ]:
import torch
from tqdm.auto import tqdm

print("="*70)
print("STEP 60 — SPATIAL INFORMATION ABLATION")
print("="*70)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def clean_prediction(text):
    """
    Extract the model's answer before the first Gemma end-of-turn token.
    """
    text = text.split("<end_of_turn>")[0]
    return text.strip()


def build_prompt_full(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Agent position:
{example['position']}

Agent rotation:
{example['rotation']}

Question:
{example['question']}"""


def build_prompt_no_pose(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Question:
{example['question']}"""


def build_prompt_question_only(example):
    return f"""You are a spatial reasoning assistant.

Question:
{example['question']}"""


def generate_answer(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return clean_prediction(text)


# ------------------------------------------------------------
# Run one condition
# ------------------------------------------------------------

def run_ablation(condition_name, prompt_builder):

    results = []

    print(f"\n{'='*70}")
    print(f"RUNNING: {condition_name}")
    print(f"{'='*70}")

    for i, example in enumerate(
        tqdm(val_examples, desc=condition_name)
    ):

        prediction = generate_answer(
            prompt_builder(example)
        )

        expected = example["answer"].strip()

        correct = (
            prediction.lower().strip()
            == expected.lower().strip()
        )

        results.append({
            "index": i,
            "scene_id": example["scene_id"],
            "question": example["question"],
            "expected": expected,
            "prediction": prediction,
            "correct": correct
        })

    accuracy = (
        100 *
        sum(r["correct"] for r in results)
        / len(results)
    )

    print(f"\n{condition_name} ACCURACY: {accuracy:.2f}%")
    print(
        f"Correct: "
        f"{sum(r['correct'] for r in results)} / "
        f"{len(results)}"
    )

    return results, accuracy


# ------------------------------------------------------------
# Make sure validation examples exist
# ------------------------------------------------------------

print("\nValidation examples:", len(val_examples))

# ------------------------------------------------------------
# A — FULL
# ------------------------------------------------------------

full_results, full_accuracy = run_ablation(
    "A — FULL SPATIAL INPUT",
    build_prompt_full
)

# ------------------------------------------------------------
# B — NO POSE
# ------------------------------------------------------------

no_pose_results, no_pose_accuracy = run_ablation(
    "B — NO POSE",
    build_prompt_no_pose
)

# ------------------------------------------------------------
# C — QUESTION ONLY
# ------------------------------------------------------------

question_only_results, question_only_accuracy = run_ablation(
    "C — QUESTION ONLY",
    build_prompt_question_only
)


# ------------------------------------------------------------
# Final comparison
# ------------------------------------------------------------

print("\n")
print("="*70)
print("STEP 60 — FINAL ABLATION RESULTS")
print("="*70)

print(f"\nA — FULL INPUT       : {full_accuracy:.2f}%")
print(f"B — NO POSE          : {no_pose_accuracy:.2f}%")
print(f"C — QUESTION ONLY    : {question_only_accuracy:.2f}%")

print("\nDROP FROM FULL INPUT")
print("-"*70)

print(
    f"Full → No Pose       : "
    f"{full_accuracy - no_pose_accuracy:+.2f} percentage points"
)

print(
    f"Full → Question Only : "
    f"{full_accuracy - question_only_accuracy:+.2f} percentage points"
)

print("\n")

if full_accuracy > no_pose_accuracy:
    print("✓ Pose information contributes positively.")
else:
    print("⚠ Pose information does not improve accuracy.")

if no_pose_accuracy > question_only_accuracy:
    print("✓ Situation/context contributes positively.")
else:
    print("⚠ Situation/context does not improve accuracy.")

print("\n"+"="*70)
print("✓ STEP 60 COMPLETE")
print("="*70)

STEP 60 — SPATIAL INFORMATION ABLATION


NameError: name 'val_examples' is not defined

In [ ]:
import os
import json

print("="*70)
print("STEP 60A — RESTORE VALIDATION DATA")
print("="*70)

VAL_Q_PATH = "/content/egospatial_data/v1_balanced_questions_val_scannetv2.json"
VAL_A_PATH = "/content/egospatial_data/v1_balanced_sqa_annotations_val_scannetv2.json"

print("\nChecking files...")

print("Questions file:", os.path.exists(VAL_Q_PATH))
print("Annotations file:", os.path.exists(VAL_A_PATH))

# Load JSON
with open(VAL_Q_PATH, "r") as f:
    val_questions_data = json.load(f)

with open(VAL_A_PATH, "r") as f:
    val_annotations_data = json.load(f)

questions = val_questions_data["questions"]
annotations = val_annotations_data["annotations"]

print("\nQuestions:", len(questions))
print("Annotations:", len(annotations))

# Reconstruct validation examples
val_examples = []

for q, a in zip(questions, annotations):

    answer = a["answers"][0]["answer"]

    position = [
        a["position"]["x"],
        a["position"]["y"],
        a["position"]["z"]
    ]

    rotation = [
        a["rotation"]["_x"],
        a["rotation"]["_y"],
        a["rotation"]["_z"],
        a["rotation"]["_w"]
    ]

    val_examples.append({
        "scene_id": q["scene_id"],
        "situation": q["situation"],
        "question": q["question"],
        "answer": answer,
        "position": position,
        "rotation": rotation
    })

print("\nValidation examples reconstructed:", len(val_examples))

# Sanity checks
print("\nFIRST EXAMPLE")
print("-"*70)
print(val_examples[0])

print("\nLAST EXAMPLE")
print("-"*70)
print(val_examples[-1])

assert len(val_examples) == 3261
assert len(questions) == len(annotations)

print("\n✓ 3,261 validation examples restored")
print("="*70)
print("✓ STEP 60A COMPLETE")
print("="*70)

STEP 60A — RESTORE VALIDATION DATA

Checking files...
Questions file: True
Annotations file: True

Questions: 3261
Annotations: 3261

Validation examples reconstructed: 3261

FIRST EXAMPLE
----------------------------------------------------------------------
{'scene_id': 'scene0249_00', 'situation': 'I am wiping a table that has a trash can next to it, while having a large table behind me with lots of chairs.', 'question': 'Which direction should I toss a used napkin?', 'answer': 'right', 'position': [-1.612321909455232, 3.8766019062927524, 0], 'rotation': [0, 0, 0.9436221923009414, -0.33102440725287985]}

LAST EXAMPLE
----------------------------------------------------------------------
{'scene_id': 'scene0693_00', 'situation': 'I am facing the vanity and the door is behind me.', 'question': 'Which direction would I turn if I needed to use the toilet?', 'answer': 'right', 'position': [0.05116582210648152, -0.6164723667008674, 0], 'rotation': [0, 0, 0, 1]}

✓ 3,261 validation example

In [ ]:
import torch
from tqdm.auto import tqdm

print("="*70)
print("STEP 60B — SPATIAL INFORMATION ABLATION")
print("="*70)

# ------------------------------------------------------------
# Prompt builders
# ------------------------------------------------------------

def build_prompt_full(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Agent position:
{example['position']}

Agent rotation:
{example['rotation']}

Question:
{example['question']}"""


def build_prompt_no_pose(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Question:
{example['question']}"""


def build_prompt_question_only(example):
    return f"""You are a spatial reasoning assistant.

Question:
{example['question']}"""


# ------------------------------------------------------------
# Prediction function
# ------------------------------------------------------------

def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    # Only take the first answer before Gemma's turn terminator
    text = text.split("<end_of_turn>")[0].strip()

    return text


# ------------------------------------------------------------
# Run one condition
# ------------------------------------------------------------

def run_condition(condition_name, prompt_builder):

    results = []

    print(f"\n{'='*70}")
    print(condition_name)
    print(f"{'='*70}")

    for i, example in enumerate(
        tqdm(val_examples, desc=condition_name)
    ):

        prediction = generate_answer(
            prompt_builder(example)
        )

        expected = example["answer"].strip()

        correct = (
            prediction.lower()
            == expected.lower()
        )

        results.append({
            "index": i,
            "scene_id": example["scene_id"],
            "question": example["question"],
            "expected": expected,
            "prediction": prediction,
            "correct": correct
        })

    correct_count = sum(r["correct"] for r in results)
    accuracy = 100 * correct_count / len(results)

    print(f"\n{condition_name} RESULT")
    print("-"*70)
    print(f"Correct   : {correct_count}/{len(results)}")
    print(f"Accuracy  : {accuracy:.2f}%")

    return results, accuracy


# ------------------------------------------------------------
# Run A — FULL
# ------------------------------------------------------------

full_ablation_results, full_ablation_accuracy = run_condition(
    "A — FULL INPUT",
    build_prompt_full
)


# ------------------------------------------------------------
# Run B — NO POSE
# ------------------------------------------------------------

no_pose_results, no_pose_accuracy = run_condition(
    "B — NO POSE",
    build_prompt_no_pose
)


# ------------------------------------------------------------
# Run C — QUESTION ONLY
# ------------------------------------------------------------

question_only_results, question_only_accuracy = run_condition(
    "C — QUESTION ONLY",
    build_prompt_question_only
)


# ------------------------------------------------------------
# Final comparison
# ------------------------------------------------------------

print("\n")
print("="*70)
print("STEP 60B — ABLATION RESULTS")
print("="*70)

print(f"\nA — FULL INPUT    : {full_ablation_accuracy:.2f}%")
print(f"B — NO POSE       : {no_pose_accuracy:.2f}%")
print(f"C — QUESTION ONLY : {question_only_accuracy:.2f}%")

print("\nACCURACY DIFFERENCES")
print("-"*70)

print(
    f"Full → No Pose       : "
    f"{full_ablation_accuracy - no_pose_accuracy:+.2f} pp"
)

print(
    f"Full → Question Only : "
    f"{full_ablation_accuracy - question_only_accuracy:+.2f} pp"
)

print(
    f"No Pose → Question Only : "
    f"{no_pose_accuracy - question_only_accuracy:+.2f} pp"
)

print("\nINTERPRETATION FLAGS")
print("-"*70)

if full_ablation_accuracy > no_pose_accuracy:
    print("✓ Removing pose reduces performance.")
else:
    print("⚠ Removing pose does NOT reduce performance.")

if no_pose_accuracy > question_only_accuracy:
    print("✓ Situation text contributes beyond the question alone.")
else:
    print("⚠ Situation text does NOT improve over question alone.")

print("\n"+"="*70)
print("✓ STEP 60B COMPLETE")
print("="*70)

STEP 60B — SPATIAL INFORMATION ABLATION

A — FULL INPUT


A — FULL INPUT:   0%|          | 0/3261 [00:00<?, ?it/s]

AttributeError: 

In [ ]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Transformers may return a BatchEncoding rather than a raw tensor
    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded

    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    text = text.split("<end_of_turn>")[0].strip()

    return text


# ------------------------------------------------------------
# QUICK SANITY TEST — DO NOT RUN 3,261 EXAMPLES YET
# ------------------------------------------------------------

test_answer = generate_answer(
    build_prompt_full(val_examples[0])
)

print("="*70)
print("STEP 60C — GENERATION SANITY TEST")
print("="*70)

print("Expected :", val_examples[0]["answer"])
print("Predicted:", repr(test_answer))

print("="*70)

STEP 60C — GENERATION SANITY TEST
Expected : right
Predicted: 'right'


In [ ]:
import os
import json
import torch
from tqdm.auto import tqdm

print("="*70)
print("STEP 60D — SPATIAL INFORMATION ABLATION")
print("="*70)


# ============================================================
# PROMPT BUILDERS
# ============================================================

def build_prompt_full(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Agent position:
{example['position']}

Agent rotation:
{example['rotation']}

Question:
{example['question']}"""


def build_prompt_no_pose(example):
    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Question:
{example['question']}"""


def build_prompt_question_only(example):
    return f"""You are a spatial reasoning assistant.

Question:
{example['question']}"""


# ============================================================
# GENERATION
# ============================================================

def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded

    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return text.split("<end_of_turn>")[0].strip()


# ============================================================
# CONDITION RUNNER
# ============================================================

def run_condition(condition_name, prompt_builder, save_path):

    # --------------------------------------------------------
    # Resume if partial results already exist
    # --------------------------------------------------------

    results = []

    if os.path.exists(save_path):

        with open(save_path, "r") as f:
            results = json.load(f)

        print(
            f"\nFound existing checkpoint: "
            f"{len(results)}/{len(val_examples)} examples"
        )

    start_index = len(results)

    print(f"\n{'='*70}")
    print(condition_name)
    print(f"{'='*70}")

    if start_index == len(val_examples):

        print("Condition already complete. Skipping generation.")

    else:

        for i in tqdm(
            range(start_index, len(val_examples)),
            desc=condition_name
        ):

            example = val_examples[i]

            prediction = generate_answer(
                prompt_builder(example)
            )

            expected = example["answer"].strip()

            correct = (
                prediction.lower().strip()
                == expected.lower().strip()
            )

            results.append({
                "index": i,
                "scene_id": example["scene_id"],
                "question": example["question"],
                "expected": expected,
                "prediction": prediction,
                "correct": correct
            })

            # Save every 100 examples
            if (i + 1) % 100 == 0:

                with open(save_path, "w") as f:
                    json.dump(results, f)

        # Final save
        with open(save_path, "w") as f:
            json.dump(results, f)

    correct_count = sum(
        r["correct"] for r in results
    )

    accuracy = (
        100 * correct_count / len(results)
    )

    print(f"\n{condition_name} RESULT")
    print("-"*70)
    print(f"Examples  : {len(results)}")
    print(f"Correct   : {correct_count}")
    print(f"Accuracy  : {accuracy:.2f}%")

    return results, accuracy


# ============================================================
# OUTPUT PATHS
# ============================================================

ABLATION_DIR = "/content/egospatial_ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)

FULL_PATH = os.path.join(
    ABLATION_DIR,
    "full.json"
)

NO_POSE_PATH = os.path.join(
    ABLATION_DIR,
    "no_pose.json"
)

QUESTION_ONLY_PATH = os.path.join(
    ABLATION_DIR,
    "question_only.json"
)


# ============================================================
# A — FULL INPUT
# ============================================================

full_ablation_results, full_ablation_accuracy = run_condition(
    "A — FULL INPUT",
    build_prompt_full,
    FULL_PATH
)


# ============================================================
# B — NO POSE
# ============================================================

no_pose_results, no_pose_accuracy = run_condition(
    "B — NO POSE",
    build_prompt_no_pose,
    NO_POSE_PATH
)


# ============================================================
# C — QUESTION ONLY
# ============================================================

question_only_results, question_only_accuracy = run_condition(
    "C — QUESTION ONLY",
    build_prompt_question_only,
    QUESTION_ONLY_PATH
)


# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n")
print("="*70)
print("STEP 60D — FINAL ABLATION RESULTS")
print("="*70)

print(
    f"\nA — FULL INPUT       : "
    f"{full_ablation_accuracy:.2f}%"
)

print(
    f"B — NO POSE          : "
    f"{no_pose_accuracy:.2f}%"
)

print(
    f"C — QUESTION ONLY    : "
    f"{question_only_accuracy:.2f}%"
)

print("\nACCURACY DIFFERENCES")
print("-"*70)

print(
    f"Full → No Pose          : "
    f"{full_ablation_accuracy - no_pose_accuracy:+.2f} pp"
)

print(
    f"Full → Question Only    : "
    f"{full_ablation_accuracy - question_only_accuracy:+.2f} pp"
)

print(
    f"No Pose → Question Only : "
    f"{no_pose_accuracy - question_only_accuracy:+.2f} pp"
)

print("\nCONTROL CHECK")
print("-"*70)

print(
    f"Original validation accuracy : 50.02%"
)

print(
    f"Reproduced full-input accuracy: "
    f"{full_ablation_accuracy:.2f}%"
)

print(
    f"Reproduction difference       : "
    f"{full_ablation_accuracy - 50.02:+.2f} pp"
)

print("\n"+"="*70)
print("✓ STEP 60D COMPLETE")
print("="*70)

STEP 60D — SPATIAL INFORMATION ABLATION

A — FULL INPUT


A — FULL INPUT:   0%|          | 0/3261 [00:00<?, ?it/s]


A — FULL INPUT RESULT
----------------------------------------------------------------------
Examples  : 3261
Correct   : 1621
Accuracy  : 49.71%

B — NO POSE


B — NO POSE:   0%|          | 0/3261 [00:00<?, ?it/s]


B — NO POSE RESULT
----------------------------------------------------------------------
Examples  : 3261
Correct   : 1625
Accuracy  : 49.83%

C — QUESTION ONLY


C — QUESTION ONLY:   0%|          | 0/3261 [00:00<?, ?it/s]


C — QUESTION ONLY RESULT
----------------------------------------------------------------------
Examples  : 3261
Correct   : 1563
Accuracy  : 47.93%


STEP 60D — FINAL ABLATION RESULTS

A — FULL INPUT       : 49.71%
B — NO POSE          : 49.83%
C — QUESTION ONLY    : 47.93%

ACCURACY DIFFERENCES
----------------------------------------------------------------------
Full → No Pose          : -0.12 pp
Full → Question Only    : +1.78 pp
No Pose → Question Only : +1.90 pp

CONTROL CHECK
----------------------------------------------------------------------
Original validation accuracy : 50.02%
Reproduced full-input accuracy: 49.71%
Reproduction difference       : -0.31 pp

✓ STEP 60D COMPLETE


In [ ]:
import json
import os
import random
import torch
from collections import Counter
from tqdm.auto import tqdm

print("="*70)
print("STEP 61 — CONTROLLED SPATIAL REASONING BENCHMARK")
print("="*70)

# ============================================================
# 1. CONTROLLED SPATIAL WORLD
# ============================================================

directions = ["north", "east", "south", "west"]

# World direction -> egocentric answer
# If agent faces a direction, determine where an object
# located in a world direction appears relative to the agent.

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },
    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },
    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },
    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ============================================================
# 2. CREATE CONTROLLED QUESTIONS
# ============================================================

objects = [
    ("red cube", "cube"),
    ("blue sphere", "sphere"),
    ("green chair", "chair"),
    ("yellow table", "table"),
    ("black backpack", "backpack"),
    ("white lamp", "lamp"),
]

benchmark = []

example_id = 0

# ------------------------------------------------------------
# A. Egocentric direction transformation
# ------------------------------------------------------------

for facing in directions:

    for world_direction in directions:

        for obj_name, obj_type in objects[:3]:

            expected = relative_map[facing][world_direction]

            situation = (
                f"I am facing {facing}. "
                f"A {obj_name} is {world_direction} of me."
            )

            question = (
                f"Where is the {obj_type} relative to me?"
            )

            benchmark.append({
                "id": example_id,
                "category": "egocentric_direction",
                "situation": situation,
                "question": question,
                "expected": expected,
                "facing": facing,
                "world_direction": world_direction
            })

            example_id += 1


# ------------------------------------------------------------
# B. Orientation-flip pairs
# Same world location, different agent orientation
# ------------------------------------------------------------

flip_pairs = [
    ("north", "south"),
    ("east", "west")
]

for facing_a, facing_b in flip_pairs:

    for world_direction in directions:

        obj_name, obj_type = objects[3]

        situation_a = (
            f"I am facing {facing_a}. "
            f"A {obj_name} is {world_direction} of me."
        )

        situation_b = (
            f"I am facing {facing_b}. "
            f"A {obj_name} is {world_direction} of me."
        )

        benchmark.append({
            "id": example_id,
            "category": "orientation_flip",
            "situation": situation_a,
            "question": f"Where is the {obj_type} relative to me?",
            "expected": relative_map[facing_a][world_direction],
            "facing": facing_a,
            "world_direction": world_direction
        })

        example_id += 1

        benchmark.append({
            "id": example_id,
            "category": "orientation_flip",
            "situation": situation_b,
            "question": f"Where is the {obj_type} relative to me?",
            "expected": relative_map[facing_b][world_direction],
            "facing": facing_b,
            "world_direction": world_direction
        })

        example_id += 1


# ------------------------------------------------------------
# C. Object-object spatial relations
# ------------------------------------------------------------

object_relations = [
    ("left", "right"),
    ("right", "left"),
    ("front", "behind"),
    ("behind", "front")
]

for relation, inverse in object_relations:

    situation = (
        f"A red cube is {relation} of a blue sphere."
    )

    question = (
        "Where is the red cube relative to the blue sphere?"
    )

    benchmark.append({
        "id": example_id,
        "category": "object_relation",
        "situation": situation,
        "question": question,
        "expected": relation
    })

    example_id += 1

    situation = (
        f"A red cube is {relation} of a blue sphere."
    )

    question = (
        "Where is the blue sphere relative to the red cube?"
    )

    benchmark.append({
        "id": example_id,
        "category": "object_relation",
        "situation": situation,
        "question": question,
        "expected": inverse
    })

    example_id += 1


# ------------------------------------------------------------
# D. Simple counting
# ------------------------------------------------------------

count_cases = [
    (1, "one"),
    (2, "two"),
    (3, "three"),
    (4, "four"),
    (5, "five"),
    (6, "six"),
]

for number, expected in count_cases:

    names = ", ".join(
        ["chairs"] * number
    )

    situation = (
        f"There are {number} chairs in the room."
    )

    question = "How many chairs are there?"

    benchmark.append({
        "id": example_id,
        "category": "counting",
        "situation": situation,
        "question": question,
        "expected": expected
    })

    example_id += 1


# ------------------------------------------------------------
# E. Spatial distractors
# ------------------------------------------------------------

distractor_cases = [
    {
        "situation": (
            "I am facing north. "
            "A red cube is east of me. "
            "A blue sphere is west of me. "
            "A green chair is north of me."
        ),
        "question": "Where is the red cube relative to me?",
        "expected": "right"
    },
    {
        "situation": (
            "I am facing east. "
            "A red cube is north of me. "
            "A blue sphere is south of me. "
            "A green chair is east of me."
        ),
        "question": "Where is the red cube relative to me?",
        "expected": "left"
    },
    {
        "situation": (
            "I am facing south. "
            "A red cube is north of me. "
            "A blue sphere is east of me. "
            "A green chair is west of me."
        ),
        "question": "Where is the red cube relative to me?",
        "expected": "behind"
    },
    {
        "situation": (
            "I am facing west. "
            "A red cube is south of me. "
            "A blue sphere is north of me. "
            "A green chair is west of me."
        ),
        "question": "Where is the red cube relative to me?",
        "expected": "left"
    }
]

for case in distractor_cases:

    benchmark.append({
        "id": example_id,
        "category": "distractors",
        "situation": case["situation"],
        "question": case["question"],
        "expected": case["expected"]
    })

    example_id += 1


# ============================================================
# 3. SHUFFLE
# ============================================================

random.seed(42)
random.shuffle(benchmark)

print("\nBenchmark size:", len(benchmark))

category_counts = Counter(
    x["category"] for x in benchmark
)

print("\nCATEGORY DISTRIBUTION")
print("-"*70)

for category, count in category_counts.items():
    print(f"{category:25s}: {count}")


# ============================================================
# 4. PROMPT
# ============================================================

def build_controlled_prompt(example):

    return f"""You are a spatial reasoning assistant.

Situation:
{example['situation']}

Question:
{example['question']}"""


# ============================================================
# 5. GENERATION
# ============================================================

def generate_controlled_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded

    input_ids = input_ids.to(model.device)

    with torch.no_grad():

        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return text.split("<end_of_turn>")[0].strip()


# ============================================================
# 6. EVALUATE
# ============================================================

results = []

for example in tqdm(
    benchmark,
    desc="Controlled benchmark"
):

    prediction = generate_controlled_answer(
        build_controlled_prompt(example)
    )

    expected = example["expected"]

    correct = (
        prediction.lower().strip()
        == expected.lower().strip()
    )

    results.append({
        **example,
        "prediction": prediction,
        "correct": correct
    })


# ============================================================
# 7. OVERALL RESULT
# ============================================================

correct_count = sum(
    r["correct"] for r in results
)

accuracy = (
    100 * correct_count / len(results)
)

print("\n")
print("="*70)
print("STEP 61 — OVERALL RESULT")
print("="*70)

print(f"\nCorrect : {correct_count}/{len(results)}")
print(f"Accuracy: {accuracy:.2f}%")


# ============================================================
# 8. CATEGORY PERFORMANCE
# ============================================================

print("\nCATEGORY PERFORMANCE")
print("-"*70)

category_results = {}

for category in sorted(category_counts):

    subset = [
        r for r in results
        if r["category"] == category
    ]

    correct = sum(
        r["correct"] for r in subset
    )

    acc = 100 * correct / len(subset)

    category_results[category] = acc

    print(
        f"{category:25s} "
        f"{correct:3d}/{len(subset):3d} "
        f"= {acc:6.2f}%"
    )


# ============================================================
# 9. DIRECTIONAL CONFUSION
# ============================================================

direction_results = [
    r for r in results
    if r["category"] in {
        "egocentric_direction",
        "orientation_flip",
        "distractors"
    }
]

print("\nDIRECTIONAL RESULT")
print("-"*70)

direction_correct = sum(
    r["correct"]
    for r in direction_results
)

print(
    f"Correct: "
    f"{direction_correct}/{len(direction_results)}"
)

print(
    f"Accuracy: "
    f"{100 * direction_correct / len(direction_results):.2f}%"
)


# ============================================================
# 10. SHOW ERRORS
# ============================================================

errors = [
    r for r in results
    if not r["correct"]
]

print("\nERROR ANALYSIS")
print("-"*70)

print(
    f"Total errors: "
    f"{len(errors)}/{len(results)}"
)

for r in errors[:40]:

    print("\nCATEGORY :", r["category"])
    print("Situation:", r["situation"])
    print("Question :", r["question"])
    print("Expected :", r["expected"])
    print("Predicted:", r["prediction"])


# ============================================================
# 11. SAVE RESULTS
# ============================================================

output_path = (
    "/content/"
    "egospatial_controlled_benchmark_results.json"
)

with open(output_path, "w") as f:
    json.dump(
        results,
        f,
        indent=2
    )

print("\n")
print("="*70)
print("✓ STEP 61 COMPLETE")
print("="*70)
print("Results saved to:")
print(output_path)
print("="*70)

STEP 61 — CONTROLLED SPATIAL REASONING BENCHMARK

Benchmark size: 82

CATEGORY DISTRIBUTION
----------------------------------------------------------------------
orientation_flip         : 16
egocentric_direction     : 48
counting                 : 6
object_relation          : 8
distractors              : 4


Controlled benchmark:   0%|          | 0/82 [00:00<?, ?it/s]



STEP 61 — OVERALL RESULT

Correct : 8/82
Accuracy: 9.76%

CATEGORY PERFORMANCE
----------------------------------------------------------------------
counting                    5/  6 =  83.33%
distractors                 0/  4 =   0.00%
egocentric_direction        0/ 48 =   0.00%
object_relation             3/  8 =  37.50%
orientation_flip            0/ 16 =   0.00%

DIRECTIONAL RESULT
----------------------------------------------------------------------
Correct: 0/68
Accuracy: 0.00%

ERROR ANALYSIS
----------------------------------------------------------------------
Total errors: 74/82

CATEGORY : orientation_flip
Situation: I am facing west. A yellow table is north of me.
Question : Where is the table relative to me?
Expected : right
Predicted: north

CATEGORY : egocentric_direction
Situation: I am facing east. A red cube is east of me.
Question : Where is the cube relative to me?
Expected : front
Predicted: east

CATEGORY : orientation_flip
Situation: I am facing south. A yell

In [ ]:
print("="*70)
print("STEP 62A — V2 SPATIAL TRANSFORMATION ENGINE")
print("="*70)


# ============================================================
# WORLD → EGOCENTRIC TRANSFORMATION
# ============================================================

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },

    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },

    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },

    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ============================================================
# VERIFY ALL 16 COMBINATIONS
# ============================================================

directions = [
    "north",
    "east",
    "south",
    "west"
]

correct_expected = {
    ("north", "north"): "front",
    ("north", "east"): "right",
    ("north", "south"): "behind",
    ("north", "west"): "left",

    ("east", "north"): "left",
    ("east", "east"): "front",
    ("east", "south"): "right",
    ("east", "west"): "behind",

    ("south", "north"): "behind",
    ("south", "east"): "left",
    ("south", "south"): "front",
    ("south", "west"): "right",

    ("west", "north"): "right",
    ("west", "east"): "behind",
    ("west", "south"): "left",
    ("west", "west"): "front"
}


passed = 0
failed = 0

for facing in directions:

    for world_direction in directions:

        result = relative_map[facing][world_direction]
        expected = correct_expected[
            (facing, world_direction)
        ]

        if result == expected:
            passed += 1
        else:
            failed += 1

        print(
            f"Facing {facing:5s} | "
            f"Object {world_direction:5s} | "
            f"→ {result:7s} | "
            f"Expected {expected:7s} | "
            f"{'✓' if result == expected else '✗'}"
        )


# ============================================================
# RESULT
# ============================================================

print("\n")
print("="*70)
print("TRANSFORMATION TEST")
print("="*70)

print(f"Passed: {passed}/16")
print(f"Failed: {failed}/16")

assert passed == 16
assert failed == 0

print("\n✓ ALL 16 SPATIAL TRANSFORMATIONS VERIFIED")
print("="*70)
print("✓ STEP 62A COMPLETE")
print("="*70)

STEP 62A — V2 SPATIAL TRANSFORMATION ENGINE
Facing north | Object north | → front   | Expected front   | ✓
Facing north | Object east  | → right   | Expected right   | ✓
Facing north | Object south | → behind  | Expected behind  | ✓
Facing north | Object west  | → left    | Expected left    | ✓
Facing east  | Object north | → left    | Expected left    | ✓
Facing east  | Object east  | → front   | Expected front   | ✓
Facing east  | Object south | → right   | Expected right   | ✓
Facing east  | Object west  | → behind  | Expected behind  | ✓
Facing south | Object north | → behind  | Expected behind  | ✓
Facing south | Object east  | → left    | Expected left    | ✓
Facing south | Object south | → front   | Expected front   | ✓
Facing south | Object west  | → right   | Expected right   | ✓
Facing west  | Object north | → right   | Expected right   | ✓
Facing west  | Object east  | → behind  | Expected behind  | ✓
Facing west  | Object south | → left    | Expected left    | ✓
Facing west

In [ ]:
import json
import random

print("="*70)
print("STEP 62B — STRUCTURED SPATIAL STATE GENERATION")
print("="*70)


# ============================================================
# VERIFIED TRANSFORMATION ENGINE
# ============================================================

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },

    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },

    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },

    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ============================================================
# OBJECT VOCABULARY
# ============================================================

objects = [
    ("red cube", "cube"),
    ("blue sphere", "sphere"),
    ("green chair", "chair"),
    ("yellow table", "table"),
    ("black backpack", "backpack"),
    ("white lamp", "lamp"),
    ("brown desk", "desk"),
    ("gray cabinet", "cabinet")
]

directions = [
    "north",
    "east",
    "south",
    "west"
]


# ============================================================
# GENERATE STRUCTURED STATES
# ============================================================

random.seed(42)

states = []

for state_id in range(100):

    facing = random.choice(directions)

    selected_objects = random.sample(
        objects,
        3
    )

    state_objects = []

    for object_name, object_type in selected_objects:

        world_direction = random.choice(
            directions
        )

        relative_direction = relative_map[
            facing
        ][world_direction]

        state_objects.append({
            "name": object_name,
            "type": object_type,
            "world_direction": world_direction,
            "relative_direction": relative_direction
        })

    states.append({
        "state_id": state_id,
        "agent": {
            "heading": facing
        },
        "objects": state_objects
    })


# ============================================================
# CONVERT TO GEMMA-FRIENDLY TEXT
# ============================================================

def state_to_text(state):

    lines = []

    lines.append("SPATIAL STATE")
    lines.append("")
    lines.append(
        f"Agent heading: "
        f"{state['agent']['heading']}"
    )

    lines.append("")
    lines.append("Objects:")

    for obj in state["objects"]:

        lines.append(
            f"- {obj['name']}: "
            f"world_direction={obj['world_direction']}; "
            f"relative_direction={obj['relative_direction']}"
        )

    return "\n".join(lines)


# ============================================================
# DISPLAY EXAMPLES
# ============================================================

print("\nGENERATED STATES")
print("-"*70)

for state in states[:5]:

    print(
        f"\nSTATE {state['state_id']}"
    )

    print(
        state_to_text(state)
    )


# ============================================================
# BASIC VALIDATION
# ============================================================

assert len(states) == 100

for state in states:

    heading = state["agent"]["heading"]

    for obj in state["objects"]:

        expected = relative_map[
            heading
        ][obj["world_direction"]]

        assert (
            obj["relative_direction"]
            == expected
        )


# ============================================================
# SAVE
# ============================================================

output_path = (
    "/content/"
    "egospatial_v2_structured_states.json"
)

with open(output_path, "w") as f:

    json.dump(
        states,
        f,
        indent=2
    )


print("\n")
print("="*70)
print("VALIDATION")
print("="*70)

print("States generated :", len(states))
print("Objects per state: 3")
print("Transformation   : 100% verified")

print("\n✓ STRUCTURED SPATIAL STATES VALIDATED")
print("="*70)
print("Saved to:")
print(output_path)
print("="*70)
print("✓ STEP 62B COMPLETE")
print("="*70)

STEP 62B — STRUCTURED SPATIAL STATE GENERATION

GENERATED STATES
----------------------------------------------------------------------

STATE 0
SPATIAL STATE

Agent heading: north

Objects:
- red cube: world_direction=east; relative_direction=right
- white lamp: world_direction=east; relative_direction=right
- green chair: world_direction=east; relative_direction=right

STATE 1
SPATIAL STATE

Agent heading: north

Objects:
- blue sphere: world_direction=north; relative_direction=front
- black backpack: world_direction=north; relative_direction=front
- yellow table: world_direction=north; relative_direction=front

STATE 2
SPATIAL STATE

Agent heading: east

Objects:
- yellow table: world_direction=north; relative_direction=left
- black backpack: world_direction=east; relative_direction=front
- brown desk: world_direction=west; relative_direction=behind

STATE 3
SPATIAL STATE

Agent heading: east

Objects:
- gray cabinet: world_direction=north; relative_direction=left
- black backpack: 

In [ ]:
import json
import torch
from tqdm.auto import tqdm
from collections import Counter, defaultdict

print("="*70)
print("STEP 62C — V1 + STRUCTURED SPATIAL STATE")
print("="*70)


# ============================================================
# LOAD STRUCTURED STATES
# ============================================================

STATE_PATH = "/content/egospatial_v2_structured_states.json"

with open(STATE_PATH, "r") as f:
    states = json.load(f)

print("\nStructured states loaded:", len(states))


# ============================================================
# CREATE QUESTIONS
# ============================================================

test_cases = []

for state in states:

    for obj in state["objects"]:

        test_cases.append({
            "state_id": state["state_id"],
            "state": state,
            "object_name": obj["name"],
            "object_type": obj["type"],
            "expected": obj["relative_direction"]
        })


print("Total test cases:", len(test_cases))


# ============================================================
# BUILD PROMPT
# ============================================================

def build_structured_prompt(case):

    state = case["state"]

    lines = []

    lines.append(
        "You are a spatial reasoning assistant."
    )

    lines.append("")
    lines.append("SPATIAL STATE")
    lines.append("")
    lines.append(
        f"Agent heading: "
        f"{state['agent']['heading']}"
    )

    lines.append("")
    lines.append("Objects:")

    for obj in state["objects"]:

        lines.append(
            f"- {obj['name']}: "
            f"world_direction={obj['world_direction']}; "
            f"relative_direction={obj['relative_direction']}"
        )

    lines.append("")
    lines.append(
        f"Question: "
        f"Where is the {case['object_type']} "
        f"relative to me?"
    )

    return "\n".join(lines)


# ============================================================
# GENERATION
# ============================================================

def generate_structured_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded

    input_ids = input_ids.to(model.device)

    with torch.no_grad():

        output = model.generate(
            input_ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return text.split("<end_of_turn>")[0].strip()


# ============================================================
# EVALUATE
# ============================================================

results_62c = []

for case in tqdm(
    test_cases,
    desc="V1 structured-state evaluation"
):

    prediction = generate_structured_answer(
        build_structured_prompt(case)
    )

    expected = case["expected"]

    correct = (
        prediction.lower().strip()
        == expected.lower().strip()
    )

    results_62c.append({
        "state_id": case["state_id"],
        "object": case["object_name"],
        "expected": expected,
        "prediction": prediction,
        "correct": correct
    })


# ============================================================
# OVERALL RESULT
# ============================================================

correct_count = sum(
    r["correct"]
    for r in results_62c
)

accuracy = (
    100 *
    correct_count /
    len(results_62c)
)


print("\n")
print("="*70)
print("STEP 62C — OVERALL RESULT")
print("="*70)

print(
    f"\nCorrect : "
    f"{correct_count}/{len(results_62c)}"
)

print(
    f"Accuracy: "
    f"{accuracy:.2f}%"
)


# ============================================================
# PERFORMANCE BY RELATION
# ============================================================

print("\nPERFORMANCE BY RELATION")
print("-"*70)

relation_stats = defaultdict(
    lambda: {"total": 0, "correct": 0}
)

for r in results_62c:

    relation = r["expected"]

    relation_stats[relation]["total"] += 1
    relation_stats[relation]["correct"] += int(
        r["correct"]
    )


for relation, stats in sorted(
    relation_stats.items()
):

    acc = (
        100 *
        stats["correct"] /
        stats["total"]
    )

    print(
        f"{relation:10s} "
        f"{stats['correct']:3d}/"
        f"{stats['total']:3d} "
        f"= {acc:6.2f}%"
    )


# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print("\nPREDICTION DISTRIBUTION")
print("-"*70)

prediction_counts = Counter(
    r["prediction"].lower()
    for r in results_62c
)

for prediction, count in prediction_counts.most_common():

    print(
        f"{prediction:20s}: {count}"
    )


# ============================================================
# SHOW FIRST 20 RESULTS
# ============================================================

print("\nFIRST 20 RESULTS")
print("-"*70)

for r in results_62c[:20]:

    print(
        f"Expected: {r['expected']:8s} | "
        f"Predicted: {r['prediction']}"
        f"{' ✓' if r['correct'] else ' ✗'}"
    )


# ============================================================
# SAVE
# ============================================================

OUTPUT_PATH = (
    "/content/"
    "egospatial_v1_structured_eval.json"
)

with open(OUTPUT_PATH, "w") as f:

    json.dump(
        results_62c,
        f,
        indent=2
    )


print("\n")
print("="*70)
print("✓ STEP 62C COMPLETE")
print("="*70)

print("Saved to:")
print(OUTPUT_PATH)

print("="*70)

STEP 62C — V1 + STRUCTURED SPATIAL STATE

Structured states loaded: 100
Total test cases: 300


V1 structured-state evaluation:   0%|          | 0/300 [00:00<?, ?it/s]



STEP 62C — OVERALL RESULT

Correct : 292/300
Accuracy: 97.33%

PERFORMANCE BY RELATION
----------------------------------------------------------------------
behind      81/ 84 =  96.43%
front       76/ 81 =  93.83%
left        67/ 67 = 100.00%
right       68/ 68 = 100.00%

PREDICTION DISTRIBUTION
----------------------------------------------------------------------
behind              : 81
front               : 76
right               : 72
left                : 68
in front            : 3

FIRST 20 RESULTS
----------------------------------------------------------------------
Expected: right    | Predicted: right ✓
Expected: right    | Predicted: right ✓
Expected: right    | Predicted: right ✓
Expected: front    | Predicted: front ✓
Expected: front    | Predicted: front ✓
Expected: front    | Predicted: front ✓
Expected: left     | Predicted: left ✓
Expected: front    | Predicted: front ✓
Expected: behind   | Predicted: behind ✓
Expected: left     | Predicted: left ✓
Expected: front 

In [ ]:
import json
import torch
from tqdm.auto import tqdm
from collections import Counter, defaultdict

print("="*70)
print("STEP 63A — STRUCTURED SPATIAL REASONING")
print("WITHOUT EXPLICIT RELATIVE ANSWER")
print("="*70)


# ============================================================
# LOAD STATES
# ============================================================

STATE_PATH = "/content/egospatial_v2_structured_states.json"

with open(STATE_PATH, "r") as f:
    states = json.load(f)

print("\nStates loaded:", len(states))


# ============================================================
# BUILD TEST CASES
# ============================================================

test_cases = []

for state in states:

    for obj in state["objects"]:

        test_cases.append({
            "state_id": state["state_id"],
            "object_name": obj["name"],
            "object_type": obj["type"],
            "world_direction": obj["world_direction"],
            "heading": state["agent"]["heading"],
            "expected": obj["relative_direction"]
        })

print("Test cases:", len(test_cases))


# ============================================================
# PROMPT
# ============================================================

def build_prompt(case):

    return f"""You are a spatial reasoning assistant.

SPATIAL STATE

Agent heading: {case['heading']}

Object:
- {case['object_name']}: world_direction={case['world_direction']}

Question:
Where is the {case['object_type']} relative to me?

Answer with only one of:
front
behind
left
right
"""


# ============================================================
# GENERATION
# ============================================================

def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded

    input_ids = input_ids.to(model.device)

    with torch.no_grad():

        output = model.generate(
            input_ids,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = (
        output[0][input_ids.shape[1]:]
    )

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return text.split("<end_of_turn>")[0].strip()


# ============================================================
# EVALUATE
# ============================================================

results_63a = []

for case in tqdm(
    test_cases,
    desc="Structured reasoning"
):

    prediction = generate_answer(
        build_prompt(case)
    )

    correct = (
        prediction.lower().strip()
        == case["expected"].lower().strip()
    )

    results_63a.append({
        "state_id": case["state_id"],
        "object": case["object_name"],
        "heading": case["heading"],
        "world_direction": case["world_direction"],
        "expected": case["expected"],
        "prediction": prediction,
        "correct": correct
    })


# ============================================================
# OVERALL
# ============================================================

correct_count = sum(
    r["correct"]
    for r in results_63a
)

accuracy = (
    100 *
    correct_count /
    len(results_63a)
)


print("\n")
print("="*70)
print("OVERALL RESULT")
print("="*70)

print(
    f"\nCorrect : "
    f"{correct_count}/{len(results_63a)}"
)

print(
    f"Accuracy: "
    f"{accuracy:.2f}%"
)


# ============================================================
# RELATION PERFORMANCE
# ============================================================

print("\nRELATION PERFORMANCE")
print("-"*70)

stats = defaultdict(
    lambda: {
        "total": 0,
        "correct": 0
    }
)

for r in results_63a:

    relation = r["expected"]

    stats[relation]["total"] += 1
    stats[relation]["correct"] += int(
        r["correct"]
    )


for relation in [
    "front",
    "behind",
    "left",
    "right"
]:

    total = stats[relation]["total"]
    correct = stats[relation]["correct"]

    print(
        f"{relation:10s} "
        f"{correct:3d}/{total:3d} "
        f"= {100*correct/total:6.2f}%"
    )


# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print("\nPREDICTION DISTRIBUTION")
print("-"*70)

pred_counts = Counter(
    r["prediction"].lower()
    for r in results_63a
)

for answer, count in pred_counts.most_common():

    print(
        f"{answer:20s}: {count}"
    )


# ============================================================
# ERRORS
# ============================================================

errors = [
    r for r in results_63a
    if not r["correct"]
]

print("\nERRORS")
print("-"*70)

print(
    f"Errors: {len(errors)}/{len(results_63a)}"
)

for r in errors[:30]:

    print(
        f"heading={r['heading']:5s} | "
        f"world={r['world_direction']:5s} | "
        f"expected={r['expected']:7s} | "
        f"predicted={r['prediction']}"
    )


# ============================================================
# SAVE
# ============================================================

OUTPUT_PATH = (
    "/content/"
    "egospatial_v1_structured_reasoning_eval.json"
)

with open(OUTPUT_PATH, "w") as f:

    json.dump(
        results_63a,
        f,
        indent=2
    )


print("\n")
print("="*70)
print("✓ STEP 63A COMPLETE")
print("="*70)

print("Saved to:")
print(OUTPUT_PATH)

print("="*70)

STEP 63A — STRUCTURED SPATIAL REASONING
WITHOUT EXPLICIT RELATIVE ANSWER

States loaded: 100
Test cases: 300


Structured reasoning:   0%|          | 0/300 [00:00<?, ?it/s]



OVERALL RESULT

Correct : 103/300
Accuracy: 34.33%

RELATION PERFORMANCE
----------------------------------------------------------------------
front       49/ 81 =  60.49%
behind       0/ 84 =   0.00%
left        54/ 67 =  80.60%
right        0/ 68 =   0.00%

PREDICTION DISTRIBUTION
----------------------------------------------------------------------
left                : 195
front               : 105

ERRORS
----------------------------------------------------------------------
Errors: 197/300
heading=north | world=east  | expected=right   | predicted=left
heading=north | world=east  | expected=right   | predicted=front
heading=north | world=east  | expected=right   | predicted=left
heading=east  | world=north | expected=left    | predicted=front
heading=east  | world=west  | expected=behind  | predicted=front
heading=east  | world=north | expected=left    | predicted=front
heading=east  | world=west  | expected=behind  | predicted=left
heading=south | world=south | expected=fron

In [ ]:
import json
import random
import os
from collections import Counter

print("="*70)
print("STEP 64A — V2 SPATIAL REASONING CURRICULUM")
print("="*70)


# ============================================================
# SPATIAL TRANSFORMATION
# ============================================================

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },
    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },
    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },
    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ============================================================
# VOCABULARY
# ============================================================

objects = [
    ("red cube", "cube"),
    ("blue sphere", "sphere"),
    ("green chair", "chair"),
    ("yellow table", "table"),
    ("black backpack", "backpack"),
    ("white lamp", "lamp"),
    ("brown desk", "desk"),
    ("gray cabinet", "cabinet"),
    ("orange box", "box"),
    ("purple bottle", "bottle"),
    ("wooden stool", "stool"),
    ("small monitor", "monitor"),
]

heading_phrases = {
    "north": [
        "north",
        "northward",
        "toward the north"
    ],
    "east": [
        "east",
        "eastward",
        "toward the east"
    ],
    "south": [
        "south",
        "southward",
        "toward the south"
    ],
    "west": [
        "west",
        "westward",
        "toward the west"
    ]
}

direction_phrases = {
    "north": [
        "north of me",
        "to my north",
        "directly north of me"
    ],
    "east": [
        "east of me",
        "to my east",
        "directly east of me"
    ],
    "south": [
        "south of me",
        "to my south",
        "directly south of me"
    ],
    "west": [
        "west of me",
        "to my west",
        "directly west of me"
    ]
}

question_templates = [
    "Where is the {obj_type} relative to me?",
    "Which direction is the {obj_type} from me?",
    "Where would I see the {obj_type}?",
    "What direction is the {obj_type} in relative to my position?"
]

distractor_templates = [
    "A {name} is {direction}.",
    "There is a {name} {direction}.",
]


# ============================================================
# EXAMPLE GENERATOR
# ============================================================

def generate_example(example_id, rng):

    heading = rng.choice(
        list(relative_map.keys())
    )

    target_name, target_type = rng.choice(
        objects
    )

    world_direction = rng.choice(
        list(relative_map[heading].keys())
    )

    expected = relative_map[
        heading
    ][world_direction]

    heading_phrase = rng.choice(
        heading_phrases[heading]
    )

    direction_phrase = rng.choice(
        direction_phrases[world_direction]
    )

    # Main target statement
    target_sentence = (
        f"A {target_name} is "
        f"{direction_phrase}."
    )

    # Add 0–2 distractors
    distractors = []

    available_objects = [
        obj for obj in objects
        if obj != (target_name, target_type)
    ]

    rng.shuffle(available_objects)

    num_distractors = rng.randint(0, 2)

    for distractor_name, _ in available_objects[
        :num_distractors
    ]:

        distractor_direction = rng.choice(
            list(direction_phrases.keys())
        )

        distractors.append(
            rng.choice(
                distractor_templates
            ).format(
                name=distractor_name,
                direction=rng.choice(
                    direction_phrases[
                        distractor_direction
                    ]
                )
            )
        )

    sentences = [
        f"I am facing {heading_phrase}.",
        target_sentence
    ]

    sentences.extend(distractors)

    # Randomize sentence order while keeping heading first
    if len(sentences) > 2:
        rest = sentences[1:]
        rng.shuffle(rest)
        sentences = [sentences[0]] + rest

    situation = " ".join(sentences)

    question = rng.choice(
        question_templates
    ).format(
        obj_type=target_type
    )

    return {
        "id": example_id,
        "situation": situation,
        "question": question,
        "answer": expected,
        "heading": heading,
        "world_direction": world_direction,
        "object": target_name,
        "object_type": target_type
    }


# ============================================================
# GENERATE SPLITS
# ============================================================

def generate_split(count, seed, split_name):

    rng = random.Random(seed)

    examples = []

    for i in range(count):

        examples.append(
            generate_example(
                i,
                rng
            )
        )

    # Ensure deterministic ordering
    return examples


train_v2 = generate_split(
    2000,
    1001,
    "train"
)

val_v2 = generate_split(
    400,
    2002,
    "validation"
)

test_v2 = generate_split(
    400,
    3003,
    "test"
)


# ============================================================
# CHECK ANSWER BALANCE
# ============================================================

print("\nANSWER DISTRIBUTION")
print("-"*70)

for split_name, split in [
    ("TRAIN", train_v2),
    ("VALIDATION", val_v2),
    ("TEST", test_v2)
]:

    counts = Counter(
        x["answer"]
        for x in split
    )

    print(f"\n{split_name}")

    for answer in [
        "front",
        "behind",
        "left",
        "right"
    ]:

        print(
            f"  {answer:8s}: "
            f"{counts[answer]}"
        )


# ============================================================
# CHECK TRANSFORMATION COVERAGE
# ============================================================

print("\nTRANSFORMATION COVERAGE")
print("-"*70)

for split_name, split in [
    ("TRAIN", train_v2),
    ("VALIDATION", val_v2),
    ("TEST", test_v2)
]:

    pairs = Counter(
        (
            x["heading"],
            x["world_direction"]
        )
        for x in split
    )

    print(
        f"\n{split_name}: "
        f"{len(pairs)}/16 heading-direction pairs"
    )


# ============================================================
# CHECK EXACT TEXT LEAKAGE
# ============================================================

train_text = {
    (
        x["situation"],
        x["question"]
    )
    for x in train_v2
}

val_text = {
    (
        x["situation"],
        x["question"]
    )
    for x in val_v2
}

test_text = {
    (
        x["situation"],
        x["question"]
    )
    for x in test_v2
}

print("\nEXACT TEXT OVERLAP")
print("-"*70)

print(
    "Train ∩ Validation:",
    len(train_text & val_text)
)

print(
    "Train ∩ Test:",
    len(train_text & test_text)
)

print(
    "Validation ∩ Test:",
    len(val_text & test_text)
)


# ============================================================
# SHOW EXAMPLES
# ============================================================

print("\nSAMPLE TRAINING EXAMPLES")
print("-"*70)

for x in train_v2[:10]:

    print("\nSituation:", x["situation"])
    print("Question :", x["question"])
    print("Answer   :", x["answer"])


# ============================================================
# SAVE
# ============================================================

BASE_DIR = "/content/egospatial_v2_data"

os.makedirs(
    BASE_DIR,
    exist_ok=True
)

paths = {
    "train": os.path.join(
        BASE_DIR,
        "train.json"
    ),
    "validation": os.path.join(
        BASE_DIR,
        "validation.json"
    ),
    "test": os.path.join(
        BASE_DIR,
        "test.json"
    )
}

with open(paths["train"], "w") as f:
    json.dump(
        train_v2,
        f,
        indent=2
    )

with open(paths["validation"], "w") as f:
    json.dump(
        val_v2,
        f,
        indent=2
    )

with open(paths["test"], "w") as f:
    json.dump(
        test_v2,
        f,
        indent=2
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(train_v2) == 2000
assert len(val_v2) == 400
assert len(test_v2) == 400

assert len(train_text & val_text) == 0
assert len(train_text & test_text) == 0
assert len(val_text & test_text) == 0

print("\n")
print("="*70)
print("V2 DATASET VALIDATION")
print("="*70)

print("Training examples   :", len(train_v2))
print("Validation examples :", len(val_v2))
print("Test examples       :", len(test_v2))

print("\nExact text leakage  : NONE")

print("\nSaved files:")

for name, path in paths.items():
    print(
        f"{name:12s}: {path}"
    )

print("\n✓ V2 CURRICULUM GENERATED AND VALIDATED")
print("="*70)
print("✓ STEP 64A COMPLETE")
print("="*70)

STEP 64A — V2 SPATIAL REASONING CURRICULUM

ANSWER DISTRIBUTION
----------------------------------------------------------------------

TRAIN
  front   : 504
  behind  : 483
  left    : 518
  right   : 495

VALIDATION
  front   : 90
  behind  : 112
  left    : 90
  right   : 108

TEST
  front   : 95
  behind  : 106
  left    : 90
  right   : 109

TRANSFORMATION COVERAGE
----------------------------------------------------------------------

TRAIN: 16/16 heading-direction pairs

VALIDATION: 16/16 heading-direction pairs

TEST: 16/16 heading-direction pairs

EXACT TEXT OVERLAP
----------------------------------------------------------------------
Train ∩ Validation: 12
Train ∩ Test: 12
Validation ∩ Test: 0

SAMPLE TRAINING EXAMPLES
----------------------------------------------------------------------

Situation: I am facing toward the north. A small monitor is east of me. There is a purple bottle directly north of me. A yellow table is to my north.
Question : Where would I see the table

AssertionError: 

In [ ]:
import json
import random
import os
from collections import Counter

# ============================================================
# V2 DATASET GENERATOR — CLEAN SPLIT VERSION
# ============================================================

relative_map = {
    "north": {"north":"front","east":"right","south":"behind","west":"left"},
    "east": {"north":"left","east":"front","south":"right","west":"behind"},
    "south": {"north":"behind","east":"left","south":"left","west":"right"},
    "west": {"north":"right","east":"behind","south":"left","west":"front"}
}

# Correct the south mapping explicitly
relative_map["south"] = {
    "north": "behind",
    "east": "left",
    "south": "front",
    "west": "right"
}

objects = [
    ("red cube", "cube"),
    ("blue sphere", "sphere"),
    ("green chair", "chair"),
    ("yellow table", "table"),
    ("black backpack", "backpack"),
    ("white lamp", "lamp"),
    ("brown desk", "desk"),
    ("gray cabinet", "cabinet"),
    ("orange box", "box"),
    ("purple bottle", "bottle"),
    ("wooden stool", "stool"),
    ("small monitor", "monitor"),
]

heading_phrases = {
    "north": ["north", "northward", "toward the north", "facing north"],
    "east": ["east", "eastward", "toward the east", "facing east"],
    "south": ["south", "southward", "toward the south", "facing south"],
    "west": ["west", "westward", "toward the west", "facing west"]
}

direction_phrases = {
    "north": [
        "north of me",
        "to my north",
        "directly north of me"
    ],
    "east": [
        "east of me",
        "to my east",
        "directly east of me"
    ],
    "south": [
        "south of me",
        "to my south",
        "directly south of me"
    ],
    "west": [
        "west of me",
        "to my west",
        "directly west of me"
    ]
}

question_templates = [
    "Where is the {obj_type} relative to me?",
    "Which direction is the {obj_type} from me?",
    "Where would I see the {obj_type}?",
    "What direction is the {obj_type} in relative to my position?"
]

distractor_templates = [
    "A {name} is {direction}.",
    "There is a {name} {direction}.",
]

# ------------------------------------------------------------
# Generate ONE example
# ------------------------------------------------------------

def generate_example(example_id, rng):

    heading = rng.choice(list(relative_map.keys()))

    target_name, target_type = rng.choice(objects)

    world_direction = rng.choice(
        list(relative_map[heading].keys())
    )

    expected = relative_map[heading][world_direction]

    heading_phrase = rng.choice(
        heading_phrases[heading]
    )

    direction_phrase = rng.choice(
        direction_phrases[world_direction]
    )

    target_sentence = (
        f"A {target_name} is {direction_phrase}."
    )

    # ----------------------------
    # Distractors
    # ----------------------------

    available_objects = [
        obj for obj in objects
        if obj != (target_name, target_type)
    ]

    rng.shuffle(available_objects)

    num_distractors = rng.randint(0, 2)

    distractors = []

    for distractor_name, _ in available_objects[:num_distractors]:

        distractor_direction = rng.choice(
            list(direction_phrases.keys())
        )

        distractors.append(
            rng.choice(distractor_templates).format(
                name=distractor_name,
                direction=rng.choice(
                    direction_phrases[distractor_direction]
                )
            )
        )

    # ----------------------------
    # Situation
    # ----------------------------

    sentences = [
        f"I am facing {heading_phrase}.",
        target_sentence
    ]

    sentences.extend(distractors)

    if len(sentences) > 2:
        rest = sentences[1:]
        rng.shuffle(rest)
        sentences = [sentences[0]] + rest

    situation = " ".join(sentences)

    # ----------------------------
    # Question
    # ----------------------------

    question = rng.choice(
        question_templates
    ).format(obj_type=target_type)

    return {
        "id": example_id,
        "situation": situation,
        "question": question,
        "answer": expected,
        "heading": heading,
        "world_direction": world_direction,
        "object": target_name,
        "object_type": target_type
    }


# ============================================================
# CLEAN SPLIT GENERATOR
# ============================================================

def generate_clean_split(
    count,
    seed,
    used_texts,
    split_name
):

    rng = random.Random(seed)

    examples = []

    attempts = 0
    max_attempts = count * 100

    while len(examples) < count:

        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                f"Could not generate enough unique "
                f"{split_name} examples."
            )

        example = generate_example(
            len(examples),
            rng
        )

        key = (
            example["situation"],
            example["question"]
        )

        # Reject exact duplicate
        if key in used_texts:
            continue

        used_texts.add(key)

        examples.append(example)

    return examples


# ============================================================
# GENERATE ALL SPLITS
# ============================================================

used_texts = set()

train_v2 = generate_clean_split(
    2000,
    1001,
    used_texts,
    "train"
)

val_v2 = generate_clean_split(
    400,
    2002,
    used_texts,
    "validation"
)

test_v2 = generate_clean_split(
    400,
    3003,
    used_texts,
    "test"
)


# ============================================================
# VERIFY
# ============================================================

train_text = {
    (x["situation"], x["question"])
    for x in train_v2
}

val_text = {
    (x["situation"], x["question"])
    for x in val_v2
}

test_text = {
    (x["situation"], x["question"])
    for x in test_v2
}

print("=" * 70)
print("V2 CLEAN DATASET VERIFICATION")
print("=" * 70)

print()
print("SIZES")
print("-" * 70)
print("Train      :", len(train_v2))
print("Validation :", len(val_v2))
print("Test       :", len(test_v2))

print()
print("EXACT TEXT OVERLAP")
print("-" * 70)

print(
    "Train ∩ Validation:",
    len(train_text & val_text)
)

print(
    "Train ∩ Test      :",
    len(train_text & test_text)
)

print(
    "Validation ∩ Test :",
    len(val_text & test_text)
)


# ============================================================
# ANSWER DISTRIBUTION
# ============================================================

print()
print("ANSWER DISTRIBUTION")
print("-" * 70)

for name, dataset in [
    ("TRAIN", train_v2),
    ("VALIDATION", val_v2),
    ("TEST", test_v2)
]:

    counts = Counter(
        x["answer"] for x in dataset
    )

    print()
    print(name)

    for answer in [
        "front",
        "behind",
        "left",
        "right"
    ]:
        print(
            f"  {answer:<8}: {counts[answer]}"
        )


# ============================================================
# TRANSFORMATION COVERAGE
# ============================================================

print()
print("TRANSFORMATION COVERAGE")
print("-" * 70)

for name, dataset in [
    ("TRAIN", train_v2),
    ("VALIDATION", val_v2),
    ("TEST", test_v2)
]:

    pairs = {
        (
            x["heading"],
            x["world_direction"]
        )
        for x in dataset
    }

    print(
        f"{name}: {len(pairs)}/16 heading-direction pairs"
    )


# ============================================================
# ASSERTIONS
# ============================================================

assert len(train_v2) == 2000
assert len(val_v2) == 400
assert len(test_v2) == 400

assert len(train_text & val_text) == 0
assert len(train_text & test_text) == 0
assert len(val_text & test_text) == 0

assert len({
    (x["heading"], x["world_direction"])
    for x in train_v2
}) == 16

assert len({
    (x["heading"], x["world_direction"])
    for x in val_v2
}) == 16

assert len({
    (x["heading"], x["world_direction"])
    for x in test_v2
}) == 16

print()
print("=" * 70)
print("ALL ASSERTIONS PASSED")
print("=" * 70)


# ============================================================
# SAVE
# ============================================================

os.makedirs(
    "/content/egospatial_v2_data",
    exist_ok=True
)

with open(
    "/content/egospatial_v2_data/train.json",
    "w"
) as f:
    json.dump(train_v2, f, indent=2)

with open(
    "/content/egospatial_v2_data/validation.json",
    "w"
) as f:
    json.dump(val_v2, f, indent=2)

with open(
    "/content/egospatial_v2_data/test.json",
    "w"
) as f:
    json.dump(test_v2, f, indent=2)

print()
print("FILES SAVED")
print("-" * 70)
print("/content/egospatial_v2_data/train.json")
print("/content/egospatial_v2_data/validation.json")
print("/content/egospatial_v2_data/test.json")

V2 CLEAN DATASET VERIFICATION

SIZES
----------------------------------------------------------------------
Train      : 2000
Validation : 400
Test       : 400

EXACT TEXT OVERLAP
----------------------------------------------------------------------
Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0

ANSWER DISTRIBUTION
----------------------------------------------------------------------

TRAIN
  front   : 543
  behind  : 477
  left    : 468
  right   : 512

VALIDATION
  front   : 99
  behind  : 111
  left    : 89
  right   : 101

TEST
  front   : 104
  behind  : 104
  left    : 97
  right   : 95

TRANSFORMATION COVERAGE
----------------------------------------------------------------------
TRAIN: 16/16 heading-direction pairs
VALIDATION: 16/16 heading-direction pairs
TEST: 16/16 heading-direction pairs

ALL ASSERTIONS PASSED

FILES SAVED
----------------------------------------------------------------------
/content/egospatial_v2_data/train.json
/content/egospatial_v

In [ ]:
# ============================================================
# STEP 64B — PREPARE V2 FOR GEMMA
# ============================================================

import json
from datasets import Dataset

TRAIN_PATH = "/content/egospatial_v2_data/train.json"
VAL_PATH   = "/content/egospatial_v2_data/validation.json"
TEST_PATH  = "/content/egospatial_v2_data/test.json"

with open(TRAIN_PATH) as f:
    train_raw = json.load(f)

with open(VAL_PATH) as f:
    val_raw = json.load(f)

with open(TEST_PATH) as f:
    test_raw = json.load(f)

print("Raw datasets:")
print("Train:", len(train_raw))
print("Val  :", len(val_raw))
print("Test :", len(test_raw))


# ============================================================
# GEMMA CHAT FORMAT
# ============================================================

SYSTEM_PROMPT = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right
"""


def make_messages(example):

    user_text = f"""Situation:
{example["situation"]}

Question:
{example["question"]}"""

    return [
        {
            "role": "user",
            "content": SYSTEM_PROMPT + "\n" + user_text
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]


# ============================================================
# BUILD DATASETS
# ============================================================

train_messages = [
    make_messages(x)
    for x in train_raw
]

val_messages = [
    make_messages(x)
    for x in val_raw
]

test_messages = [
    make_messages(x)
    for x in test_raw
]


train_dataset = Dataset.from_dict({
    "messages": train_messages
})

val_dataset = Dataset.from_dict({
    "messages": val_messages
})

test_dataset = Dataset.from_dict({
    "messages": test_messages
})


print()
print("Prepared datasets:")
print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))


# ============================================================
# VERIFY ONE EXAMPLE
# ============================================================

print()
print("=" * 70)
print("SAMPLE")
print("=" * 70)

sample = train_dataset[0]

print()
print("USER:")
print(sample["messages"][0]["content"])

print()
print("MODEL TARGET:")
print(sample["messages"][1]["content"])


# ============================================================
# CRITICAL LEAKAGE CHECK
# ============================================================

for dataset_name, dataset in [
    ("train", train_dataset),
    ("validation", val_dataset),
    ("test", test_dataset)
]:

    for example in dataset:

        user_content = example["messages"][0]["content"]
        answer = example["messages"][1]["content"]

        # The answer must NOT literally appear
        # as a standalone relative-direction field.
        assert "relative_direction" not in user_content

    print(
        f"{dataset_name}: answer-field leakage check PASSED"
    )


print()
print("=" * 70)
print("STEP 64B DATA PREPARATION COMPLETE")
print("=" * 70)

Raw datasets:
Train: 2000
Val  : 400
Test : 400

Prepared datasets:
Train: 2000
Val  : 400
Test : 400

SAMPLE

USER:
You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right

Situation:
I am facing facing north. A blue sphere is directly east of me. There is a purple bottle directly north of me. A yellow table is north of me.

Question:
Where would I see the table?

MODEL TARGET:
front
train: answer-field leakage check PASSED
validation: answer-field leakage check PASSED
test: answer-field leakage check PASSED

STEP 64B DATA PREPARATION COMPLETE


In [ ]:
# ============================================================
# STEP 64B.1 — REMOVE "facing facing" ARTIFACT
# ============================================================

def clean_facing_phrase(text):
    return text.replace(
        "I am facing facing ",
        "I am facing "
    )


for dataset in [train_raw, val_raw, test_raw]:
    for example in dataset:
        example["situation"] = clean_facing_phrase(
            example["situation"]
        )


# Rebuild messages
train_messages = [make_messages(x) for x in train_raw]
val_messages   = [make_messages(x) for x in val_raw]
test_messages  = [make_messages(x) for x in test_raw]

train_dataset = Dataset.from_dict({"messages": train_messages})
val_dataset   = Dataset.from_dict({"messages": val_messages})
test_dataset  = Dataset.from_dict({"messages": test_messages})


# Verify no artifact remains
for name, dataset in [
    ("train", train_dataset),
    ("validation", val_dataset),
    ("test", test_dataset)
]:
    for example in dataset:
        assert "facing facing" not in example["messages"][0]["content"]

    print(f"{name}: wording check PASSED")


# Save cleaned raw JSON
with open(TRAIN_PATH, "w") as f:
    json.dump(train_raw, f, indent=2)

with open(VAL_PATH, "w") as f:
    json.dump(val_raw, f, indent=2)

with open(TEST_PATH, "w") as f:
    json.dump(test_raw, f, indent=2)


print()
print("=" * 70)
print("CLEAN V2 DATASET SAVED")
print("=" * 70)

print()
print(train_dataset[0]["messages"][0]["content"])
print()
print("TARGET:", train_dataset[0]["messages"][1]["content"])

train: wording check PASSED
validation: wording check PASSED
test: wording check PASSED

CLEAN V2 DATASET SAVED

You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right

Situation:
I am facing north. A blue sphere is directly east of me. There is a purple bottle directly north of me. A yellow table is north of me.

Question:
Where would I see the table?

TARGET: front


In [ ]:
# ============================================================
# STEP 64C — V2 GEMMA TOKENIZATION + LABEL VERIFICATION
# ============================================================

import torch
from collections import Counter

MAX_SEQ_LENGTH = 256


# ------------------------------------------------------------
# 1. FORMAT USING GEMMA'S NATIVE CHAT TEMPLATE
# ------------------------------------------------------------

def tokenize_example(example):

    messages = example["messages"]

    # Full conversation INCLUDING the answer
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Prompt ONLY — no model answer
    prompt_text = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True
    )

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    input_ids = torch.tensor(
        full_tokens["input_ids"],
        dtype=torch.long
    )

    attention_mask = torch.tensor(
        full_tokens["attention_mask"],
        dtype=torch.long
    )

    labels = input_ids.clone()

    prompt_len = len(prompt_tokens["input_ids"])

    # Mask everything before the model response
    labels[:prompt_len] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


# ------------------------------------------------------------
# 2. TOKENIZE ALL SPLITS
# ------------------------------------------------------------

train_tokenized = [
    tokenize_example(x)
    for x in train_dataset
]

val_tokenized = [
    tokenize_example(x)
    for x in val_dataset
]

test_tokenized = [
    tokenize_example(x)
    for x in test_dataset
]


print("=" * 70)
print("TOKENIZATION COMPLETE")
print("=" * 70)

print()
print("Train:", len(train_tokenized))
print("Val  :", len(val_tokenized))
print("Test :", len(test_tokenized))


# ------------------------------------------------------------
# 3. SEQUENCE LENGTH STATISTICS
# ------------------------------------------------------------

def length_stats(dataset):

    lengths = [
        len(x["input_ids"])
        for x in dataset
    ]

    supervised = [
        int((x["labels"] != -100).sum())
        for x in dataset
    ]

    return {
        "min": min(lengths),
        "max": max(lengths),
        "avg": sum(lengths) / len(lengths),
        "supervised_min": min(supervised),
        "supervised_max": max(supervised),
        "supervised_avg": sum(supervised) / len(supervised)
    }


for name, dataset in [
    ("TRAIN", train_tokenized),
    ("VALIDATION", val_tokenized),
    ("TEST", test_tokenized)
]:

    stats = length_stats(dataset)

    print()
    print(name)
    print("-" * 70)
    print(
        f"Sequence length:"
        f" min={stats['min']}"
        f" max={stats['max']}"
        f" avg={stats['avg']:.2f}"
    )

    print(
        f"Supervised tokens:"
        f" min={stats['supervised_min']}"
        f" max={stats['supervised_max']}"
        f" avg={stats['supervised_avg']:.2f}"
    )


# ------------------------------------------------------------
# 4. CRITICAL SUPERVISION CHECK
# ------------------------------------------------------------

for name, dataset in [
    ("TRAIN", train_tokenized),
    ("VALIDATION", val_tokenized),
    ("TEST", test_tokenized)
]:

    zero_supervision = 0
    invalid = 0

    for item in dataset:

        supervised_count = int(
            (item["labels"] != -100).sum()
        )

        # Every example MUST supervise something
        if supervised_count == 0:
            zero_supervision += 1

        # Labels must equal input_ids wherever supervised
        mask = item["labels"] != -100

        if not torch.equal(
            item["labels"][mask],
            item["input_ids"][mask]
        ):
            invalid += 1

    print()
    print(
        f"{name}:"
        f" zero-supervision={zero_supervision}"
        f" invalid-labels={invalid}"
    )

    assert zero_supervision == 0
    assert invalid == 0


# ------------------------------------------------------------
# 5. INSPECT ONE COMPLETE EXAMPLE
# ------------------------------------------------------------

sample = train_tokenized[0]

print()
print("=" * 70)
print("SAMPLE TOKENIZATION")
print("=" * 70)

print()
print("FULL DECODED:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )
)

print()
print("SUPERVISED TARGET:")
print(
    tokenizer.decode(
        sample["input_ids"][
            sample["labels"] != -100
        ],
        skip_special_tokens=False
    )
)

print()
print(
    "Total tokens:",
    len(sample["input_ids"])
)

print(
    "Supervised tokens:",
    int((sample["labels"] != -100).sum())
)


# ------------------------------------------------------------
# 6. MAKE SURE ANSWER IS THE ONLY SEMANTIC TARGET
# ------------------------------------------------------------

allowed_answers = {
    "front",
    "behind",
    "left",
    "right"
}

for raw, tokenized in zip(
    train_raw,
    train_tokenized
):

    answer = raw["answer"]

    assert answer in allowed_answers

    supervised_text = tokenizer.decode(
        tokenized["input_ids"][
            tokenized["labels"] != -100
        ],
        skip_special_tokens=True
    ).strip()

    assert answer in supervised_text


print()
print("=" * 70)
print("ALL 64C ASSERTIONS PASSED")
print("=" * 70)

TOKENIZATION COMPLETE

Train: 2000
Val  : 400
Test : 400

TRAIN
----------------------------------------------------------------------
Sequence length: min=78 max=105 avg=89.10
Supervised tokens: min=3 max=3 avg=3.00

VALIDATION
----------------------------------------------------------------------
Sequence length: min=78 max=104 avg=89.21
Supervised tokens: min=3 max=3 avg=3.00

TEST
----------------------------------------------------------------------
Sequence length: min=78 max=103 avg=89.83
Supervised tokens: min=3 max=3 avg=3.00

TRAIN: zero-supervision=0 invalid-labels=0

VALIDATION: zero-supervision=0 invalid-labels=0

TEST: zero-supervision=0 invalid-labels=0

SAMPLE TOKENIZATION

FULL DECODED:
<bos><start_of_turn>user
You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right

Situation:
I am facing north. A blue sphere is directly east of me. There is a purple b

In [ ]:
# ============================================================
# STEP 64D — TRAIN EGO-SPATIAL-GEMMA V2
# ============================================================

import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model
)

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "google/gemma-2-2b-it"

OUTPUT_DIR = "/content/egospatial_gemma_v2"

MAX_SEQ_LENGTH = 256

BATCH_SIZE = 2
GRAD_ACCUM = 4

LEARNING_RATE = 1e-4
WARMUP_STEPS = 20

NUM_EPOCHS = 1

SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 1. FRESH TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 2. FRESH BASE MODEL
# ============================================================

print("=" * 70)
print("LOADING FRESH GEMMA")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

print()
print("Model:", type(model).__name__)
print("Device:", model.device)
print("Dtype:", model.dtype)


# ============================================================
# 3. FRESH LORA
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()


# ============================================================
# 4. COLLATOR
# ============================================================

def v2_collator(features):

    max_len = max(
        len(f["input_ids"])
        for f in features
    )

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:

        pad_len = max_len - len(f["input_ids"])

        input_ids.append(
            torch.cat([
                f["input_ids"],
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])
        )

        attention_masks.append(
            torch.cat([
                f["attention_mask"],
                torch.zeros(
                    pad_len,
                    dtype=torch.long
                )
            ])
        )

        labels.append(
            torch.cat([
                f["labels"],
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels)
    }


# ============================================================
# 5. TRAIN DATASET
# ============================================================

# Convert our tokenized examples into Dataset format
from datasets import Dataset

train_ds = Dataset.from_dict({
    "input_ids": [
        x["input_ids"].tolist()
        for x in train_tokenized
    ],
    "attention_mask": [
        x["attention_mask"].tolist()
        for x in train_tokenized
    ],
    "labels": [
        x["labels"].tolist()
        for x in train_tokenized
    ]
})


# ============================================================
# 6. PRE-FLIGHT
# ============================================================

print()
print("=" * 70)
print("PRE-FLIGHT")
print("=" * 70)

batch = v2_collator([
    train_tokenized[0],
    train_tokenized[1]
])

print("Batch shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)

print(
    "Supervised tokens:",
    int((batch["labels"] != -100).sum())
)

print(
    "Dataset:",
    len(train_ds)
)

print(
    "Effective batch:",
    BATCH_SIZE * GRAD_ACCUM
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 7. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,

    fp16=True,
    gradient_checkpointing=False,

    optim="adamw_torch",

    logging_steps=10,

    save_strategy="epoch",

    save_total_limit=2,

    seed=SEED,
    data_seed=SEED,

    remove_unused_columns=False,

    report_to="none"
)


# ============================================================
# 8. TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=v2_collator
)


# ============================================================
# 9. FINAL PREFLIGHT CHECK
# ============================================================

print()
print("=" * 70)
print("TRAINER READY")
print("=" * 70)

print("Train examples:", len(train_ds))
print("Epochs:", NUM_EPOCHS)
print("Batch:", BATCH_SIZE)
print("Grad accumulation:", GRAD_ACCUM)
print("Effective batch:", BATCH_SIZE * GRAD_ACCUM)
print("Learning rate:", LEARNING_RATE)
print("Warmup steps:", WARMUP_STEPS)

print()
print("Starting V2 training...")


# ============================================================
# 10. TRAIN
# ============================================================

train_result = trainer.train()


# ============================================================
# 11. SAVE
# ============================================================

FINAL_DIR = f"{OUTPUT_DIR}/final"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print()
print("=" * 70)
print("V2 TRAINING COMPLETE")
print("=" * 70)

print("Final model:", FINAL_DIR)
print("Training loss:", train_result.training_loss)
print("Global steps:", trainer.state.global_step)

LOADING FRESH GEMMA


NameError: name 'HF_TOKEN' is not defined

In [ ]:
# ============================================================
# STEP 64D.1 — RESTORE HUGGING FACE AUTH
# ============================================================

from huggingface_hub import login, whoami

login()

print("Authenticated as:", whoami()["name"])

Authenticated as: Platinum04


In [ ]:
# Restore the token variable required by the training cell

from huggingface_hub import HfFolder

HF_TOKEN = HfFolder.get_token()

assert HF_TOKEN is not None

print("HF token available:", True)

ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/__init__.py)

In [ ]:
from huggingface_hub import login, whoami, get_token

token = get_token()

if token is None:
    login()
    token = get_token()

print("Authenticated as:", whoami()["name"])
print("HF token available:", token is not None)

HF_TOKEN = token

Authenticated as: Platinum04
HF token available: True


In [ ]:
# ============================================================
# STEP 64D — EGO-SPATIAL-GEMMA V2 TRAINING
# ============================================================
#
# Fresh Gemma 2B + Fresh LoRA
# V2 synthetic spatial reasoning curriculum
#
# IMPORTANT:
# - Does NOT load V1 adapter
# - Does NOT resume from checkpoint
# - Uses ONLY train_tokenized
# - Validation/test remain untouched
# ============================================================

import os
import gc
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model


# ============================================================
# 0. CONFIGURATION
# ============================================================

MODEL_NAME = "google/gemma-2-2b-it"

OUTPUT_DIR = "/content/egospatial_gemma_v2"

MAX_SEQ_LENGTH = 256

BATCH_SIZE = 2
GRAD_ACCUMULATION = 4

EFFECTIVE_BATCH_SIZE = (
    BATCH_SIZE * GRAD_ACCUMULATION
)

LEARNING_RATE = 1e-4

# 2,000 examples / effective batch 8 = 250 optimizer steps
# 20 warmup steps = 8% of training
WARMUP_STEPS = 20

NUM_EPOCHS = 1

SEED = 42

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 1. ENVIRONMENT CHECK
# ============================================================

print("=" * 70)
print("STEP 64D — V2 TRAINING")
print("=" * 70)

print()
print("Environment")
print("-" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), \
    "CUDA is required for V2 training."

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

assert HF_TOKEN is not None

print("HF authentication: available")


# ============================================================
# 2. VERIFY V2 TOKENIZED DATA EXISTS
# ============================================================

assert "train_tokenized" in globals(), \
    "train_tokenized is missing. Run STEP 64C first."

assert len(train_tokenized) == 2000
assert len(val_tokenized) == 400
assert len(test_tokenized) == 400

print()
print("V2 tokenized data")
print("-" * 70)
print("Train:", len(train_tokenized))
print("Validation:", len(val_tokenized))
print("Test:", len(test_tokenized))


# ============================================================
# 3. FREE ANY OLD MODEL OBJECTS
# ============================================================

print()
print("Clearing previous model objects...")

for variable_name in [
    "model",
    "trainer"
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 4. LOAD FRESH TOKENIZER
# ============================================================

print()
print("=" * 70)
print("LOADING FRESH TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(
    "Tokenizer:",
    type(tokenizer).__name__
)

print(
    "Pad token:",
    repr(tokenizer.pad_token)
)


# ============================================================
# 5. LOAD FRESH BASE GEMMA
# ============================================================

print()
print("=" * 70)
print("LOADING FRESH GEMMA 2B")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

print()
print("Model:", type(model).__name__)
print("Device:", model.device)
print("Dtype:", model.dtype)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 6. FRESH LORA
# ============================================================

print()
print("=" * 70)
print("CREATING FRESH V2 LoRA")
print("=" * 70)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()


# ============================================================
# 7. DYNAMIC PADDING COLLATOR
# ============================================================

def v2_collator(features):

    max_len = max(
        len(f["input_ids"])
        for f in features
    )

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:

        input_ids_tensor = torch.tensor(
            f["input_ids"],
            dtype=torch.long
        )

        attention_tensor = torch.tensor(
            f["attention_mask"],
            dtype=torch.long
        )

        labels_tensor = torch.tensor(
            f["labels"],
            dtype=torch.long
        )

        pad_len = (
            max_len
            - len(input_ids_tensor)
        )

        if pad_len > 0:

            input_ids_tensor = torch.cat([
                input_ids_tensor,
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])

            attention_tensor = torch.cat([
                attention_tensor,
                torch.zeros(
                    pad_len,
                    dtype=torch.long
                )
            ])

            labels_tensor = torch.cat([
                labels_tensor,
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])

        input_ids.append(input_ids_tensor)
        attention_masks.append(attention_tensor)
        labels.append(labels_tensor)

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels)
    }


# ============================================================
# 8. CONVERT TOKENIZED DATA TO HF DATASET
# ============================================================

train_ds = Dataset.from_dict({
    "input_ids": [
        x["input_ids"].tolist()
        if torch.is_tensor(x["input_ids"])
        else x["input_ids"]
        for x in train_tokenized
    ],

    "attention_mask": [
        x["attention_mask"].tolist()
        if torch.is_tensor(x["attention_mask"])
        else x["attention_mask"]
        for x in train_tokenized
    ],

    "labels": [
        x["labels"].tolist()
        if torch.is_tensor(x["labels"])
        else x["labels"]
        for x in train_tokenized
    ]
})


# ============================================================
# 9. PREFLIGHT BATCH
# ============================================================

print()
print("=" * 70)
print("TRAINING PREFLIGHT")
print("=" * 70)

preflight_batch = v2_collator([
    train_tokenized[0],
    train_tokenized[1]
])

print()
print(
    "Batch input shape:",
    tuple(preflight_batch["input_ids"].shape)
)

print(
    "Batch labels shape:",
    tuple(preflight_batch["labels"].shape)
)

print(
    "Supervised tokens:",
    int(
        (preflight_batch["labels"] != -100).sum()
    )
)

print(
    "Dataset size:",
    len(train_ds)
)

print(
    "Per-device batch:",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUMULATION
)

print(
    "Effective batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Expected optimizer steps:",
    len(train_ds)
    // EFFECTIVE_BATCH_SIZE
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Warmup steps:",
    WARMUP_STEPS
)

print(
    "Output directory:",
    OUTPUT_DIR
)


# ============================================================
# 10. FINITE FORWARD-PASS CHECK
# ============================================================

print()
print("Running finite forward-pass check...")

model.eval()

with torch.no_grad():

    device = next(
        p for p in model.parameters()
        if p.requires_grad
    ).device

    test_batch = {
        key: value.to(device)
        for key, value in preflight_batch.items()
    }

    outputs = model(
        input_ids=test_batch["input_ids"],
        attention_mask=test_batch["attention_mask"],
        labels=test_batch["labels"]
    )

    test_loss = outputs.loss


print(
    "Forward-pass loss:",
    float(test_loss)
)

assert torch.isfinite(test_loss), \
    "Forward-pass loss is not finite."

print("Forward-pass check: PASSED")


# ============================================================
# 11. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    warmup_steps=WARMUP_STEPS,

    fp16=True,

    gradient_checkpointing=False,

    optim="adamw_torch",

    logging_strategy="steps",

    logging_steps=10,

    save_strategy="epoch",

    save_total_limit=2,

    seed=SEED,

    data_seed=SEED,

    remove_unused_columns=False,

    report_to="none",

    dataloader_pin_memory=True,

    dataloader_num_workers=0
)


# ============================================================
# 12. TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=v2_collator
)


# ============================================================
# 13. TRAIN
# ============================================================

print()
print("=" * 70)
print("TRAINER READY")
print("=" * 70)

print()
print("Starting V2 training...")
print()
print("DO NOT INTERRUPT THE RUNTIME.")
print()


train_result = trainer.train()


# ============================================================
# 14. SAVE FINAL ADAPTER
# ============================================================

FINAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final"
)

print()
print("=" * 70)
print("SAVING V2 MODEL")
print("=" * 70)

trainer.save_model(
    FINAL_DIR
)

tokenizer.save_pretrained(
    FINAL_DIR
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("V2 TRAINING COMPLETE")
print("=" * 70)

print()
print("Final directory:")
print(FINAL_DIR)

print()
print(
    "Training loss:",
    train_result.training_loss
)

print(
    "Global steps:",
    trainer.state.global_step
)

print(
    "Epoch:",
    trainer.state.epoch
)

print()
print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print()
print("=" * 70)
print("NEXT: CONTROLLED V2 EVALUATION")
print("=" * 70)

STEP 64D — V2 TRAINING

Environment
----------------------------------------------------------------------
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
HF authentication: available

V2 tokenized data
----------------------------------------------------------------------
Train: 2000
Validation: 400
Test: 400

Clearing previous model objects...
GPU memory allocated: 4.95 GB

LOADING FRESH TOKENIZER
Tokenizer: GemmaTokenizer
Pad token: '<pad>'

LOADING FRESH GEMMA 2B


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


Model: Gemma2ForCausalLM
Device: cuda:0
Dtype: torch.float16
GPU memory allocated: 9.82 GB

CREATING FRESH V2 LoRA
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881

TRAINING PREFLIGHT

Batch input shape: (2, 97)
Batch labels shape: (2, 97)
Supervised tokens: 6
Dataset size: 2000
Per-device batch: 2
Gradient accumulation: 4
Effective batch: 8
Expected optimizer steps: 250
Learning rate: 0.0001
Warmup steps: 20
Output directory: /content/egospatial_gemma_v2

Running finite forward-pass check...


/tmp/ipykernel_680/361945081.py:263: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids_tensor = torch.tensor(
/tmp/ipykernel_680/361945081.py:268: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_tensor = torch.tensor(
/tmp/ipykernel_680/361945081.py:273: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(


Forward-pass loss: 11.741360664367676
Forward-pass check: PASSED

TRAINER READY

Starting V2 training...

DO NOT INTERRUPT THE RUNTIME.



Step,Training Loss
10,8.535974
20,0.543114
30,0.464454
40,0.216628
50,0.311084
60,0.180084
70,0.137902
80,0.118311
90,0.142200
100,0.125672



SAVING V2 MODEL

V2 TRAINING COMPLETE

Final directory:
/content/egospatial_gemma_v2/final

Training loss: 0.502695803642273
Global steps: 250
Epoch: 1.0

GPU memory allocated: 10.16 GB

NEXT: CONTROLLED V2 EVALUATION


In [ ]:
# ============================================================
# STEP 64E — CONTROLLED V2 SPATIAL REASONING TEST
# ============================================================
#
# V2 TEST:
# 400 completely held-out examples
#
# INPUT:
#   agent heading
#   world-relative object direction
#   question
#
# HIDDEN TARGET:
#   front / behind / left / right
#
# NO relative_direction IN INPUT
# NO FURTHER TRAINING
# ============================================================

import os
import gc
import json
import torch

from collections import Counter, defaultdict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import PeftModel


# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "google/gemma-2-2b-it"

V2_ADAPTER = (
    "/content/egospatial_gemma_v2/final"
)

MAX_NEW_TOKENS = 5

ALLOWED_ANSWERS = {
    "front",
    "behind",
    "left",
    "right"
}


# ============================================================
# 1. VERIFY TEST SET
# ============================================================

assert "test_raw" in globals(), \
    "test_raw is missing."

assert len(test_raw) == 400

print("=" * 70)
print("STEP 64E — CONTROLLED V2 TEST")
print("=" * 70)

print()
print("Test examples:", len(test_raw))
print("Adapter:", V2_ADAPTER)


# ============================================================
# 2. CLEAR TRAINING MODEL
# ============================================================

print()
print("Clearing training model...")

if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 3. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 4. LOAD FRESH BASE GEMMA
# ============================================================

print()
print("=" * 70)
print("LOADING FRESH BASE GEMMA")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

base_model.config.use_cache = True


# ============================================================
# 5. LOAD V2 ADAPTER
# ============================================================

print()
print("=" * 70)
print("LOADING V2 ADAPTER")
print("=" * 70)

model = PeftModel.from_pretrained(
    base_model,
    V2_ADAPTER
)

model.eval()

print(
    "Model:",
    type(model).__name__
)

print(
    "Device:",
    model.device
)

print(
    "Dtype:",
    model.dtype
)


# ============================================================
# 6. BUILD EXACT EVALUATION PROMPT
# ============================================================

SYSTEM_PROMPT = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right
"""


def build_prompt(example):

    return f"""<bos><start_of_turn>user
{SYSTEM_PROMPT}

Situation:
{example["situation"]}

Question:
{example["question"]}<end_of_turn>
<start_of_turn>model
"""


# ============================================================
# 7. CONTROLLED GENERATION
# ============================================================

def extract_answer(text):

    text = text.strip().lower()

    # Remove common Gemma special-token artifacts
    for token in [
        "<end_of_turn>",
        "<eos>",
        "<bos>",
        "<start_of_turn>",
        "<end_of_turn>"
    ]:
        text = text.replace(token, "")

    text = text.strip()

    # Exact answer first
    if text in ALLOWED_ANSWERS:
        return text

    # Otherwise inspect first line / first token
    first_line = text.split("\n")[0].strip()

    if first_line in ALLOWED_ANSWERS:
        return first_line

    # Controlled fallback:
    # only accept an allowed answer if it is the first
    # meaningful word.
    words = first_line.split()

    if words and words[0] in ALLOWED_ANSWERS:
        return words[0]

    return "INVALID"


# ============================================================
# 8. RUN TEST
# ============================================================

predictions = []
ground_truth = []

print()
print("=" * 70)
print("RUNNING 400-EXAMPLE CONTROLLED TEST")
print("=" * 70)

for i, example in enumerate(test_raw):

    prompt = build_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        generated = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            temperature=None,

            top_p=None,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=tokenizer.eos_token_id
        )

    # Only decode newly generated tokens
    new_tokens = generated[
        0,
        inputs["input_ids"].shape[1]:
    ]

    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=False
    )

    prediction = extract_answer(
        raw_output
    )

    predictions.append(prediction)
    ground_truth.append(
        example["answer"]
    )

    if (i + 1) % 50 == 0:
        print(
            f"Evaluated {i + 1}/400"
        )


# ============================================================
# 9. OVERALL ACCURACY
# ============================================================

correct = sum(
    p == g
    for p, g in zip(
        predictions,
        ground_truth
    )
)

total = len(test_raw)

accuracy = (
    correct / total * 100
)

print()
print("=" * 70)
print("OVERALL RESULT")
print("=" * 70)

print()
print(
    f"Correct: {correct}/{total}"
)

print(
    f"Accuracy: {accuracy:.2f}%"
)


# ============================================================
# 10. ANSWER-FAMILY ACCURACY
# ============================================================

print()
print("=" * 70)
print("ANSWER ACCURACY")
print("=" * 70)

for answer in [
    "front",
    "behind",
    "left",
    "right"
]:

    indices = [
        i
        for i, g in enumerate(
            ground_truth
        )
        if g == answer
    ]

    family_correct = sum(
        predictions[i] == answer
        for i in indices
    )

    family_total = len(indices)

    family_accuracy = (
        family_correct / family_total * 100
        if family_total
        else 0
    )

    print(
        f"{answer:<8}: "
        f"{family_correct}/{family_total} "
        f"({family_accuracy:.2f}%)"
    )


# ============================================================
# 11. CONFUSION MATRIX
# ============================================================

labels = [
    "front",
    "behind",
    "left",
    "right"
]

confusion = {
    truth: Counter()
    for truth in labels
}

for truth, prediction in zip(
    ground_truth,
    predictions
):

    confusion[truth][prediction] += 1


print()
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print()
print(
    f"{'Truth':<10}"
    + "".join(
        f"{label:<10}"
        for label in labels
    )
    + "INVALID"
)

for truth in labels:

    print(
        f"{truth:<10}"
        + "".join(
            f"{confusion[truth][pred]:<10}"
            for pred in labels
        )
        + f"{confusion[truth]['INVALID']}"
    )


# ============================================================
# 12. ACCURACY BY AGENT HEADING
# ============================================================

heading_results = defaultdict(
    lambda: [0, 0]
)

for example, prediction in zip(
    test_raw,
    predictions
):

    heading = example["heading"]

    heading_results[heading][1] += 1

    if prediction == example["answer"]:
        heading_results[heading][0] += 1


print()
print("=" * 70)
print("ACCURACY BY AGENT HEADING")
print("=" * 70)

for heading in [
    "north",
    "east",
    "south",
    "west"
]:

    c, n = heading_results[heading]

    print(
        f"{heading:<8}: "
        f"{c}/{n} "
        f"({c/n*100:.2f}%)"
    )


# ============================================================
# 13. ACCURACY BY WORLD DIRECTION
# ============================================================

world_results = defaultdict(
    lambda: [0, 0]
)

for example, prediction in zip(
    test_raw,
    predictions
):

    direction = example[
        "world_direction"
    ]

    world_results[direction][1] += 1

    if prediction == example["answer"]:
        world_results[direction][0] += 1


print()
print("=" * 70)
print("ACCURACY BY WORLD DIRECTION")
print("=" * 70)

for direction in [
    "north",
    "east",
    "south",
    "west"
]:

    c, n = world_results[direction]

    print(
        f"{direction:<8}: "
        f"{c}/{n} "
        f"({c/n*100:.2f}%)"
    )


# ============================================================
# 14. PREDICTION DISTRIBUTION
# ============================================================

print()
print("=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

prediction_counts = Counter(
    predictions
)

for answer in [
    "front",
    "behind",
    "left",
    "right",
    "INVALID"
]:

    print(
        f"{answer:<8}: "
        f"{prediction_counts[answer]}"
    )


# ============================================================
# 15. SHOW ERRORS
# ============================================================

print()
print("=" * 70)
print("FIRST 20 ERRORS")
print("=" * 70)

error_count = 0

for i, (
    example,
    prediction,
    truth
) in enumerate(
    zip(
        test_raw,
        predictions,
        ground_truth
    )
):

    if prediction != truth:

        print()
        print(
            f"[{i}]"
        )

        print(
            "Heading:",
            example["heading"]
        )

        print(
            "World direction:",
            example["world_direction"]
        )

        print(
            "Situation:",
            example["situation"]
        )

        print(
            "Question:",
            example["question"]
        )

        print(
            "Expected:",
            truth
        )

        print(
            "Predicted:",
            prediction
        )

        error_count += 1

        if error_count >= 20:
            break


# ============================================================
# 16. SAVE RESULTS
# ============================================================

results = []

for i, (
    example,
    prediction
) in enumerate(
    zip(
        test_raw,
        predictions
    )
):

    results.append({
        "id": example["id"],
        "heading": example["heading"],
        "world_direction": example[
            "world_direction"
        ],
        "object": example["object"],
        "object_type": example[
            "object_type"
        ],
        "expected": example["answer"],
        "prediction": prediction,
        "correct": prediction == example["answer"]
    })


RESULT_PATH = (
    "/content/"
    "egospatial_v2_controlled_results.json"
)

with open(
    RESULT_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


print()
print("=" * 70)
print("V2 CONTROLLED TEST COMPLETE")
print("=" * 70)

print()
print(
    "Results saved to:"
)

print(RESULT_PATH)

print()
print(
    "V1 controlled baseline: 9.76%"
)

print(
    f"V2 controlled result: {accuracy:.2f}%"
)

print()
print("=" * 70)

STEP 64E — CONTROLLED V2 TEST

Test examples: 400
Adapter: /content/egospatial_gemma_v2/final

Clearing training model...
GPU memory allocated: 5.06 GB

LOADING FRESH BASE GEMMA


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


LOADING V2 ADAPTER
Model: PeftModelForCausalLM
Device: cuda:0
Dtype: torch.float16

RUNNING 400-EXAMPLE CONTROLLED TEST
Evaluated 50/400
Evaluated 100/400
Evaluated 150/400
Evaluated 200/400
Evaluated 250/400
Evaluated 300/400
Evaluated 350/400
Evaluated 400/400

OVERALL RESULT

Correct: 305/400
Accuracy: 76.25%

ANSWER ACCURACY
front   : 104/104 (100.00%)
behind  : 104/104 (100.00%)
left    : 2/97 (2.06%)
right   : 95/95 (100.00%)

CONFUSION MATRIX

Truth     front     behind    left      right     INVALID
front     104       0         0         0         0
behind    0         104       0         0         0
left      0         0         2         95        0
right     0         0         0         95        0

ACCURACY BY AGENT HEADING
north   : 84/105 (80.00%)
east    : 74/90 (82.22%)
south   : 69/94 (73.40%)
west    : 78/111 (70.27%)

ACCURACY BY WORLD DIRECTION
north   : 89/105 (84.76%)
east    : 79/104 (75.96%)
south   : 71/104 (68.27%)
west    : 66/87 (75.86%)

PREDICTION DISTR

In [ ]:
# ============================================================
# STEP 65 — 16-TRANSFORMATION DIAGNOSTIC
# ============================================================
#
# NO TRAINING
# NO DATASET MODIFICATION
# NO ADAPTER MODIFICATION
#
# Tests every heading × world-direction combination
# equally.
# ============================================================

import torch
from collections import defaultdict, Counter

# ------------------------------------------------------------
# VERIFIED TRANSFORMATION MAP
# ------------------------------------------------------------

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },

    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },

    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },

    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ------------------------------------------------------------
# FIXED DIAGNOSTIC EXAMPLES
# ------------------------------------------------------------

objects = [
    ("cube", "red"),
    ("sphere", "blue"),
    ("chair", "green"),
    ("table", "yellow")
]

heading_phrases = {
    "north": "north",
    "east": "east",
    "south": "south",
    "west": "west"
}

direction_phrases = {
    "north": "north of me",
    "east": "east of me",
    "south": "south of me",
    "west": "west of me"
}

question_templates = [
    "Where is the {obj} relative to me?",
    "Which direction is the {obj} from me?",
    "Where would I see the {obj}?",
    "What direction is the {obj} in relative to my position?"
]


# ------------------------------------------------------------
# CREATE 16 DIAGNOSTIC CASES
# ------------------------------------------------------------

diagnostic_cases = []

case_id = 0

for heading in [
    "north",
    "east",
    "south",
    "west"
]:

    for world_direction in [
        "north",
        "east",
        "south",
        "west"
    ]:

        obj, color = objects[
            case_id % len(objects)
        ]

        question = question_templates[
            case_id % len(question_templates)
        ].format(
            obj=obj
        )

        situation = (
            f"I am facing {heading_phrases[heading]}. "
            f"A {color} {obj} is "
            f"{direction_phrases[world_direction]}."
        )

        expected = relative_map[
            heading
        ][world_direction]

        diagnostic_cases.append({
            "id": case_id,
            "heading": heading,
            "world_direction": world_direction,
            "object": obj,
            "situation": situation,
            "question": question,
            "expected": expected
        })

        case_id += 1


# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

SYSTEM_PROMPT = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right
"""


def build_prompt(example):

    return f"""<bos><start_of_turn>user
{SYSTEM_PROMPT}

Situation:
{example["situation"]}

Question:
{example["question"]}<end_of_turn>
<start_of_turn>model
"""


# ------------------------------------------------------------
# EXTRACTION
# ------------------------------------------------------------

ALLOWED = {
    "front",
    "behind",
    "left",
    "right"
}


def extract_answer(text):

    text = text.lower().strip()

    for token in [
        "<end_of_turn>",
        "<eos>",
        "<bos>",
        "<start_of_turn>"
    ]:
        text = text.replace(token, "")

    text = text.strip()

    first_line = text.split("\n")[0].strip()

    if first_line in ALLOWED:
        return first_line

    words = first_line.split()

    if words and words[0] in ALLOWED:
        return words[0]

    return "INVALID"


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

print("=" * 70)
print("STEP 65 — 16-TRANSFORMATION DIAGNOSTIC")
print("=" * 70)

results = []

for example in diagnostic_cases:

    prompt = build_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        generated = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = generated[
        0,
        inputs["input_ids"].shape[1]:
    ]

    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=False
    )

    prediction = extract_answer(
        raw_output
    )

    results.append({
        **example,
        "prediction": prediction,
        "correct": prediction == example["expected"]
    })


# ------------------------------------------------------------
# PRINT TRANSFORMATION TABLE
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRANSFORMATION MATRIX")
print("=" * 70)

print()
print(
    f"{'Facing':<10}"
    f"{'World':<10}"
    f"{'Expected':<10}"
    f"{'Predicted':<10}"
    f"{'Result'}"
)

print("-" * 55)

for r in results:

    status = (
        "✓"
        if r["correct"]
        else "✗"
    )

    print(
        f"{r['heading']:<10}"
        f"{r['world_direction']:<10}"
        f"{r['expected']:<10}"
        f"{r['prediction']:<10}"
        f"{status}"
    )


# ------------------------------------------------------------
# OVERALL
# ------------------------------------------------------------

correct = sum(
    r["correct"]
    for r in results
)

total = len(results)

print()
print("=" * 70)
print("RESULT")
print("=" * 70)

print()
print(
    f"Correct: {correct}/{total}"
)

print(
    f"Accuracy: {correct / total * 100:.2f}%"
)


# ------------------------------------------------------------
# PER-RELATION
# ------------------------------------------------------------

print()
print("PER-RELATION")
print("-" * 70)

for relation in [
    "front",
    "behind",
    "left",
    "right"
]:

    subset = [
        r for r in results
        if r["expected"] == relation
    ]

    c = sum(
        r["correct"]
        for r in subset
    )

    print(
        f"{relation:<8}: "
        f"{c}/{len(subset)} "
        f"({c / len(subset) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# PREDICTION DISTRIBUTION
# ------------------------------------------------------------

print()
print("PREDICTIONS")
print("-" * 70)

counts = Counter(
    r["prediction"]
    for r in results
)

for relation in [
    "front",
    "behind",
    "left",
    "right",
    "INVALID"
]:

    print(
        f"{relation:<8}: "
        f"{counts[relation]}"
    )


# ------------------------------------------------------------
# FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(results) == 16

print()
print("=" * 70)
print("STEP 65 COMPLETE")
print("=" * 70)

STEP 65 — 16-TRANSFORMATION DIAGNOSTIC

TRANSFORMATION MATRIX

Facing    World     Expected  Predicted Result
-------------------------------------------------------
north     north     front     front     ✓
north     east      right     right     ✓
north     south     behind    behind    ✓
north     west      left      right     ✗
east      north     left      right     ✗
east      east      front     front     ✓
east      south     right     right     ✓
east      west      behind    behind    ✓
south     north     behind    behind    ✓
south     east      left      right     ✗
south     south     front     front     ✓
south     west      right     right     ✓
west      north     right     right     ✓
west      east      behind    behind    ✓
west      south     left      right     ✗
west      west      front     front     ✓

RESULT

Correct: 12/16
Accuracy: 75.00%

PER-RELATION
----------------------------------------------------------------------
front   : 4/4 (100.00%)
behind  : 4/

In [ ]:
# ============================================================
# STEP 66A — TARGETED LEFT-RELATION CORRECTION CURRICULUM
# ============================================================

import json
import random
import os
from collections import Counter

# ------------------------------------------------------------
# VERIFIED TRANSFORMATION MAP
# ------------------------------------------------------------

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },
    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },
    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },
    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


# ------------------------------------------------------------
# VOCABULARY
# ------------------------------------------------------------

objects = [
    ("red cube", "cube"),
    ("blue sphere", "sphere"),
    ("green chair", "chair"),
    ("yellow table", "table"),
    ("black backpack", "backpack"),
    ("white lamp", "lamp"),
    ("brown desk", "desk"),
    ("gray cabinet", "cabinet"),
    ("orange box", "box"),
    ("purple bottle", "bottle"),
    ("wooden stool", "stool"),
    ("small monitor", "monitor"),
]


heading_phrases = {
    "north": [
        "north",
        "northward",
        "toward the north"
    ],
    "east": [
        "east",
        "eastward",
        "toward the east"
    ],
    "south": [
        "south",
        "southward",
        "toward the south"
    ],
    "west": [
        "west",
        "westward",
        "toward the west"
    ]
}


direction_phrases = {
    "north": [
        "north of me",
        "to my north",
        "directly north of me"
    ],
    "east": [
        "east of me",
        "to my east",
        "directly east of me"
    ],
    "south": [
        "south of me",
        "to my south",
        "directly south of me"
    ],
    "west": [
        "west of me",
        "to my west",
        "directly west of me"
    ]
}


question_templates = [
    "Where is the {obj_type} relative to me?",
    "Which direction is the {obj_type} from me?",
    "Where would I see the {obj_type}?",
    "What direction is the {obj_type} in relative to my position?"
]


# ------------------------------------------------------------
# GENERATOR
# ------------------------------------------------------------

def make_correction_example(example_id, heading, world_direction, rng):

    target_name, target_type = rng.choice(objects)

    expected = relative_map[
        heading
    ][world_direction]

    heading_phrase = rng.choice(
        heading_phrases[heading]
    )

    direction_phrase = rng.choice(
        direction_phrases[world_direction]
    )

    situation = (
        f"I am facing {heading_phrase}. "
        f"A {target_name} is {direction_phrase}."
    )

    question = rng.choice(
        question_templates
    ).format(
        obj_type=target_type
    )

    return {
        "id": example_id,
        "situation": situation,
        "question": question,
        "answer": expected,
        "heading": heading,
        "world_direction": world_direction,
        "object": target_name,
        "object_type": target_type
    }


# ------------------------------------------------------------
# BUILD CURRICULUM
#
# 800 examples total:
#
# LEFT      = 400
# FRONT     = 133
# BEHIND    = 133
# RIGHT     = 134
#
# This gives LEFT a strong corrective signal while preserving
# the other three learned relations.
# ------------------------------------------------------------

rng = random.Random(6601)

left_pairs = [
    ("north", "west"),
    ("east", "north"),
    ("south", "east"),
    ("west", "south")
]

other_pairs = [
    (heading, direction)
    for heading in relative_map
    for direction in relative_map[heading]
    if relative_map[heading][direction] != "left"
]


correction_examples = []

example_id = 0


# ------------------------------------------------------------
# 400 LEFT examples
# ------------------------------------------------------------

for _ in range(400):

    heading, world_direction = rng.choice(
        left_pairs
    )

    correction_examples.append(
        make_correction_example(
            example_id,
            heading,
            world_direction,
            rng
        )
    )

    example_id += 1


# ------------------------------------------------------------
# 400 NON-LEFT examples
# ------------------------------------------------------------

for _ in range(400):

    heading, world_direction = rng.choice(
        other_pairs
    )

    correction_examples.append(
        make_correction_example(
            example_id,
            heading,
            world_direction,
            rng
        )
    )

    example_id += 1


# Shuffle
rng.shuffle(correction_examples)


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

answer_counts = Counter(
    x["answer"]
    for x in correction_examples
)

pair_counts = Counter(
    (
        x["heading"],
        x["world_direction"]
    )
    for x in correction_examples
)


print("=" * 70)
print("STEP 66A — CORRECTION CURRICULUM")
print("=" * 70)

print()
print("Total examples:", len(correction_examples))

print()
print("ANSWER DISTRIBUTION")
print("-" * 70)

for answer in [
    "front",
    "behind",
    "left",
    "right"
]:
    print(
        f"{answer:<8}: {answer_counts[answer]}"
    )


print()
print("LEFT TRANSFORMATION COUNTS")
print("-" * 70)

for pair in left_pairs:
    print(
        f"{pair[0]:<7} + "
        f"{pair[1]:<7}: "
        f"{pair_counts[pair]}"
    )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

CORRECTION_DIR = (
    "/content/egospatial_v2_correction"
)

os.makedirs(
    CORRECTION_DIR,
    exist_ok=True
)

CORRECTION_PATH = os.path.join(
    CORRECTION_DIR,
    "left_correction.json"
)

with open(
    CORRECTION_PATH,
    "w"
) as f:
    json.dump(
        correction_examples,
        f,
        indent=2
    )


# ------------------------------------------------------------
# ASSERTIONS
# ------------------------------------------------------------

assert len(correction_examples) == 800
assert answer_counts["left"] == 400

assert all(
    pair_counts[pair] > 0
    for pair in left_pairs
)

# Make sure every example has a valid transformation
for x in correction_examples:
    assert (
        relative_map[
            x["heading"]
        ][
            x["world_direction"]
        ]
        == x["answer"]
    )

print()
print("=" * 70)
print("ALL 66A ASSERTIONS PASSED")
print("=" * 70)

print()
print("Saved:")
print(CORRECTION_PATH)

STEP 66A — CORRECTION CURRICULUM

Total examples: 800

ANSWER DISTRIBUTION
----------------------------------------------------------------------
front   : 146
behind  : 118
left    : 400
right   : 136

LEFT TRANSFORMATION COUNTS
----------------------------------------------------------------------
north   + west   : 89
east    + north  : 108
south   + east   : 110
west    + south  : 93

ALL 66A ASSERTIONS PASSED

Saved:
/content/egospatial_v2_correction/left_correction.json


In [ ]:
# ============================================================
# STEP 66B — TOKENIZE LEFT-CORRECTION CURRICULUM
# ============================================================

import json
import torch
from datasets import Dataset

CORRECTION_PATH = (
    "/content/egospatial_v2_correction/"
    "left_correction.json"
)

MAX_SEQ_LENGTH = 256


# ============================================================
# 1. LOAD
# ============================================================

with open(CORRECTION_PATH) as f:
    correction_raw = json.load(f)

assert len(correction_raw) == 800

print("=" * 70)
print("STEP 66B — CORRECTION TOKENIZATION")
print("=" * 70)

print()
print("Examples:", len(correction_raw))


# ============================================================
# 2. GEMMA CHAT FORMAT
# ============================================================

SYSTEM_PROMPT = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right
"""


def make_correction_messages(example):

    user_text = f"""Situation:
{example["situation"]}

Question:
{example["question"]}"""

    return [
        {
            "role": "user",
            "content": SYSTEM_PROMPT + "\n" + user_text
        },
        {
            "role": "model",
            "content": example["answer"]
        }
    ]


correction_messages = [
    make_correction_messages(x)
    for x in correction_raw
]


# ============================================================
# 3. TOKENIZE
# ============================================================

def tokenize_correction(example):

    messages = example["messages"]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_text = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True
    )

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    input_ids = torch.tensor(
        full_tokens["input_ids"],
        dtype=torch.long
    )

    attention_mask = torch.tensor(
        full_tokens["attention_mask"],
        dtype=torch.long
    )

    labels = input_ids.clone()

    prompt_len = len(
        prompt_tokens["input_ids"]
    )

    labels[:prompt_len] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


correction_dataset = Dataset.from_dict({
    "messages": correction_messages
})

correction_tokenized = [
    tokenize_correction(x)
    for x in correction_dataset
]


# ============================================================
# 4. STATISTICS
# ============================================================

lengths = [
    len(x["input_ids"])
    for x in correction_tokenized
]

supervised_counts = [
    int(
        (x["labels"] != -100).sum()
    )
    for x in correction_tokenized
]


print()
print("Sequence length")
print("-" * 70)

print(
    "Min:",
    min(lengths)
)

print(
    "Max:",
    max(lengths)
)

print(
    "Average:",
    round(
        sum(lengths) / len(lengths),
        2
    )
)


print()
print("Supervised tokens")
print("-" * 70)

print(
    "Min:",
    min(supervised_counts)
)

print(
    "Max:",
    max(supervised_counts)
)

print(
    "Average:",
    round(
        sum(supervised_counts)
        / len(supervised_counts),
        2
    )
)


# ============================================================
# 5. SUPERVISION CHECK
# ============================================================

zero_supervision = 0
invalid_labels = 0

for item in correction_tokenized:

    mask = (
        item["labels"] != -100
    )

    supervised = int(mask.sum())

    if supervised == 0:
        zero_supervision += 1

    if not torch.equal(
        item["labels"][mask],
        item["input_ids"][mask]
    ):
        invalid_labels += 1


print()
print("Supervision checks")
print("-" * 70)

print(
    "Zero supervision:",
    zero_supervision
)

print(
    "Invalid labels:",
    invalid_labels
)


assert zero_supervision == 0
assert invalid_labels == 0


# ============================================================
# 6. VERIFY ANSWER TARGETS
# ============================================================

allowed_answers = {
    "front",
    "behind",
    "left",
    "right"
}

for raw, tokenized in zip(
    correction_raw,
    correction_tokenized
):

    assert raw["answer"] in allowed_answers

    mask = (
        tokenized["labels"] != -100
    )

    target_text = tokenizer.decode(
        tokenized["input_ids"][mask],
        skip_special_tokens=True
    ).strip()

    assert raw["answer"] in target_text


# ============================================================
# 7. CRITICAL — VERIFY LEFT IS NOT IN INPUT
# ============================================================

for raw, messages in zip(
    correction_raw,
    correction_messages
):

    user_text = messages[0]["content"]

    # The target answer must not be explicitly supplied
    assert "relative_direction" not in user_text

    # We specifically don't want the literal target word
    # accidentally appearing in the spatial description.
    if raw["answer"] == "left":
        assert "left" not in user_text.lower()


# ============================================================
# 8. SAMPLE
# ============================================================

sample_index = next(
    i
    for i, x in enumerate(correction_raw)
    if x["answer"] == "left"
)

sample_raw = correction_raw[sample_index]
sample_tok = correction_tokenized[sample_index]

print()
print("=" * 70)
print("LEFT-TARGET SAMPLE")
print("=" * 70)

print()
print(
    tokenizer.decode(
        sample_tok["input_ids"],
        skip_special_tokens=False
    )
)

print()
print(
    "SUPERVISED TARGET:"
)

print(
    tokenizer.decode(
        sample_tok["input_ids"][
            sample_tok["labels"] != -100
        ],
        skip_special_tokens=False
    )
)

print()
print(
    "Expected:",
    sample_raw["answer"]
)


# ============================================================
# 9. FINAL
# ============================================================

print()
print("=" * 70)
print("ALL 66B ASSERTIONS PASSED")
print("=" * 70)

STEP 66B — CORRECTION TOKENIZATION

Examples: 800

Sequence length
----------------------------------------------------------------------
Min: 78
Max: 85
Average: 80.53

Supervised tokens
----------------------------------------------------------------------
Min: 3
Max: 3
Average: 3.0

Supervision checks
----------------------------------------------------------------------
Zero supervision: 0
Invalid labels: 0


AssertionError: 

In [ ]:
# ============================================================
# STEP 66B — FIXED CORRECTION VALIDATION
# ============================================================

import torch

print("=" * 70)
print("STEP 66B — FIXED VALIDATION")
print("=" * 70)


# ============================================================
# 1. BASIC CHECKS
# ============================================================

assert len(correction_raw) == 800
assert len(correction_tokenized) == 800

print()
print("Examples:", len(correction_raw))


# ============================================================
# 2. SUPERVISION CHECK
# ============================================================

zero_supervision = 0
invalid_labels = 0

for item in correction_tokenized:

    mask = item["labels"] != -100

    supervised_count = int(mask.sum())

    if supervised_count == 0:
        zero_supervision += 1

    if not torch.equal(
        item["labels"][mask],
        item["input_ids"][mask]
    ):
        invalid_labels += 1


print()
print("SUPERVISION")
print("-" * 70)
print("Zero supervision:", zero_supervision)
print("Invalid labels:", invalid_labels)

assert zero_supervision == 0
assert invalid_labels == 0


# ============================================================
# 3. ANSWER CHECK
# ============================================================

allowed_answers = {
    "front",
    "behind",
    "left",
    "right"
}

for raw, tokenized in zip(
    correction_raw,
    correction_tokenized
):

    assert raw["answer"] in allowed_answers

    mask = tokenized["labels"] != -100

    target_text = tokenizer.decode(
        tokenized["input_ids"][mask],
        skip_special_tokens=True
    ).strip()

    assert raw["answer"] in target_text


print()
print("ANSWER TARGET CHECK: PASSED")


# ============================================================
# 4. NO ANSWER LEAKAGE IN SITUATION + QUESTION
# ============================================================
#
# IMPORTANT:
# We exclude the SYSTEM PROMPT because it intentionally
# contains the four possible answer words.
#
# We inspect ONLY:
#
#   Situation
#   Question
#
# ============================================================

leakage_count = 0

for raw in correction_raw:

    semantic_input = (
        raw["situation"]
        + " "
        + raw["question"]
    ).lower()

    answer = raw["answer"].lower()

    if answer in semantic_input:
        leakage_count += 1

        print()
        print("LEAKAGE FOUND:")
        print("Situation:", raw["situation"])
        print("Question:", raw["question"])
        print("Answer:", raw["answer"])


print()
print(
    "Target word appearing in "
    "situation/question:",
    leakage_count
)

assert leakage_count == 0


# ============================================================
# 5. VERIFY TRANSFORMATION LABEL
# ============================================================

relative_map = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left"
    },

    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind"
    },

    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right"
    },

    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front"
    }
}


for raw in correction_raw:

    expected = relative_map[
        raw["heading"]
    ][
        raw["world_direction"]
    ]

    assert raw["answer"] == expected


print(
    "Transformation-map verification: PASSED"
)


# ============================================================
# 6. LEFT-SPECIFIC CHECK
# ============================================================

left_examples = [
    x for x in correction_raw
    if x["answer"] == "left"
]

assert len(left_examples) == 400

print()
print(
    "LEFT correction examples:",
    len(left_examples)
)


# Verify all four left-producing transformations exist

left_pairs = {
    ("north", "west"),
    ("east", "north"),
    ("south", "east"),
    ("west", "south")
}

observed_left_pairs = {
    (
        x["heading"],
        x["world_direction"]
    )
    for x in left_examples
}

print(
    "LEFT transformation pairs:",
    len(observed_left_pairs),
    "/ 4"
)

assert observed_left_pairs == left_pairs


# ============================================================
# 7. SAMPLE
# ============================================================

sample = left_examples[0]

print()
print("=" * 70)
print("LEFT-TARGET SAMPLE")
print("=" * 70)

print()
print("Situation:")
print(sample["situation"])

print()
print("Question:")
print(sample["question"])

print()
print("Expected target:")
print(sample["answer"])


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("ALL 66B ASSERTIONS PASSED")
print("=" * 70)

print()
print("Correction curriculum is ready for training.")

STEP 66B — FIXED VALIDATION

Examples: 800

SUPERVISION
----------------------------------------------------------------------
Zero supervision: 0
Invalid labels: 0

ANSWER TARGET CHECK: PASSED

Target word appearing in situation/question: 0
Transformation-map verification: PASSED

LEFT correction examples: 400
LEFT transformation pairs: 4 / 4

LEFT-TARGET SAMPLE

Situation:
I am facing toward the east. A purple bottle is to my north.

Question:
Where would I see the bottle?

Expected target:
left

ALL 66B ASSERTIONS PASSED

Correction curriculum is ready for training.


In [ ]:
# ============================================================
# STEP 66C — TARGETED V2 LEFT-RELATION CORRECTION
# ============================================================
#
# Starting point:
#   EgoSpatial-Gemma V2
#
# Training:
#   800-example targeted correction curriculum
#
# Goal:
#   Fix systematic LEFT -> RIGHT confusion
#
# IMPORTANT:
#   Original V2 remains untouched.
# ============================================================

import os
import gc
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

from peft import PeftModel


# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "google/gemma-2-2b-it"

V2_ADAPTER = (
    "/content/egospatial_gemma_v2/final"
)

CORRECTION_OUTPUT = (
    "/content/egospatial_gemma_v2_left_corrected"
)

CORRECTION_STEPS = 100

BATCH_SIZE = 2
GRAD_ACCUMULATION = 4

EFFECTIVE_BATCH = (
    BATCH_SIZE * GRAD_ACCUMULATION
)

LEARNING_RATE = 5e-5

WARMUP_STEPS = 10

SEED = 66

os.makedirs(
    CORRECTION_OUTPUT,
    exist_ok=True
)


# ============================================================
# 1. VERIFY CORRECTION DATA
# ============================================================

assert "correction_tokenized" in globals(), \
    "correction_tokenized is missing. Run 66B first."

assert len(correction_tokenized) == 800

print("=" * 70)
print("STEP 66C — TARGETED V2 CORRECTION")
print("=" * 70)

print()
print("Correction examples:", len(correction_tokenized))
print("Training steps:", CORRECTION_STEPS)
print("Batch:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUMULATION)
print("Effective batch:", EFFECTIVE_BATCH)
print("Learning rate:", LEARNING_RATE)


# ============================================================
# 2. CLEAR CURRENT MODEL
# ============================================================

print()
print("Clearing current model...")

if "model" in globals():
    del model

if "trainer" in globals():
    del trainer

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 3. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 4. LOAD FRESH BASE GEMMA
# ============================================================

print()
print("=" * 70)
print("LOADING BASE GEMMA")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

base_model.config.use_cache = False


# ============================================================
# 5. LOAD EXISTING V2 ADAPTER
# ============================================================

print()
print("=" * 70)
print("LOADING EXISTING V2 ADAPTER")
print("=" * 70)

model = PeftModel.from_pretrained(
    base_model,
    V2_ADAPTER,
    is_trainable=True
)

model.train()

model.print_trainable_parameters()


# ============================================================
# 6. COLLATOR
# ============================================================

def correction_collator(features):

    max_len = max(
        len(f["input_ids"])
        for f in features
    )

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:

        input_ids_tensor = torch.as_tensor(
            f["input_ids"],
            dtype=torch.long
        )

        attention_tensor = torch.as_tensor(
            f["attention_mask"],
            dtype=torch.long
        )

        labels_tensor = torch.as_tensor(
            f["labels"],
            dtype=torch.long
        )

        pad_len = (
            max_len
            - len(input_ids_tensor)
        )

        if pad_len > 0:

            input_ids_tensor = torch.cat([
                input_ids_tensor,
                torch.full(
                    (pad_len,),
                    tokenizer.pad_token_id,
                    dtype=torch.long
                )
            ])

            attention_tensor = torch.cat([
                attention_tensor,
                torch.zeros(
                    pad_len,
                    dtype=torch.long
                )
            ])

            labels_tensor = torch.cat([
                labels_tensor,
                torch.full(
                    (pad_len,),
                    -100,
                    dtype=torch.long
                )
            ])

        input_ids.append(
            input_ids_tensor
        )

        attention_masks.append(
            attention_tensor
        )

        labels.append(
            labels_tensor
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(labels)
    }


# ============================================================
# 7. BUILD HF DATASET
# ============================================================

correction_ds = Dataset.from_dict({

    "input_ids": [
        x["input_ids"].tolist()
        if torch.is_tensor(x["input_ids"])
        else x["input_ids"]
        for x in correction_tokenized
    ],

    "attention_mask": [
        x["attention_mask"].tolist()
        if torch.is_tensor(x["attention_mask"])
        else x["attention_mask"]
        for x in correction_tokenized
    ],

    "labels": [
        x["labels"].tolist()
        if torch.is_tensor(x["labels"])
        else x["labels"]
        for x in correction_tokenized
    ]
})


# ============================================================
# 8. PREFLIGHT
# ============================================================

print()
print("=" * 70)
print("CORRECTION TRAINING PREFLIGHT")
print("=" * 70)

batch = correction_collator([
    correction_tokenized[0],
    correction_tokenized[1]
])

print()
print(
    "Batch shape:",
    tuple(batch["input_ids"].shape)
)

print(
    "Labels shape:",
    tuple(batch["labels"].shape)
)

print(
    "Supervised tokens:",
    int(
        (batch["labels"] != -100).sum()
    )
)

print(
    "Dataset:",
    len(correction_ds)
)

print(
    "Expected optimizer steps:",
    CORRECTION_STEPS
)


# ============================================================
# 9. FORWARD PASS
# ============================================================

print()
print("Running forward-pass check...")

model.eval()

device = next(
    p for p in model.parameters()
    if p.requires_grad
).device

batch_gpu = {
    k: v.to(device)
    for k, v in batch.items()
}

with torch.no_grad():

    outputs = model(
        input_ids=batch_gpu["input_ids"],
        attention_mask=batch_gpu["attention_mask"],
        labels=batch_gpu["labels"]
    )

    forward_loss = outputs.loss


print(
    "Forward-pass loss:",
    float(forward_loss)
)

assert torch.isfinite(forward_loss)

print(
    "Forward-pass check: PASSED"
)


# ============================================================
# 10. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir=CORRECTION_OUTPUT,

    max_steps=CORRECTION_STEPS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    warmup_steps=WARMUP_STEPS,

    fp16=True,

    gradient_checkpointing=False,

    optim="adamw_torch",

    logging_strategy="steps",

    logging_steps=10,

    save_strategy="steps",

    save_steps=50,

    save_total_limit=2,

    seed=SEED,

    data_seed=SEED,

    remove_unused_columns=False,

    report_to="none",

    dataloader_pin_memory=True,

    dataloader_num_workers=0
)


# ============================================================
# 11. TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=correction_ds,
    data_collator=correction_collator
)


# ============================================================
# 12. TRAIN
# ============================================================

print()
print("=" * 70)
print("STARTING TARGETED CORRECTION")
print("=" * 70)

print()
print(
    "Starting from V2 adapter:"
)

print(V2_ADAPTER)

print()
print(
    "Saving corrected adapter to:"
)

print(CORRECTION_OUTPUT)

print()
print("Training...")


train_result = trainer.train()


# ============================================================
# 13. SAVE CORRECTED MODEL
# ============================================================

FINAL_CORRECTED = os.path.join(
    CORRECTION_OUTPUT,
    "final"
)

print()
print("=" * 70)
print("SAVING CORRECTED V2")
print("=" * 70)

trainer.save_model(
    FINAL_CORRECTED
)

tokenizer.save_pretrained(
    FINAL_CORRECTED
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("STEP 66C COMPLETE")
print("=" * 70)

print()
print("Corrected adapter:")
print(FINAL_CORRECTED)

print()
print(
    "Training loss:",
    train_result.training_loss
)

print(
    "Global steps:",
    trainer.state.global_step
)

print(
    "Epoch:",
    trainer.state.epoch
)

print()
print("Original V2 remains at:")
print(V2_ADAPTER)

print()
print("=" * 70)
print("NEXT: 400-EXAMPLE CONTROLLED RETEST")
print("=" * 70)

STEP 66C — TARGETED V2 CORRECTION

Correction examples: 800
Training steps: 100
Batch: 2
Gradient accumulation: 4
Effective batch: 8
Learning rate: 5e-05

Clearing current model...
GPU memory allocated: 5.06 GB

LOADING BASE GEMMA


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


LOADING EXISTING V2 ADAPTER
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881

CORRECTION TRAINING PREFLIGHT

Batch shape: (2, 82)
Labels shape: (2, 82)
Supervised tokens: 6
Dataset: 800
Expected optimizer steps: 100

Running forward-pass check...
Forward-pass loss: 0.23650585114955902
Forward-pass check: PASSED

STARTING TARGETED CORRECTION

Starting from V2 adapter:
/content/egospatial_gemma_v2/final

Saving corrected adapter to:
/content/egospatial_gemma_v2_left_corrected

Training...


Step,Training Loss
10,0.156574
20,0.126664
30,0.112365
40,0.109172
50,0.165842
60,0.123383
70,0.165254
80,0.080754
90,0.068811
100,0.065296



SAVING CORRECTED V2

STEP 66C COMPLETE

Corrected adapter:
/content/egospatial_gemma_v2_left_corrected/final

Training loss: 0.1174114203453064
Global steps: 100
Epoch: 1.0

Original V2 remains at:
/content/egospatial_gemma_v2/final

NEXT: 400-EXAMPLE CONTROLLED RETEST


In [ ]:
# ================================================================
# STEP 66D — CONTROLLED RETEST OF CORRECTED V2
# ================================================================

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "google/gemma-2-2b-it"
CORRECTED_ADAPTER = "/content/egospatial_gemma_v2_left_corrected/final"
TEST_PATH = "/content/egospatial_v2_data/test.json"

print("=" * 70)
print("STEP 66D — CONTROLLED RETEST")
print("=" * 70)

# ------------------------------------------------
# Load EXACT held-out test set
# ------------------------------------------------

with open(TEST_PATH, "r") as f:
    test_data = json.load(f)

print(f"Test examples: {len(test_data)}")

assert len(test_data) == 400, "ERROR: Test set is not the original 400 examples."

# ------------------------------------------------
# Load fresh base Gemma
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FRESH BASE GEMMA")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

# ------------------------------------------------
# Load CORRECTED adapter
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOADING CORRECTED V2 ADAPTER")
print("=" * 70)

model = PeftModel.from_pretrained(
    base_model,
    CORRECTED_ADAPTER,
    is_trainable=False
)

model.eval()

print("Corrected adapter loaded.")
print(f"Adapter: {CORRECTED_ADAPTER}")

# ------------------------------------------------
# Prompt builder
# ------------------------------------------------

SYSTEM_PROMPT = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right"""

def make_prompt(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Situation:\n{example['situation']}\n\n"
                f"Question:\n{example['question']}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# ------------------------------------------------
# Evaluation
# ------------------------------------------------

valid_labels = {"front", "behind", "left", "right"}

correct = 0
invalid = 0

confusion = {
    "front":  {"front": 0, "behind": 0, "left": 0, "right": 0},
    "behind": {"front": 0, "behind": 0, "left": 0, "right": 0},
    "left":   {"front": 0, "behind": 0, "left": 0, "right": 0},
    "right":  {"front": 0, "behind": 0, "left": 0, "right": 0},
}

errors = []

print("\n" + "=" * 70)
print("RUNNING 400-EXAMPLE CONTROLLED TEST")
print("=" * 70)

for i, example in enumerate(test_data):

    prompt = make_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip().lower()

    # Keep only the first valid answer if extra text appears
    predicted_label = None

    for label in ["front", "behind", "left", "right"]:
        if prediction.startswith(label):
            predicted_label = label
            break

    truth = example["answer"].strip().lower()

    if predicted_label is None:
        invalid += 1
        errors.append({
            "index": i,
            "truth": truth,
            "prediction": prediction,
            "situation": example["situation"],
            "question": example["question"]
        })

    else:
        confusion[truth][predicted_label] += 1

        if predicted_label == truth:
            correct += 1
        else:
            errors.append({
                "index": i,
                "truth": truth,
                "prediction": predicted_label,
                "situation": example["situation"],
                "question": example["question"]
            })

    if (i + 1) % 50 == 0:
        print(f"Evaluated {i + 1}/400")

# ------------------------------------------------
# Results
# ------------------------------------------------

accuracy = correct / len(test_data) * 100

print("\n" + "=" * 70)
print("STEP 66D RESULTS")
print("=" * 70)

print(f"Correct:       {correct}/400")
print(f"Accuracy:      {accuracy:.2f}%")
print(f"Invalid:       {invalid}")

print("\nConfusion Matrix:")
print("                 Predicted")
print("Truth       front  behind  left  right")

for truth in ["front", "behind", "left", "right"]:
    row = confusion[truth]
    print(
        f"{truth:<10} "
        f"{row['front']:>5} "
        f"{row['behind']:>7} "
        f"{row['left']:>5} "
        f"{row['right']:>6}"
    )

# ------------------------------------------------
# Per-class accuracy
# ------------------------------------------------

print("\nPer-class accuracy:")

for label in ["front", "behind", "left", "right"]:

    total = sum(confusion[label].values())
    class_correct = confusion[label][label]

    if total:
        class_acc = class_correct / total * 100
    else:
        class_acc = 0

    print(
        f"{label:<8}: "
        f"{class_correct}/{total} "
        f"({class_acc:.2f}%)"
    )

# ------------------------------------------------
# Compare with original V2
# ------------------------------------------------

original_v2 = 76.25
improvement = accuracy - original_v2

print("\n" + "=" * 70)
print("COMPARISON WITH ORIGINAL V2")
print("=" * 70)

print(f"Original V2:       {original_v2:.2f}%")
print(f"Corrected V2:      {accuracy:.2f}%")
print(f"Change:            {improvement:+.2f} percentage points")

# ------------------------------------------------
# Save results
# ------------------------------------------------

results = {
    "model": "EgoSpatial-Gemma V2 + targeted left correction",
    "test_examples": 400,
    "correct": correct,
    "accuracy": accuracy,
    "invalid": invalid,
    "original_v2_accuracy": original_v2,
    "change_percentage_points": improvement,
    "confusion_matrix": confusion,
    "errors": errors
}

RESULT_PATH = "/content/egospatial_v2_corrected_control_results.json"

with open(RESULT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to:")
print(RESULT_PATH)

print("\n" + "=" * 70)
print("STEP 66D COMPLETE")
print("=" * 70)

STEP 66D — CONTROLLED RETEST


FileNotFoundError: [Errno 2] No such file or directory: '/content/egospatial_v2_data/test.json'

In [ ]:
import os

print("=" * 70)
print("SEARCHING FOR ORIGINAL V2 TEST SET")
print("=" * 70)

matches = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "test.json":
            path = os.path.join(root, file)
            matches.append(path)

print(f"\nFound {len(matches)} test.json file(s):")

for path in matches:
    print(path)

print("\nChecking V2 directories...")

for path in [
    "/content/egospatial_v2_data",
    "/content/egospatial_gemma_v2",
    "/content/egospatial_gemma_v2_left_corrected"
]:
    print(f"{path}: {'EXISTS' if os.path.exists(path) else 'MISSING'}")

SEARCHING FOR ORIGINAL V2 TEST SET

Found 0 test.json file(s):

Checking V2 directories...
/content/egospatial_v2_data: MISSING
/content/egospatial_gemma_v2: MISSING
/content/egospatial_gemma_v2_left_corrected: MISSING


In [ ]:
import os

print("=" * 70)
print("CHECKING COMMON PERSISTENT / MOUNTED LOCATIONS")
print("=" * 70)

locations = [
    "/content",
    "/content/drive",
    "/mnt/data"
]

for location in locations:
    print(f"\n{location}:")
    if os.path.exists(location):
        try:
            for item in os.listdir(location)[:50]:
                print("  ", item)
        except Exception as e:
            print("   Cannot list:", e)
    else:
        print("   MISSING")

CHECKING COMMON PERSISTENT / MOUNTED LOCATIONS

/content:
   .config
   sample_data

/content/drive:
   MISSING

/mnt/data:
   MISSING


# PERSISTENCE — Experiment Artifacts

Important experiment artifacts must not remain only in the temporary Colab runtime.

## Permanent storage

- Notebook and source code → GitHub
- LoRA adapters → Hugging Face
- Generated datasets → Hugging Face
- Evaluation results → GitHub + Hugging Face

## Rule

Before moving to the next major experiment:

1. Verify the experiment.
2. Save the model adapter.
3. Save the dataset/results.
4. Push the notebook update to GitHub.
5. Only then continue.

In [ ]:
# ============================================================
# EGO SPATIAL-GEMMA — PERSISTENCE SETUP
# ============================================================

!pip -q install -U huggingface_hub

from huggingface_hub import login, HfApi
import os

print("Hugging Face persistence tools ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 15.1 MB/s eta 0:00:00
Hugging Face persistence tools ready.


In [ ]:
from huggingface_hub import login, whoami

login()

print("Logged in as:", whoami()["name"])

Logged in as: Platinum04


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Persistent repositories for EgoSpatial-Gemma
repos = [
    "Platinum04/EgoSpatial-Gemma-v2",
    "Platinum04/EgoSpatial-Gemma-data",
]

for repo_id in repos:
    try:
        api.create_repo(
            repo_id=repo_id,
            repo_type="model" if "v2" in repo_id else "dataset",
            private=True,
            exist_ok=True
        )
        print(f"✓ Ready: {repo_id}")
    except Exception as e:
        print(f"✗ Could not create {repo_id}: {e}")

✓ Ready: Platinum04/EgoSpatial-Gemma-v2
✓ Ready: Platinum04/EgoSpatial-Gemma-data


In [ ]:
# ============================================================
# EGO SPATIAL-GEMMA — PERSISTENT BACKUP FUNCTION
# ============================================================

from huggingface_hub import HfApi
import os

api = HfApi()

MODEL_REPO = "Platinum04/EgoSpatial-Gemma-v2"
DATA_REPO = "Platinum04/EgoSpatial-Gemma-data"


def backup_folder(local_path, repo_id, repo_type="model", path_in_repo=None):
    """
    Upload an entire local folder to Hugging Face.

    Nothing is deleted from the local Colab runtime.
    Existing files with the same names are updated.
    """

    if not os.path.exists(local_path):
        print(f"⚠️ NOT FOUND: {local_path}")
        return

    print(f"Uploading: {local_path}")
    print(f"Destination: {repo_id}")

    api.upload_folder(
        folder_path=local_path,
        repo_id=repo_id,
        repo_type=repo_type,
        path_in_repo=path_in_repo,
    )

    print(f"✓ BACKUP COMPLETE: {repo_id}")


print("✓ Persistence system ready.")

✓ Persistence system ready.


# STEP 01 — Rebuild V2 Controlled Spatial Reasoning Dataset

## Objective

Reconstruct the clean V2 controlled spatial reasoning dataset used to test egocentric direction transformation.

The dataset will contain:

- 2,000 training examples
- 400 validation examples
- 400 test examples
- 4 target labels: front, behind, left, right
- 4 agent headings: north, east, south, west
- 4 world directions: north, east, south, west
- All 16 heading × world-direction transformations
- Zero exact overlap between train, validation, and test
- No target-direction leakage in the situation or question
- Deterministic generation using a fixed random seed

The dataset will be validated before being uploaded to persistent Hugging Face storage.

In [ ]:
# ============================================================
# STEP 01 — REBUILD CLEAN V2 DATASET
# DETERMINISTIC + GUARANTEED UNIQUE SPLITS
# ============================================================

import json
import os
import random
from itertools import product
from collections import Counter

SEED = 42

TRAIN_SIZE = 2000
VAL_SIZE = 400
TEST_SIZE = 400

OUTPUT_DIR = "/content/egospatial_v2_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)

# ------------------------------------------------------------
# 16 spatial transformations
# ------------------------------------------------------------

RELATIVE_MAP = {
    "north": {
        "north": "front",
        "east": "right",
        "south": "behind",
        "west": "left",
    },
    "east": {
        "north": "left",
        "east": "front",
        "south": "right",
        "west": "behind",
    },
    "south": {
        "north": "behind",
        "east": "left",
        "south": "front",
        "west": "right",
    },
    "west": {
        "north": "right",
        "east": "behind",
        "south": "left",
        "west": "front",
    },
}

HEADINGS = ["north", "east", "south", "west"]
WORLD_DIRECTIONS = ["north", "east", "south", "west"]

# ------------------------------------------------------------
# Larger vocabulary
# ------------------------------------------------------------

OBJECTS = [
    "bottle",
    "chair",
    "book",
    "lamp",
    "box",
    "backpack",
    "cup",
    "plant",
    "ball",
    "phone",
    "tablet",
    "shoe",
    "pillow",
    "clock",
    "camera",
    "mug",
    "basket",
    "bag",
    "notebook",
    "keyboard",
    "mouse",
    "speaker",
    "vase",
    "folder",
    "pen",
    "stool",
    "bin",
    "monitor",
    "scissors",
    "remote",
]

COLORS = [
    "red",
    "blue",
    "green",
    "yellow",
    "purple",
    "orange",
    "brown",
    "white",
    "black",
]

# Multiple linguistic templates create additional
# unique examples without changing the underlying task.
SITUATION_TEMPLATES = [
    "I am facing {heading}. There is a {color} {obj} to the {direction}.",
    "I am facing {heading}, and a {color} {obj} is to the {direction}.",
    "I face {heading}. A {color} {obj} is positioned to the {direction}.",
    "My facing direction is {heading}. A {color} {obj} is located to the {direction}.",
]

QUESTION_TEMPLATES = [
    "Where is the {obj} relative to me?",
    "Which direction is the {obj} from me?",
    "What direction is the {obj} relative to me?",
    "Where would I find the {obj} relative to my facing direction?",
]

# ------------------------------------------------------------
# Build candidate pool
# ------------------------------------------------------------

candidates = []

for (
    heading,
    world_direction,
    obj,
    color,
    situation_template,
    question_template,
) in product(
    HEADINGS,
    WORLD_DIRECTIONS,
    OBJECTS,
    COLORS,
    SITUATION_TEMPLATES,
    QUESTION_TEMPLATES,
):

    answer = RELATIVE_MAP[heading][world_direction]

    situation = situation_template.format(
        heading=heading,
        color=color,
        obj=obj,
        direction=world_direction,
    )

    question = question_template.format(
        obj=obj
    )

    candidates.append({
        "heading": heading,
        "world_direction": world_direction,
        "situation": situation,
        "question": question,
        "answer": answer,
    })

# ------------------------------------------------------------
# Deterministic shuffle
# ------------------------------------------------------------

random.shuffle(candidates)

required = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

assert len(candidates) >= required, (
    f"Not enough unique candidates: "
    f"{len(candidates)} < {required}"
)

# ------------------------------------------------------------
# Split
# ------------------------------------------------------------

train_data = candidates[:TRAIN_SIZE]

val_data = candidates[
    TRAIN_SIZE:
    TRAIN_SIZE + VAL_SIZE
]

test_data = candidates[
    TRAIN_SIZE + VAL_SIZE:
    TRAIN_SIZE + VAL_SIZE + TEST_SIZE
]

# ------------------------------------------------------------
# Add IDs
# ------------------------------------------------------------

for i, item in enumerate(train_data):
    item["id"] = f"train_{i:05d}"

for i, item in enumerate(val_data):
    item["id"] = f"val_{i:05d}"

for i, item in enumerate(test_data):
    item["id"] = f"test_{i:05d}"

# ------------------------------------------------------------
# Exact uniqueness checks
# ------------------------------------------------------------

def example_key(x):
    return (
        x["heading"],
        x["world_direction"],
        x["situation"],
        x["question"],
    )

train_keys = {example_key(x) for x in train_data}
val_keys = {example_key(x) for x in val_data}
test_keys = {example_key(x) for x in test_data}

assert len(train_keys) == TRAIN_SIZE
assert len(val_keys) == VAL_SIZE
assert len(test_keys) == TEST_SIZE

assert train_keys.isdisjoint(val_keys)
assert train_keys.isdisjoint(test_keys)
assert val_keys.isdisjoint(test_keys)

# ------------------------------------------------------------
# Verify all 16 transformations occur in every split
# ------------------------------------------------------------

for split_name, split in [
    ("train", train_data),
    ("validation", val_data),
    ("test", test_data),
]:

    pairs = {
        (x["heading"], x["world_direction"])
        for x in split
    }

    assert len(pairs) == 16, (
        f"{split_name}: only {len(pairs)}/16 transformations"
    )

# ------------------------------------------------------------
# Verify transformation labels
# ------------------------------------------------------------

for split in [train_data, val_data, test_data]:

    for x in split:

        expected = RELATIVE_MAP[
            x["heading"]
        ][
            x["world_direction"]
        ]

        assert x["answer"] == expected

# ------------------------------------------------------------
# Verify no target leakage
# ------------------------------------------------------------

TARGET_WORDS = {"front", "behind", "left", "right"}

for split in [train_data, val_data, test_data]:

    for x in split:

        text = (
            x["situation"] + " " +
            x["question"]
        ).lower()

        words = set(text.replace(",", "").replace(".", "").split())

        assert not (
            words & TARGET_WORDS
        ), f"Target leakage detected: {text}"

# ------------------------------------------------------------
# Save JSON files
# ------------------------------------------------------------

paths = {
    "train": os.path.join(
        OUTPUT_DIR,
        "train.json"
    ),
    "validation": os.path.join(
        OUTPUT_DIR,
        "validation.json"
    ),
    "test": os.path.join(
        OUTPUT_DIR,
        "test.json"
    ),
}

with open(
    paths["train"],
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        train_data,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    paths["validation"],
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        val_data,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    paths["test"],
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        test_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("=" * 60)
print("V2 DATASET GENERATION COMPLETE")
print("=" * 60)

print(f"Candidate pool: {len(candidates)}")
print(f"Train:          {len(train_data)}")
print(f"Validation:     {len(val_data)}")
print(f"Test:           {len(test_data)}")

print()
print("Exact split overlap:")
print("Train / Val :", len(train_keys & val_keys))
print("Train / Test:", len(train_keys & test_keys))
print("Val / Test  :", len(val_keys & test_keys))

print()
print("Answer distribution:")

for split_name, split in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data),
]:
    counts = Counter(
        x["answer"]
        for x in split
    )

    print(
        f"{split_name}: "
        f"front={counts['front']}, "
        f"behind={counts['behind']}, "
        f"left={counts['left']}, "
        f"right={counts['right']}"
    )

print()
print("Files:")

for name, path in paths.items():
    print(f"✓ {name}: {path}")

print()
print("✓ ALL VALIDATION CHECKS PASSED.")

V2 DATASET GENERATION COMPLETE
Candidate pool: 69120
Train:          2000
Validation:     400
Test:           400

Exact split overlap:
Train / Val : 0
Train / Test: 0
Val / Test  : 0

Answer distribution:
Train: front=474, behind=523, left=497, right=506
Validation: front=97, behind=91, left=119, right=93
Test: front=110, behind=89, left=97, right=104

Files:
✓ train: /content/egospatial_v2_data/train.json
✓ validation: /content/egospatial_v2_data/validation.json
✓ test: /content/egospatial_v2_data/test.json

✓ ALL VALIDATION CHECKS PASSED.


In [ ]:
# ============================================================
# STEP 01 — BACKUP V2 DATASET TO HUGGING FACE
# ============================================================

backup_folder(
    local_path="/content/egospatial_v2_data",
    repo_id=DATA_REPO,
    repo_type="dataset",
    path_in_repo="v2"
)

print()
print("🔒 V2 DATASET IS NOW PERSISTED.")

Uploading: /content/egospatial_v2_data
Destination: Platinum04/EgoSpatial-Gemma-data
✓ BACKUP COMPLETE: Platinum04/EgoSpatial-Gemma-data

🔒 V2 DATASET IS NOW PERSISTED.


# STEP 02 — V2 Tokenization

## Objective

Load the persisted V2 dataset and convert each example into the native Gemma chat format.

The model receives:

- A fixed spatial-reasoning instruction
- The situation
- The question

The model must generate exactly one spatial label:

- front
- behind
- left
- right

The target direction is not explicitly provided in the input.

Before training, verify:

- Dataset sizes
- Chat-template formatting
- Sequence lengths
- Supervised token counts
- Valid labels
- Zero examples with missing supervision

In [3]:
# ============================================================
# STEP 02A — RESTORE V2 DATASET FROM HUGGING FACE
# ============================================================

from huggingface_hub import snapshot_download
import os
import json

DATA_DIR = "/content/egospatial_v2_data"

# Download the persisted V2 dataset
snapshot_download(
    repo_id="Platinum04/EgoSpatial-Gemma-data",
    repo_type="dataset",
    allow_patterns=["v2/*.json"],
    local_dir="/content/egospatial_v2_restore"
)

# Move the V2 files into our standard working directory
os.makedirs(DATA_DIR, exist_ok=True)

restore_dir = "/content/egospatial_v2_restore/v2"

for filename in [
    "train.json",
    "validation.json",
    "test.json"
]:
    source = os.path.join(restore_dir, filename)
    destination = os.path.join(DATA_DIR, filename)

    assert os.path.exists(source), f"Missing from HF: {source}"

    import shutil
    shutil.copy2(source, destination)

# Load and verify
with open(os.path.join(DATA_DIR, "train.json"), "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open(os.path.join(DATA_DIR, "validation.json"), "r", encoding="utf-8") as f:
    val_raw = json.load(f)

with open(os.path.join(DATA_DIR, "test.json"), "r", encoding="utf-8") as f:
    test_raw = json.load(f)

print("=" * 60)
print("V2 DATASET RESTORED FROM HUGGING FACE")
print("=" * 60)

print(f"Train:      {len(train_raw)}")
print(f"Validation: {len(val_raw)}")
print(f"Test:       {len(test_raw)}")

print()
print("Local files:")

for filename in [
    "train.json",
    "validation.json",
    "test.json"
]:
    path = os.path.join(DATA_DIR, filename)
    print(f"✓ {path}")

print()
print("Example:")
print(json.dumps(train_raw[0], indent=2))

print()
print("✓ DATASET RESTORATION COMPLETE.")

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6aa2f185-0d23aeab5a85558c536f53fe;8a77aea0-10f0-4973-9f41-e1b411efac88)

Repository Not Found for url: https://huggingface.co/api/datasets/Platinum04/EgoSpatial-Gemma-data/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [4]:
from huggingface_hub import login, whoami

login()

print("Logged in as:", whoami()["name"])

Logged in as: Platinum04


In [5]:
# ============================================================
# STEP 02A — RESTORE V2 DATASET FROM HUGGING FACE
# ============================================================

from huggingface_hub import snapshot_download
import os
import json
import shutil

DATA_DIR = "/content/egospatial_v2_data"
RESTORE_DIR = "/content/egospatial_v2_restore"

# Download the persisted V2 dataset
snapshot_download(
    repo_id="Platinum04/EgoSpatial-Gemma-data",
    repo_type="dataset",
    allow_patterns=["v2/*.json"],
    local_dir=RESTORE_DIR
)

# Recreate our working directory
os.makedirs(DATA_DIR, exist_ok=True)

# Restore the three files
for filename in ["train.json", "validation.json", "test.json"]:
    source = os.path.join(RESTORE_DIR, "v2", filename)
    destination = os.path.join(DATA_DIR, filename)

    assert os.path.exists(source), f"Missing from Hugging Face: {source}"

    shutil.copy2(source, destination)

# Load them
with open(os.path.join(DATA_DIR, "train.json"), "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open(os.path.join(DATA_DIR, "validation.json"), "r", encoding="utf-8") as f:
    val_raw = json.load(f)

with open(os.path.join(DATA_DIR, "test.json"), "r", encoding="utf-8") as f:
    test_raw = json.load(f)

print("=" * 60)
print("V2 DATASET RESTORED")
print("=" * 60)

print(f"Train:      {len(train_raw)}")
print(f"Validation: {len(val_raw)}")
print(f"Test:       {len(test_raw)}")

print()
print("Files restored:")
print("✓ train.json")
print("✓ validation.json")
print("✓ test.json")

print()
print("✓ DATASET RESTORATION COMPLETE.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

V2 DATASET RESTORED
Train:      2000
Validation: 400
Test:       400

Files restored:
✓ train.json
✓ validation.json
✓ test.json

✓ DATASET RESTORATION COMPLETE.


# STEP 02B — Native Gemma Chat Formatting

The V2 examples will be formatted using Gemma's native chat template.

Input:

- Spatial reasoning instruction
- Situation
- Question

Target:

- Exactly one of: front, behind, left, right

The answer is not included in the user prompt.

In [6]:
# ============================================================
# STEP 02B — LOAD GEMMA TOKENIZER
# ============================================================

import torch
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Model:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Tokenizer loaded.
Model: google/gemma-2-2b-it
Tokenizer: GemmaTokenizer


# STEP 02C — Build Native Gemma Training Format

Convert each V2 example into Gemma's native user → model conversation format.

The target answer remains separate from the user prompt so that the spatial label is supervised rather than leaked into the input.

In [7]:
# ============================================================
# STEP 02C — NATIVE GEMMA CHAT FORMAT
# ============================================================

SYSTEM_INSTRUCTION = """You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right"""

def build_messages(example):
    user_content = f"""{SYSTEM_INSTRUCTION}

Situation:
{example["situation"]}

Question:
{example["question"]}"""

    return [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]


# Build one example
sample_messages = build_messages(train_raw[0])

# Apply Gemma's native chat template
sample_text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False
)

print("=" * 70)
print("NATIVE GEMMA CHAT EXAMPLE")
print("=" * 70)
print(sample_text)

print()
print("=" * 70)
print("EXPECTED ANSWER")
print("=" * 70)
print(train_raw[0]["answer"])

NATIVE GEMMA CHAT EXAMPLE
<bos><start_of_turn>user
You are a spatial reasoning assistant.

Determine the direction of the object relative to the person's facing direction.

Answer with exactly one of:
front
behind
left
right

Situation:
I face east. A black speaker is positioned to the east.

Question:
What direction is the speaker relative to me?<end_of_turn>
<start_of_turn>model
front<end_of_turn>


EXPECTED ANSWER
front
